# K3 + Hengck: timeout-only corrective candidate
Unchanged prediction policy. Internal runtime budget follows the 12-hour competition limit.
Ordinary private GPU commits remain capped at 7200 seconds through the Kaggle push timeout.
V1 hidden failure cause is unconfirmed; this candidate only addresses reproducible premature deadlines.


In [ ]:
FILES = {'hengck_original_helpers.py': 'import torch\nimport numpy as np\nimport pandas as pd\nimport polars as pl\nimport tracksdata as td\n\n@torch.no_grad()\ndef cluster_predicted_nodes(out, radius=1.0):\n    """\n    Greedy score-ordered duplicate suppression on refined node coordinates.\n\n    For each frame:\n      1) sort valid predictions by refine_logit descending;\n      2) keep the highest-scoring remaining prediction;\n      3) suppress remaining predictions within `radius`;\n      4) compact kept nodes to IDs 0..K-1.\n\n    We keep an actual model prediction rather than averaging coordinates/features,\n    so node_feature and refine_zyx remain mutually consistent.\n    """\n    if radius is None or radius <= 0:\n        n = int(out.mask.sum().item())\n        return out, {"num_before": n, "num_after": n, "num_removed": 0}\n\n    B, N, _ = out.refine_zyx.shape\n    device = out.refine_zyx.device\n\n    new_refine_zyx = torch.zeros_like(out.refine_zyx)\n    new_refine_logit = torch.zeros_like(out.refine_logit)\n    new_peak_zyx = torch.zeros_like(out.peak_zyx)\n    new_peak_logit = torch.zeros_like(out.peak_logit)\n    new_mask = torch.zeros_like(out.mask)\n    new_node_feature = [torch.zeros_like(f) for f in out.node_feature]\n\n    new_count = []\n    num_before = 0\n    num_after = 0\n    r2 = float(radius) ** 2\n\n    for b in range(B):\n        valid = torch.where(out.mask[b])[0]\n        nv = int(len(valid))\n        num_before += nv\n\n        if nv == 0:\n            new_count.append(0)\n            continue\n\n        zyx = out.refine_zyx[b, valid].float()\n        score = out.refine_logit[b, valid].float()\n        order = torch.argsort(score, descending=True)\n\n        suppressed = torch.zeros(nv, dtype=torch.bool, device=device)\n        keep_local = []\n\n        for j in order.tolist():\n            if bool(suppressed[j]):\n                continue\n\n            keep_local.append(j)\n            d2 = ((zyx - zyx[j]) ** 2).sum(dim=-1)\n            suppressed |= d2 <= r2\n\n        keep_local = torch.as_tensor(keep_local, dtype=torch.long, device=device)\n        keep = valid[keep_local]\n        k = int(len(keep))\n\n        num_after += k\n        new_count.append(k)\n        new_mask[b, :k] = True\n        new_refine_zyx[b, :k] = out.refine_zyx[b, keep]\n        new_refine_logit[b, :k] = out.refine_logit[b, keep]\n        new_peak_zyx[b, :k] = out.peak_zyx[b, keep]\n        new_peak_logit[b, :k] = out.peak_logit[b, keep]\n\n        for level, f in enumerate(out.node_feature):\n            new_node_feature[level][b, :k] = f[b, keep]\n\n    out.refine_zyx = new_refine_zyx\n    out.refine_logit = new_refine_logit\n    out.peak_zyx = new_peak_zyx\n    out.peak_logit = new_peak_logit\n    out.node_feature = new_node_feature\n    out.mask = new_mask\n    out.count = new_count\n    return out\n\ndef edge_top_cumulative_bidirectional(\n    edge_logit,\n    mask_src,\n    mask_dst,\n    t_start,\n    no_link_logit=0.0,\n    cumulative=0.99,\n    no_link_weight=1.0,\n):\n    """\n    Bidirectional version of edge_top_cumulative.\n\n    edge_logit : (B, Ns, Nd)\n        Raw pairwise edge logits.\n\n    For each real edge i -> j:\n        p_dst(j | i) = softmax over destination\n        p_src(i | j) = softmax over source\n\n        score(i, j) = p_dst(j | i) * p_src(i | j)\n\n    NO_LINK exists only on the forward/source->destination side:\n        score(i, NO_LINK) = p_dst(NO_LINK | i) * no_link_weight\n\n    Scores are then normalized over destinations for each source so that\n    cumulative probability selection remains meaningful.\n\n    Returns list of dict:\n        t\n        src_id\n        dst_id       (-1 = NO_LINK)\n        prob\n        rank\n        cumulative_prob\n        logit\n        prob_dst\n        prob_src\n    """\n\n    edge_logit = edge_logit.float()\n    mask_src = mask_src.bool()\n    mask_dst = mask_dst.bool()\n\n    B, Ns, Nd = edge_logit.shape\n    rows = []\n\n    for b in range(B):\n        # -------------------------------------------------------------\n        # Real edge logits\n        # -------------------------------------------------------------\n        real_logits = edge_logit[b]  # (Ns, Nd)\n\n        # -------------------------------------------------------------\n        # Forward probability P(dst | src)\n        # Includes explicit NO_LINK.\n        # -------------------------------------------------------------\n        dst_logits = real_logits.masked_fill(~mask_dst[b][None, :], -1e9)\n\n        no_link = torch.full(\n            (Ns, 1),\n            float(no_link_logit),\n            dtype=dst_logits.dtype,\n            device=dst_logits.device,\n        )\n\n        dst_logits_with_no_link = torch.cat([dst_logits, no_link], dim=-1)\n        prob_dst = torch.softmax(dst_logits_with_no_link, dim=-1)  # (Ns, Nd+1)\n\n        # -------------------------------------------------------------\n        # Reverse probability P(src | dst)\n        # Only real edges participate.\n        # -------------------------------------------------------------\n        src_logits = real_logits.masked_fill(~mask_src[b][:, None], -1e9)\n        src_logits = src_logits.masked_fill(~mask_dst[b][None, :], -1e9)\n\n        prob_src = torch.softmax(src_logits, dim=0)  # (Ns, Nd)\n\n        # Invalid destinations should contribute exactly zero.\n        prob_src = prob_src * mask_dst[b][None, :].to(prob_src.dtype)\n\n        # -------------------------------------------------------------\n        # Bidirectional edge score\n        # -------------------------------------------------------------\n        score_real = prob_dst[:, :Nd] * prob_src\n\n        score_no_link = prob_dst[:, Nd:Nd + 1] * float(no_link_weight)\n\n        score = torch.cat([score_real, score_no_link], dim=-1)\n\n        # Normalize again over destination so each row sums to one.\n        prob = score / score.sum(dim=-1, keepdim=True).clamp_min(1e-12)\n\n        # -------------------------------------------------------------\n        # Cumulative candidate selection\n        # -------------------------------------------------------------\n        src_ids = torch.where(mask_src[b])[0]\n\n        for src_id in src_ids.tolist():\n            p = prob[src_id]\n\n            p_sorted, idx_sorted = torch.sort(p, descending=True)\n            cum = torch.cumsum(p_sorted, dim=0)\n\n            hit = torch.where(cum >= cumulative)[0]\n            if len(hit):\n                n_keep = int(hit[0]) + 1\n            else:\n                n_keep = len(p_sorted)\n\n            # if USE_ILP:\n            #     n_keep = max(n_keep, 2)\n\n            for rank in range(n_keep):\n                idx = int(idx_sorted[rank])\n\n                if idx == Nd:\n                    dst_id = -1\n                    p_src_value = 1.0\n                    raw_logit = float(no_link_logit)\n                else:\n                    dst_id = idx\n                    p_src_value = float(prob_src[src_id, idx])\n                    raw_logit = float(real_logits[src_id, idx])\n\n                rows.append({\n                    "t": int(t_start + b),\n                    "src_id": int(src_id),\n                    "dst_id": dst_id,\n                    "rank": rank,\n                    "prob": float(p_sorted[rank]),\n                    "cumulative_prob": float(cum[rank]),\n                    "logit": raw_logit,\n                    "prob_dst": float(prob_dst[src_id, idx]),\n                    "prob_src": p_src_value,\n                })\n\n    return rows\n\ndef select_top1_with_dst_correction(predict_edge_df):\n    """\n    Keep forward top-1, then resolve destination conflicts using original logits.\n\n    For each conflicting (t, dst_id):\n      - Softmax over sources whose forward top-1 selected this destination.\n      - Keep the source with maximum probability.\n      - No birth option is added in this experiment.\n\n    Non-conflicting edges stay unchanged. Original prob/rank are preserved.\n    NO_LINK rows are excluded from the returned graph.\n    """\n    required = {"t", "src_id", "dst_id", "rank", "prob", "logit"}\n    missing = required - set(predict_edge_df.columns)\n    if missing:\n        raise ValueError(f"Missing columns: {sorted(missing)}")\n\n    df = predict_edge_df[\n        (predict_edge_df["rank"] == 0) & (predict_edge_df["dst_id"] >= 0)\n    ].copy()\n    df = df.drop_duplicates(["t", "src_id", "dst_id"]).reset_index(drop=True)\n\n    if df.groupby(["t", "src_id"]).size().gt(1).any():\n        raise ValueError("Input contains multiple distinct rank-0 edges for one source")\n\n    if not np.isfinite(df["logit"].to_numpy(dtype=np.float64)).all():\n        raise ValueError("Selected edge logits must be finite")\n\n    keys = ["t", "dst_id"]\n    groups = df.groupby(keys, sort=False)\n    df["dst_num_competing_src"] = groups["src_id"].transform("size")\n\n    # Stable softmax over competing sources for each destination.\n    score = df["logit"].astype(np.float64)\n    weight = np.exp(score - groups["logit"].transform("max"))\n    denominator = weight.groupby([df["t"], df["dst_id"]]).transform("sum")\n    df["dst_competition_prob"] = weight / denominator\n\n    # Softmax preserves logit ordering; source ID breaks exact ties deterministically.\n    df = df.sort_values(\n        ["t", "dst_id", "logit", "src_id"], ascending=[True, True, False, True],\n        kind="stable",\n    )\n    df = df.drop_duplicates(keys, keep="first")\n    return df.sort_values(["t", "src_id"], kind="stable").reset_index(drop=True)\n\ndef prediction_to_geff_graph(predict_node_df, predict_edge_df, downsample=(1, 4, 4)):\n    """\n    Convert our per-frame local node IDs and directed t->t+1 edges into a\n    tracksdata graph suitable for save_graph(... .geff).\n\n    predict_node_df columns:\n        t, node_id, z, y, x, p\n\n    predict_edge_df columns:\n        t, src_id, dst_id, ...\n\n    src_id/dst_id are LOCAL node IDs inside frame t / t+1, so the key\n    (t, node_id) is used to map them to graph-global node IDs.\n\n    Our inference coordinates are on the downsampled 64^3 grid.  Kaggle\'s\n    truth GEFF uses the original image voxel coordinate system, therefore\n    coordinates are multiplied by downsample=(1,4,4) before saving.\n    """\n    required_node = {"t", "node_id", "z", "y", "x"}\n    required_edge = {"t", "src_id", "dst_id"}\n    missing_node = required_node - set(predict_node_df.columns)\n    missing_edge = required_edge - set(predict_edge_df.columns)\n    if missing_node:\n        raise ValueError(f"predict_node_df missing columns: {sorted(missing_node)}")\n    if missing_edge:\n        raise ValueError(f"predict_edge_df missing columns: {sorted(missing_edge)}")\n\n    graph = td.graph.InMemoryGraph()\n    for key in ["z", "y", "x"]:\n        graph.add_node_attr_key(key, pl.Float64, -999999.0)\n\n    dz, dy, dx = map(float, downsample)\n\n    node_records = []\n    local_keys = []\n    for r in predict_node_df.itertuples(index=False):\n        node_records.append({\n            "t": int(r.t),\n            "z": float(r.z) * dz,\n            "y": float(r.y) * dy,\n            "x": float(r.x) * dx,\n        })\n        local_keys.append((int(r.t), int(r.node_id)))\n\n    graph_node_ids = graph.bulk_add_nodes(node_records)\n    node_map = {\n        key: int(graph_id)\n        for key, graph_id in zip(local_keys, graph_node_ids)\n    }\n\n    edge_records = []\n    skipped_edges = 0\n\n    # Optional attributes are useful for inspection but are not required\n    # by the Kaggle evaluator.\n    graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)\n\n    for r in predict_edge_df.itertuples(index=False):\n        t = int(r.t)\n        src_key = (t, int(r.src_id))\n        dst_key = (t + 1, int(r.dst_id))\n\n        src = node_map.get(src_key)\n        dst = node_map.get(dst_key)\n        if src is None or dst is None:\n            skipped_edges += 1\n            continue\n\n        prob = float(getattr(r, "prob", 1.0))\n        edge_records.append({\n            "source_id": src,\n            "target_id": dst,\n            "edge_prob": prob,\n        })\n\n    if edge_records:\n        graph.bulk_add_edges(edge_records)\n\n    return graph, skipped_edges\n', 'hengck_original_config.py': 'from model_v12 import DotDict\nCFG = DotDict(\n    unet_cfg=DotDict(\n        channel=(64, 128, 256),\n        dropout=0.1,\n        gradient_checkpointing=True,\n\n        node_peak_kernel=3,\n        node_peak_threshold=0.2,\n        max_detected_nodes=1024,\n    ),\n    tx_cfg=DotDict(\n        feat_dims=(64, 128, 256),\n        hidden_dim=256,\n        embed_dim=128,\n        n_heads=4,\n        n_layers=4,\n        dropout=0.1,\n    ),\n)\n', 'hengck_plain_checkpoint.py': "import hashlib, json\nfrom pathlib import Path\nimport torch\nGLOBAL = 'loss_and_metric_v12.DotDict'\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef restricted_load(path):\n    before = torch.serialization.get_safe_globals()\n    torch.serialization.clear_safe_globals()\n    try:\n        unsafe = sorted(torch.serialization.get_unsafe_globals_in_checkpoint(path))\n        if unsafe != [GLOBAL]:\n            raise ValueError('Unexpected globals: ' + repr(unsafe))\n        if hasattr(path, 'seek'):\n            path.seek(0)\n        with torch.serialization.safe_globals([(dict, GLOBAL)]):\n            saved = torch.load(path, map_location='cpu', weights_only=True)\n        assert not torch.serialization.get_safe_globals()\n        return saved, unsafe\n    finally:\n        torch.serialization.clear_safe_globals()\n        torch.serialization.add_safe_globals(before)\n        assert set(torch.serialization.get_safe_globals()) == set(before)\nWEIGHT_SHA = 'ecd8869de9cf405c1a93a563b4810faad5fd2378f57341e18c4e6b5a87201552'\nSTATE_SHA = '465a949f23627c618b540d5a52065f7192b2eb9e27f51275a6b4ed8f3a8c06c6'\n\ndef load_checkpoint(weights, receipt_path):\n    weights, receipt_path = Path(weights), Path(receipt_path)\n    assert weights.stat().st_size == 43076221 and digest(weights) == WEIGHT_SHA\n    loss_sha = '40cc8013c47cc4d2f2684dfccb153016e9ef9372ad2db10ef376aa3e04e3133a'\n    assert digest(weights.parent / 'loss_and_metric_v12.py') == loss_sha\n    receipt = dict(execution=3, weights_sha256=WEIGHT_SHA, weights_bytes=43076221,\n        reviewed_dotdict_source_sha256=loss_sha, weights_only=True, complete=False,\n        unreviewed_globals_allowed=False, replacement_type='builtins.dict')\n    receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')\n    saved, unsafe = restricted_load(weights)\n    state = saved['model_state_dict']\n    assert isinstance(state, dict) and state and all(isinstance(k, str) and torch.is_tensor(v) for k, v in state.items())\n    state_hash = hashlib.sha256()\n    for key, tensor in sorted(state.items()):\n        assert tensor.device.type == 'cpu' and torch.isfinite(tensor).all()\n        state_hash.update(json.dumps([key, str(tensor.dtype), list(tensor.shape)]).encode())\n        state_hash.update(tensor.detach().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())\n    assert state_hash.hexdigest() == STATE_SHA\n    receipt.update(complete=True, static_globals=unsafe, allowed_globals=[GLOBAL],\n        allowlist_restored=True, state_tensor_count=len(state), state_numel=sum(t.numel() for t in state.values()),\n        tensor_state_sha256=state_hash.hexdigest(), cpu_verified_tensor_parity=True)\n    receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')\n    return saved\n", 'hengck_frame_cache.py': '"""Immutable per-frame representations; preserve dtype, coordinates and feature order."""\nimport hashlib\nimport json\nimport numpy as np\n\nFIELDS = (\'zyx\', \'peak_zyx\', \'refine_logit\', \'peak_logit\', \'feature0\', \'feature1\', \'feature2\')\nTAILS = ((3,), (3,), (), (), (64,), (128,), (256,))\n\n\ndef freeze_frame(values):\n    if set(values) != set(FIELDS):\n        raise ValueError(\'Unexpected frame fields\')\n    count = len(values[\'zyx\'])\n    if not 0 < count <= 1024:\n        raise ValueError(\'Unexpected detected-node count\')\n    result = {}\n    for key, tail in zip(FIELDS, TAILS):\n        array = np.asarray(values[key])\n        if array.shape != (count, *tail) or array.dtype.kind != \'f\' or not np.isfinite(array).all():\n            raise ValueError(\'Invalid frame tensor: \' + key)\n        result[key] = array.copy()\n        result[key].flags.writeable = False\n    return result\n\n\ndef frame_hash(frame):\n    digest = hashlib.sha256()\n    for key in FIELDS:\n        array = np.asarray(frame[key])\n        digest.update(json.dumps([key, str(array.dtype), list(array.shape)]).encode())\n        digest.update(array.tobytes())\n    return digest.hexdigest()\n\n\ndef pack_frame(frame, requested_width):\n    if not isinstance(requested_width, int) or requested_width < 1:\n        raise ValueError(\'Invalid requested padding width\')\n    count = len(frame[\'zyx\'])\n    width = max(count, requested_width)\n    result = {key: np.zeros((width, *frame[key].shape[1:]), dtype=frame[key].dtype) for key in FIELDS}\n    for key in FIELDS:\n        result[key][:count] = frame[key]\n    result[\'mask\'] = np.arange(width) < count\n    return result\n\n\ndef unpack_frame(packed):\n    mask = np.asarray(packed[\'mask\'])\n    if mask.dtype != bool or mask.ndim != 1 or not mask.any():\n        raise ValueError(\'Invalid packed frame mask\')\n    return freeze_frame({key: np.asarray(packed[key])[mask] for key in FIELDS})\n\n', 'hengck_test_runtime.py': '"""One canonical TRAIN inference cache, with the actual author\'s candidate bank."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom hengck_frame_cache import FIELDS, freeze_frame, frame_hash, pack_frame, unpack_frame\nfrom src.k3_hengck_production_adapter import decode_arrays\n\n\ndef file_sha(path):\n    with path.open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef run(model_directory, output_directory, test_directory):\n    import torch\n    import pandas as pd\n    import zarr\n    import model_v12\n    import hengck_original_helpers as original\n    from hengck_original_config import CFG\n    from hengck_plain_checkpoint import load_checkpoint\n\n    started = time.monotonic()\n    output = Path(output_directory)\n    output.mkdir(parents=True, exist_ok=True)\n    assert file_sha(Path(model_v12.__file__)) == \'586ac08e4429fa0ca0db0c06afdf6d7ef77d459c0260bbf286bf47169d484082\'\n    assert torch.cuda.is_available()\n    torch.set_num_threads(4)\n    saved = load_checkpoint(Path(model_directory) / \'00000008.pth\', output / \'checkpoint_load.json\')\n    model = model_v12.End2EndCellLinker(CFG)\n    model.load_state_dict(saved[\'model_state_dict\'], strict=True)\n    del saved\n    model = model.to(\'cuda\').eval()\n    torch.cuda.reset_peak_memory_stats()\n    files, videos = {}, {}\n\n    def deadline():\n        if time.monotonic() - started > 42900:\n            raise TimeoutError(\'Hengck inference time budget exceeded\')\n\n    for path in sorted(Path(test_directory).glob(\'*.zarr\')):\n        name = path.stem\n        deadline()\n        clip_started = time.monotonic()\n        group = zarr.open_group(str(path), mode=\'r\')\n        arr = group[\'0\']\n        assert len(arr.shape) == 4 and tuple(arr.shape[1:]) == (64, 256, 256) and arr.shape[0] >= 2\n        frame_count = int(arr.shape[0])\n        chunks = [(s, min(s+21, frame_count)) for s in range(0, frame_count-1, 20)]\n        quantiles = dict(group.attrs)[\'image_statistics\'][\'quantiles\']\n        low, high = float(quantiles[\'0.001\']), float(quantiles[\'0.999\'])\n        assert high > low\n        canonical, input_hashes, counts, node_rows, candidate_rows = {}, {}, {}, [], []\n        boundaries, matrices, pair_shapes = [], [], []\n        with torch.no_grad(), torch.amp.autocast(\'cuda\', dtype=torch.float16):\n            for first, stop in chunks:\n                deadline()\n                (output / \'progress.json\').write_text(json.dumps(dict(dataset=name, chunk=[first, stop], stage=\'unet\')))\n                volume = arr[first:stop, ::1, ::4, ::4].astype(np.float32)\n                volume = np.clip((volume - low) / (high - low + 1e-6), 0.0, None).astype(np.float32)\n                inputs = torch.from_numpy(volume).unsqueeze(1).to(\'cuda\')\n                out = original.cluster_predicted_nodes(model.unet(inputs), radius=3.0)\n                width = int(out.mask.shape[1])\n                for b, frame in enumerate(range(first, stop)):\n                    mask = out.mask[b].detach().cpu().numpy()\n                    tensors = dict(zyx=out.refine_zyx[b], peak_zyx=out.peak_zyx[b],\n                        refine_logit=out.refine_logit[b], peak_logit=out.peak_logit[b],\n                        **{f\'feature{i}\': value[b] for i, value in enumerate(out.node_feature)})\n                    arrays = {key: value.detach().cpu().numpy() for key, value in tensors.items()}\n                    current = freeze_frame({key: value[mask] for key, value in arrays.items()})\n                    packed = pack_frame(current, width)\n                    assert np.array_equal(packed[\'mask\'], mask)\n                    assert all(np.array_equal(packed[key], arrays[key]) for key in FIELDS)\n                    if frame not in canonical:\n                        canonical[frame] = current\n                        input_hashes[frame] = frame_hash(current)\n                        counts[frame] = len(current[\'zyx\'])\n                        node_rows.extend(dict(t=frame, node_id=i, z=float(p[0]), y=float(p[1]), x=float(p[2])) for i, p in enumerate(current[\'zyx\']))\n                    else:\n                        assert frame == first and first > 0 and first % 20 == 0\n                        boundaries.append(dict(frame=frame, original_repeated_exact=frame_hash(current) == input_hashes[frame],\n                            canonical_count=counts[frame], repeated_count=len(current[\'zyx\']),\n                            canonical_sha256=input_hashes[frame], repeated_sha256=frame_hash(current)))\n                del out, inputs, volume, tensors, arrays, packed, current\n                torch.cuda.synchronize()\n                (output / \'progress.json\').write_text(json.dumps(dict(dataset=name, chunk=[first, stop], stage=\'linker\')))\n                for frame in range(first, stop - 1):\n                    deadline()\n                    assert len(matrices) == frame\n                    src, dst = pack_frame(canonical[frame], width), pack_frame(canonical[frame + 1], width)\n                    assert frame_hash(unpack_frame(src)) == input_hashes[frame]\n                    assert frame_hash(unpack_frame(dst)) == input_hashes[frame + 1]\n                    def gpu(array):\n                        return torch.from_numpy(array).unsqueeze(0).to(\'cuda\')\n                    logits = model.linker([gpu(src[f\'feature{i}\']) for i in range(3)],\n                        [gpu(dst[f\'feature{i}\']) for i in range(3)], gpu(src[\'zyx\']), gpu(dst[\'zyx\']),\n                        gpu(src[\'mask\']), gpu(dst[\'mask\']), (64, 64, 64)).edge_logit\n                    assert torch.isfinite(logits).all()\n                    rows = original.edge_top_cumulative_bidirectional(logits, gpu(src[\'mask\']), gpu(dst[\'mask\']),\n                        t_start=frame, no_link_logit=0, cumulative=.99)\n                    assert all(r[\'dst_id\'] == -1 or 0 <= r[\'dst_id\'] < counts[frame + 1] for r in rows)\n                    candidate_rows.extend(rows)\n                    raw = logits[0].detach().cpu().numpy()\n                    valid = raw[np.ix_(src[\'mask\'], dst[\'mask\'])].copy()\n                    assert valid.shape == (counts[frame], counts[frame + 1])\n                    matrices.append(valid.reshape(-1))\n                    pair_shapes.append(valid.shape)\n                    del logits, raw, valid, src, dst, rows\n        assert set(canonical) == set(range(frame_count)) and len(matrices) == frame_count-1\n        assert len({str(m.dtype) for m in matrices}) == 1\n        nodes = pd.DataFrame(node_rows)\n        candidates = pd.DataFrame(candidate_rows)\n        selected = original.select_top1_with_dst_correction(candidates)\n        graph, skipped = original.prediction_to_geff_graph(nodes, selected)\n        assert skipped == 0 and graph.num_nodes() == len(nodes)\n        assert not selected.groupby([\'t\', \'src_id\']).size().gt(1).any()\n        assert not selected.groupby([\'t\', \'dst_id\']).size().gt(1).any()\n        assert graph.num_edges() == len(selected)\n        sizes = np.asarray([m.size for m in matrices], dtype=np.int64)\n        offsets = np.concatenate([np.zeros(1, dtype=np.int64), np.cumsum(sizes)])\n        shape = np.asarray(pair_shapes, dtype=np.int64)\n        assert np.array_equal(sizes, shape[:, 0] * shape[:, 1])\n        arrays = dict(node_frame=nodes[\'t\'].to_numpy(dtype=np.int64), node_local_id=nodes[\'node_id\'].to_numpy(dtype=np.int64),\n            node_zyx=nodes[[\'z\', \'y\', \'x\']].to_numpy(dtype=np.float32), pair_src_counts=shape[:, 0], pair_dst_counts=shape[:, 1],\n            pair_offsets=offsets, edge_logits=np.concatenate(matrices), baseline_edges=selected[[\'t\', \'src_id\', \'dst_id\']].to_numpy(dtype=np.int64),\n            candidate_pair=candidates[\'t\'].to_numpy(dtype=np.int64), candidate_src=candidates[\'src_id\'].to_numpy(dtype=np.int64),\n            candidate_dst=candidates[\'dst_id\'].to_numpy(dtype=np.int64), candidate_rank=candidates[\'rank\'].to_numpy(dtype=np.int64),\n            candidate_prob=candidates[\'prob\'].to_numpy(dtype=np.float64), candidate_logit=candidates[\'logit\'].to_numpy(dtype=np.float64))\n        assert all(np.isfinite(value).all() for value in arrays.values())\n        payload, summary = decode_arrays(arrays)\n        (output / (name + \'.json\')).write_text(json.dumps(payload, allow_nan=False))\n        videos[name] = dict(**summary, frames=frame_count, pairs=frame_count-1,\n            unet_batches=[stop-first for first,stop in chunks],\n            candidates=len(candidates), raw_dtype=str(arrays[\'edge_logits\'].dtype),\n            repeated_boundary_exact=all(b[\'original_repeated_exact\'] for b in boundaries),\n            elapsed_seconds=time.monotonic()-clip_started)\n        del payload, canonical, nodes, candidates, selected, graph, arrays, matrices, candidate_rows, node_rows\n    assert videos\n    loader = json.loads((output / \'checkpoint_load.json\').read_text())\n    return dict(complete=True, input_split=\'test\', gt_read=False, linker_batch=1,\n        canonical_boundary_reuse=True, checkpoint_load=loader, videos=videos,\n        elapsed_seconds=time.monotonic()-started,\n        peak_cuda_allocated_bytes=int(torch.cuda.max_memory_allocated()),\n        peak_cuda_reserved_bytes=int(torch.cuda.max_memory_reserved()))\n', 'src/__init__.py': '', 'src/hengck_division_geometry.py': '"""One fixed, GT-blind geometry decoder on unfiltered canonical Hengck top1."""\nfrom collections import Counter, defaultdict\nimport math\nimport numpy as np\nfrom scipy.spatial import cKDTree\n\nSCALE = np.array([1.625, .40625, .40625])  # original voxel ZYX, not native grid\nLIMITS = dict(existing_child_um=10., parent_um=9., sister_um=14., divergence_um=2.25,\n              symmetry=.6, frame_fraction=.0076, global_fraction=.00375)\nGATES = (\'existing_child_distance\', \'parent_distance\', \'sister_distance\',\n         \'nearest_free_sister\', \'successors_present\', \'divergence\', \'symmetry\')\n\n\ndef geometry_shadow(nodes, edges, proposals):\n    """Evaluate all fixed gates, without selecting edges or accessing annotations."""\n    outgoing, incoming, frames = {}, set(), defaultdict(list)\n    for key, node in nodes.items():\n        if int(node[\'node_id\']) != key or not np.isfinite([node[a] for a in (\'z\',\'y\',\'x\')]).all():\n            raise ValueError(\'Invalid node identity/coordinates\')\n        frames[int(node[\'t\'])].append(key)\n    for edge in edges:\n        s, d = int(edge[\'source_id\']), int(edge[\'target_id\'])\n        if s not in nodes or d not in nodes or s in outgoing or d in incoming:\n            raise ValueError(\'Expected unfiltered 1:1 baseline\')\n        if nodes[d][\'t\'] != nodes[s][\'t\'] + 1:\n            raise ValueError(\'Nonconsecutive baseline edge\')\n        outgoing[s] = d\n        incoming.add(d)\n    position = {i:np.array([n[a] for a in (\'z\',\'y\',\'x\')],dtype=float)*SCALE for i,n in nodes.items()}\n    free, trees = {}, {}\n    for t, ids in frames.items():\n        free[t] = sorted(i for i in ids if i not in incoming)\n        if free[t]:\n            trees[t] = cKDTree(np.stack([position[i] for i in free[t]]))\n    distance = lambda a,b: float(np.linalg.norm(position[a]-position[b]))\n    rows, seen = [], set()\n    for proposal in proposals:\n        s,d,c = (int(proposal[k]) for k in (\'source_id\',\'target_id\',\'existing_child_id\'))\n        if (s,d) in seen:\n            raise ValueError(\'Duplicate proposal\')\n        seen.add((s,d))\n        if (s not in outgoing or outgoing[s]!=c or d not in nodes or d in incoming\n                or d==c or nodes[d][\'t\']!=nodes[s][\'t\']+1):\n            raise ValueError(\'Proposal not a free second daughter of a 1:1 parent\')\n        child_dist,parent_dist,sister_dist = distance(s,c),distance(s,d),distance(c,d)\n        _, nearest = trees[int(nodes[d][\'t\'])].query(position[c])\n        nearest_id = free[int(nodes[d][\'t\'])][int(nearest)]\n        successors = c in outgoing and d in outgoing\n        separation = distance(outgoing[c],outgoing[d]) if successors else None\n        divergence = separation-sister_dist if successors else None\n        symmetry = abs(child_dist-parent_dist)/max((child_dist+parent_dist)/2,1e-6)\n        flags = dict(existing_child_distance=child_dist<=LIMITS[\'existing_child_um\'],\n            parent_distance=parent_dist<=LIMITS[\'parent_um\'], sister_distance=sister_dist<=LIMITS[\'sister_um\'],\n            nearest_free_sister=d==nearest_id, successors_present=successors,\n            divergence=successors and divergence>=LIMITS[\'divergence_um\'], symmetry=symmetry<=LIMITS[\'symmetry\'])\n        rows.append(dict(**proposal, frame=int(nodes[s][\'t\']), child_distance_um=child_dist,\n            parent_distance_um=parent_dist, sister_distance_um=sister_dist, nearest_free_id=nearest_id,\n            grandchild_distance_um=separation, divergence_um=divergence, symmetry=symmetry,\n            rank_score=parent_dist+.15*sister_dist, gates=flags, passed=all(flags.values()),\n            first_failure=next((g for g in GATES if not flags[g]),None)))\n    stages = {\'input\':len(rows)}\n    surviving = rows\n    for gate in GATES:\n        surviving = [r for r in surviving if r[\'gates\'][gate]]\n        stages[gate] = len(surviving)\n    return rows, dict(stages=stages, first_failures=dict(Counter(r[\'first_failure\'] or \'passed\' for r in rows)),\n        source_counts=dict(Counter(int(nodes[s][\'t\']) for s in outgoing)), baseline_edges=len(edges))\n\n\ndef select_forks(rows, source_counts, baseline_edge_count):\n    """Author rank/caps; deterministic source/target tie order; never read GT."""\n    global_cap = max(1, int(round(max(1,baseline_edge_count)*LIMITS[\'global_fraction\'])))\n    by_frame = defaultdict(list)\n    for row in rows:\n        if row[\'passed\']:\n            if not math.isfinite(row[\'rank_score\']):\n                raise ValueError(\'Nonfinite rank\')\n            by_frame[row[\'frame\']].append(row)\n    used_sources,used_targets,added = set(),set(),[]\n    reasons = Counter()\n    for frame in sorted(by_frame):\n        if frame not in source_counts or source_counts[frame]<1:\n            raise ValueError(\'Missing baseline source count\')\n        frame_cap = max(1,int(round(source_counts[frame]*LIMITS[\'frame_fraction\'])))\n        count = 0\n        for row in sorted(by_frame[frame],key=lambda r:(r[\'rank_score\'],r[\'source_id\'],r[\'target_id\'])):\n            s,d = row[\'source_id\'],row[\'target_id\']\n            if len(added)>=global_cap:\n                reasons[\'global_cap\']+=1\n            elif count>=frame_cap:\n                reasons[\'frame_cap\']+=1\n            elif s in used_sources or d in used_targets:\n                reasons[\'conflict\']+=1\n            else:\n                added.append(dict(source_id=s,target_id=d,edge_prob=None))\n                used_sources.add(s)\n                used_targets.add(d)\n                count+=1\n    return added, dict(global_cap=global_cap, added=len(added), rejected=dict(reasons))\n', 'src/k3_hengck_fork_transfer.py': '"""GT-blind projection of fixed Hengck forks onto unchanged K3 nodes/edges."""\nfrom collections import Counter,defaultdict\nimport numpy as np\nfrom scipy.spatial import cKDTree\nfrom src.hengck_division_geometry import select_forks\n\nSCALE=np.array([1.625,.40625,.40625])\nRADIUS_UM=4.875\n\n\ndef mutual_mapping(k3_nodes,hengck_nodes):\n    frames=[]\n    for nodes in (k3_nodes,hengck_nodes):\n        by_frame=defaultdict(list)\n        for i,n in nodes.items():\n            if int(n[\'node_id\'])!=i: raise ValueError(\'Node identity differs\')\n            by_frame[int(n[\'t\'])].append(i)\n        frames.append(by_frame)\n    mapping,distances={},{}\n    for t in sorted(set(frames[0])&set(frames[1])):\n        ki,hi=sorted(frames[0][t]),sorted(frames[1][t])\n        def positions(nodes,ids):\n            values=np.array([[nodes[i][a] for a in (\'z\',\'y\',\'x\')] for i in ids],dtype=float)*SCALE\n            if not np.isfinite(values).all(): raise ValueError(\'Nonfinite coordinates\')\n            return values\n        kp,hp=positions(k3_nodes,ki),positions(hengck_nodes,hi)\n        # k=2 gives infinity for a missing second neighbour, preserving singleton matches.\n        hd,hj=cKDTree(kp).query(hp,k=2)\n        kd,kj=cKDTree(hp).query(kp,k=2)\n        for h in range(len(hi)):\n            k=int(hj[h,0])\n            if (hd[h,0]>RADIUS_UM or int(kj[k,0])!=h\n                    or np.isclose(hd[h,0],hd[h,1],atol=1e-9,rtol=0)\n                    or np.isclose(kd[k,0],kd[k,1],atol=1e-9,rtol=0)):\n                continue\n            mapping[hi[h]]=ki[k]\n            distances[hi[h]]=float(hd[h,0])\n    if len(set(mapping.values()))!=len(mapping): raise ValueError(\'Nonunique correspondence\')\n    return mapping,distances\n\n\ndef project_forks(k3_nodes,k3_edges,hengck_nodes,hengck_edges,hengck_added,rank_rows):\n    mapping,distances=mutual_mapping(k3_nodes,hengck_nodes)\n    ko,ho=defaultdict(set),defaultdict(set)\n    incoming=set()\n    for edge in k3_edges:\n        ko[int(edge[\'source_id\'])].add(int(edge[\'target_id\']))\n        incoming.add(int(edge[\'target_id\']))\n    for edge in hengck_edges:\n        ho[int(edge[\'source_id\'])].add(int(edge[\'target_id\']))\n    rank={(r[\'source_id\'],r[\'target_id\']):r for r in rank_rows}\n    rows,evidence,reasons=[],[],Counter()\n    seen=set()\n    for added in hengck_added:\n        hs,hd=int(added[\'source_id\']),int(added[\'target_id\'])\n        if hs in seen or len(ho[hs])!=2 or hd not in ho[hs] or (hs,hd) not in rank:\n            raise ValueError(\'Expected original fixed Hg fork and rank evidence\')\n        seen.add(hs)\n        daughters=sorted(ho[hs])\n        ksource=mapping.get(hs)\n        kchildren=[mapping.get(d) for d in daughters]\n        record=dict(hengck_mother=hs,hengck_daughters=daughters,k3_mother=ksource,k3_daughters=kchildren,\n            correspondence_distances_um=[distances.get(i) for i in (hs,*daughters)])\n        if ksource is None or any(d is None for d in kchildren): reason=\'missing_mutual_correspondence\'\n        elif len({ksource,*kchildren})!=3: raise ValueError(\'Nonunique triple\')\n        elif len(ko[ksource])!=1: reason=\'k3_mother_outdegree_not_one\'\n        elif not ko[ksource]<=set(kchildren): reason=\'k3_existing_child_disagrees\'\n        else:\n            target=next(d for d in kchildren if d not in ko[ksource])\n            if target in incoming: reason=\'k3_second_daughter_occupied\'\n            else:\n                if any(k3_nodes[d][\'t\']!=k3_nodes[ksource][\'t\']+1 for d in kchildren):\n                    raise ValueError(\'Wrong target frame\')\n                reason=\'eligible\'\n                record[\'target_id\']=target\n                rows.append(dict(source_id=ksource,target_id=target,frame=int(k3_nodes[ksource][\'t\']),\n                    passed=True,rank_score=rank[hs,hd][\'rank_score\'],hengck_mother=hs))\n        record[\'reason\']=reason\n        evidence.append(record)\n        reasons[reason]+=1\n    source_counts=Counter(int(k3_nodes[s][\'t\']) for s,d in ko.items() if len(d)==1)\n    added,selection=select_forks(rows,source_counts,len(k3_edges))\n    selected={(e[\'source_id\'],e[\'target_id\']) for e in added}\n    for record in evidence:\n        record[\'selected\']=record[\'reason\']==\'eligible\' and (record[\'k3_mother\'],record[\'target_id\']) in selected\n    return added,dict(matched_nodes=len(mapping),radius_um=RADIUS_UM,proposal_reasons=dict(reasons),\n        selection=selection,evidence=evidence)\n', 'src/fork_transfer_csv.py': '"""Append verified forks to an unchanged integer K3 CSV, no prediction repairs."""\nimport csv\nimport hashlib\nfrom collections import Counter,defaultdict\nfrom pathlib import Path\n\nCOLUMNS=[\'id\',\'dataset\',\'row_type\',\'node_id\',\'t\',\'z\',\'y\',\'x\',\'source_id\',\'target_id\']\n\n\ndef digest(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\n\ndef append_forks(source,output,additions):\n    source,output=Path(source),Path(output)\n    if source.resolve()==output.resolve() or output.exists():\n        raise ValueError(\'Never overwrite the baseline or an existing result\')\n    source_sha=digest(source)\n    nodes=defaultdict(dict)\n    incoming,outgoing,pairs=Counter(),Counter(),set()\n    rows=[]\n    with source.open(newline=\'\',encoding=\'utf-8\') as stream:\n        reader=csv.DictReader(stream)\n        if reader.fieldnames!=COLUMNS: raise ValueError(\'Unexpected CSV schema\')\n        for index,row in enumerate(reader):\n            if int(row[\'id\'])!=index: raise ValueError(\'Baseline row IDs not sequential\')\n            values={k:int(row[k]) for k in COLUMNS[3:]}\n            ds=row[\'dataset\']\n            if row[\'row_type\']==\'node\':\n                i=values[\'node_id\']\n                if i<0 or i in nodes[ds] or min(values[k] for k in (\'t\',\'z\',\'y\',\'x\'))<0:\n                    raise ValueError(\'Invalid baseline node\')\n                if values[\'source_id\']!=-1 or values[\'target_id\']!=-1: raise ValueError(\'Invalid node sentinel\')\n                nodes[ds][i]=values[\'t\']\n            elif row[\'row_type\']==\'edge\':\n                s,d=values[\'source_id\'],values[\'target_id\']\n                if any(values[k]!=-1 for k in (\'node_id\',\'t\',\'z\',\'y\',\'x\')) or (ds,s,d) in pairs:\n                    raise ValueError(\'Invalid baseline edge\')\n                pairs.add((ds,s,d)); outgoing[ds,s]+=1; incoming[ds,d]+=1\n            else: raise ValueError(\'Unknown row type\')\n            rows.append(row)\n    if set(additions)!=set(nodes): raise ValueError(\'Incomplete prediction coverage\')\n    for ds,s,d in pairs:\n        if s not in nodes[ds] or d not in nodes[ds] or nodes[ds][d]!=nodes[ds][s]+1:\n            raise ValueError(\'Invalid baseline endpoint/frame\')\n    if max(outgoing.values(),default=0)>2 or max(incoming.values(),default=0)>1:\n        raise ValueError(\'Invalid baseline degrees\')\n    new_rows=[]\n    for ds in sorted(additions):\n        for edge in additions[ds]:\n            s,d=int(edge[\'source_id\']),int(edge[\'target_id\'])\n            if (s not in nodes[ds] or d not in nodes[ds] or nodes[ds][d]!=nodes[ds][s]+1\n                    or (ds,s,d) in pairs or outgoing[ds,s]!=1 or incoming[ds,d]!=0):\n                raise ValueError(\'Only one free second daughter of an existing parent may be appended\')\n            pairs.add((ds,s,d)); outgoing[ds,s]+=1; incoming[ds,d]+=1\n            row={k:-1 for k in COLUMNS}\n            row.update(id=len(rows)+len(new_rows),dataset=ds,row_type=\'edge\',source_id=s,target_id=d)\n            new_rows.append(row)\n    if digest(source)!=source_sha: raise ValueError(\'Baseline changed during validation\')\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with output.open(\'x\',newline=\'\',encoding=\'utf-8\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=COLUMNS)\n        writer.writeheader(); writer.writerows(rows); writer.writerows(new_rows)\n    with output.open(newline=\'\',encoding=\'utf-8\') as stream:\n        restored=list(csv.DictReader(stream))\n    if restored[:len(rows)]!=rows or len(restored)!=len(rows)+len(new_rows):\n        raise ValueError(\'CSV changed original rows\')\n    if digest(source)!=source_sha: raise ValueError(\'Baseline was modified\')\n    return dict(baseline_sha256=source_sha,output_sha256=digest(output),original_rows=len(rows),\n        original_rows_values_order_and_ids_exact=True,added_edges=len(new_rows),\n        output_rows=len(restored),node_count=sum(map(len,nodes.values())),datasets=sorted(nodes))\n', 'src/k3_hengck_production_adapter.py': '"""Prediction-only conversion of frozen Hengck arrays and final integer K3 CSV."""\nimport csv\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport numpy as np\nfrom src.hengck_division_opportunity import pair_opportunities\nfrom src.hengck_division_geometry import geometry_shadow, select_forks\nfrom src.k3_hengck_fork_transfer import project_forks\nfrom src.fork_transfer_csv import append_forks\n\n\ndef decode_arrays(data):\n    coords = data[\'node_zyx\'].astype(np.float64) * [1, 4, 4]\n    frames = data[\'node_frame\']\n    counts = np.bincount(frames, minlength=len(data[\'pair_src_counts\']) + 1)\n    offsets = np.concatenate([np.zeros(1, dtype=np.int64), np.cumsum(counts)])\n    if (not np.array_equal(counts[:-1], data[\'pair_src_counts\'])\n            or not np.array_equal(counts[1:], data[\'pair_dst_counts\'])\n            or not np.array_equal(data[\'node_local_id\'], np.concatenate([np.arange(c) for c in counts]))):\n        raise ValueError(\'Canonical node identity differs\')\n    nodes = {i: dict(node_id=i, t=int(t), z=float(p[0]), y=float(p[1]), x=float(p[2]))\n             for i, (t, p) in enumerate(zip(frames, coords))}\n    edges = [dict(source_id=int(offsets[t]+i), target_id=int(offsets[t+1]+j), edge_prob=0.)\n             for t, i, j in data[\'baseline_edges\']]\n    proposals = []\n    for frame in range(len(counts)-1):\n        lo, hi = data[\'pair_offsets\'][frame:frame+2]\n        raw = data[\'edge_logits\'][lo:hi].reshape(counts[frame], counts[frame+1])\n        chosen = data[\'candidate_pair\'] == frame\n        rows = {k: data[\'candidate_\'+k][chosen] for k in (\'src\', \'dst\', \'rank\')}\n        selected = data[\'baseline_edges\']\n        pair = pair_opportunities(raw, rows, selected[selected[:, 0] == frame, 1:])\n        for r in pair[\'free_positive_options\']:\n            proposals.append(dict(source_id=int(offsets[frame]+r[\'mother\']),\n                target_id=int(offsets[frame+1]+r[\'second_daughter\']),\n                existing_child_id=int(offsets[frame+1]+r[\'existing_child\']),\n                existing_log_odds=r[\'existing_log_odds\'], second_log_odds=r[\'second_log_odds\'], bank_rank=r[\'rank\']))\n    rows, info = geometry_shadow(nodes, edges, proposals)\n    added, selection = select_forks(rows, info[\'source_counts\'], len(edges))\n    selected_pairs = {(e[\'source_id\'], e[\'target_id\']) for e in added}\n    ranks = [r for r in rows if (r[\'source_id\'], r[\'target_id\']) in selected_pairs]\n    payload = dict(nodes=list(nodes.values()), edges=edges+added, added=added, ranks=ranks)\n    summary = dict(nodes=len(nodes), baseline_edges=len(edges), forks=len(added),\n                   proposals=len(proposals), selection=selection)\n    return payload, summary\n\n\ndef transfer_csv(baseline, output, temporary, expected):\n    nodes, edges = defaultdict(dict), defaultdict(list)\n    with Path(baseline).open(newline=\'\', encoding=\'utf-8\') as stream:\n        for row in csv.DictReader(stream):\n            name = row[\'dataset\']\n            if row[\'row_type\'] == \'node\':\n                n = {k: int(row[k]) for k in (\'node_id\', \'t\', \'z\', \'y\', \'x\')}\n                if n[\'node_id\'] in nodes[name]:\n                    raise ValueError(\'Duplicate K3 node\')\n                nodes[name][n[\'node_id\']] = n\n            elif row[\'row_type\'] == \'edge\':\n                edges[name].append(dict(source_id=int(row[\'source_id\']), target_id=int(row[\'target_id\']), edge_prob=None))\n            else:\n                raise ValueError(\'Invalid K3 row\')\n    if set(nodes) != set(expected):\n        raise ValueError(\'TEST coverage differs\')\n    additions, summaries = {}, {}\n    for name in sorted(expected):\n        h = json.loads((Path(temporary)/(name+\'.json\')).read_text())\n        hn = {n[\'node_id\']: n for n in h[\'nodes\']}\n        added, info = project_forks(nodes[name], edges[name], hn, h[\'edges\'], h[\'added\'], h[\'ranks\'])\n        additions[name] = added\n        degrees = Counter(e[\'source_id\'] for e in edges[name])\n        summaries[name] = dict(nodes=len(nodes[name]), baseline_edges=len(edges[name]),\n            baseline_forks=sum(d == 2 for d in degrees.values()), added_edges=len(added),\n            matched_nodes=info[\'matched_nodes\'], proposal_reasons=info[\'proposal_reasons\'], selection=info[\'selection\'])\n    receipt = append_forks(baseline, output, additions)\n    return dict(csv=receipt, videos=summaries, radius_um=4.875, gt_read=False,\n                deepcenter_recomputed=False, late_postprocessing_repeated=False)\n', 'src/hengck_division_opportunity.py': "import numpy as np\nfrom scipy.special import logsumexp\n\ndef pair_opportunities(raw, rows, selected):\n    raw = np.asarray(raw, dtype=np.float64)\n    if raw.ndim != 2 or min(raw.shape)<1 or not np.isfinite(raw).all():\n        raise ValueError('Invalid raw pair')\n    weights = 2*raw - logsumexp(raw, axis=0, keepdims=True)\n    ns, nd = raw.shape\n    selected = np.asarray(selected)\n    if selected.ndim != 2 or selected.shape[1] != 2 or selected.dtype.kind not in 'iu':\n        raise ValueError('Invalid selected edges')\n    outgoing, owners = {}, {}\n    for i, j in selected:\n        i, j = int(i), int(j)\n        if not (0<=i<ns and 0<=j<nd) or i in outgoing or j in owners:\n            raise ValueError('Expected valid 1:1 baseline')\n        outgoing[i], owners[j] = j, i\n    bank, ranks, free, occupied = set(), {}, [], []\n    for s, d, rank in zip(rows['src'], rows['dst'], rows['rank']):\n        s, d, rank = int(s), int(d), int(rank)\n        if not 0<=s<ns or not -1<=d<nd or rank<0:\n            raise ValueError('Invalid candidate identity')\n        if (s,d) in ranks:\n            raise ValueError('Duplicate candidate')\n        ranks[s,d] = rank\n        if d < 0:\n            continue\n        bank.add((s,d))\n        child = outgoing.get(s)\n        if child is None or child == d or weights[s,child]<=0 or weights[s,d]<=0:\n            continue\n        record = dict(mother=s, existing_child=child, second_daughter=d, rank=rank,\n            existing_log_odds=float(weights[s,child]), second_log_odds=float(weights[s,d]))\n        if d in owners:\n            occupied.append(dict(**record, current_other_parent=owners[d]))\n        else:\n            free.append(record)\n    return dict(weights=weights, bank=bank, ranks=ranks, outgoing=outgoing, owners=owners,\n        free_positive_options=free, occupied_positive_options=occupied)\n", 'k3_cells.json': '[[1, "# Only change this selector when intentionally choosing the output arm.\\nHF_SUBMISSION_ARM = \\"control\\"  # \\"control\\" or \\"fork_preserve\\"\\nHF_ANCHOR_MIN_PROB = 0.80\\nHF_PRESERVE_FORKS = False\\n"], [2, "\'\'\'Biohub Harmonic Fusion\\n\\nProduction 3D lineage reconstruction with dual temporal models,\\ndual edge-feature TTA, and geometry-validated divisions.\\n\\nRecord edition.\'\'\'\\n\\nimport os\\nos.environ[\\"BIOHUB_OUTPUT_MOTION_RELINK\\"] = \\"0\\"\\nBIOHUB_PRESET = \'harmonic_v3_division_wide\'\\nBIOHUB_SCORE_AXIS = \'unscored paired diagnostic; do not inherit 0.947 from the source card\'\\n\\nos.environ[\\"BIOHUB_OUTPUT_FILTER_SHORT_TRACKS\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_DET_THRESHOLD\\"] = \\"0.965\\"\\nos.environ[\\"BIOHUB_MOTION_RELINK_LEARNED_BONUS\\"] = \'1.0\'\\nos.environ[\\"BIOHUB_ILP_APPEARANCE_WEIGHT\\"] = \\"0.0\\"\\nos.environ[\\"BIOHUB_ILP_DISAPPEARANCE_WEIGHT\\"] = \\"2\\"\\nos.environ[\\"BIOHUB_GAP_CLOSE_MAX_GAP\\"] = \\"2\\"\\nos.environ[\\"BIOHUB_GAP_CLOSE_UM\\"] = \\"5.0\\"\\nos.environ[\\"BIOHUB_GAP_DENSITY_ADAPTIVE\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_GAP_DENSITY_REFERENCE_UM\\"] = \\"6.5\\"\\nos.environ[\\"BIOHUB_GAP_DENSITY_GAIN\\"] = \\"0.040\\"\\nos.environ[\\"BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM\\"] = \\"0.125\\"\\nos.environ[\\"BIOHUB_GAP_DENSITY_NEIGHBORS\\"] = \\"3\\"\\nos.environ[\\"BIOHUB_OUTPUT_MIN_TRACK_LEN\\"] = \\"6\\"\\nos.environ[\\"BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_OUTPUT_GAP2_RECOVERY\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_SAFE_DIV_MAX_UM\\"] = \\"9.0\\"  \\n\\n\\nos.environ[\\"BIOHUB_SAFE_DIV_SISTER_MAX_UM\\"] = \\"14.0\\"  \\n\\n\\n\\nos.environ[\\"BIOHUB_SAFE_DIV_SISTER_SYMMETRY_TAU\\"] = \\"0.6\\"  \\nos.environ[\\"BIOHUB_SAFE_DIV_DIVERGE_UM\\"] = \\"2.25\\"  \\n\\n\\nos.environ[\\"BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM\\"] = \\"10.0\\"\\nos.environ[\\"BIOHUB_SAFE_DIV_FRAME_FRAC_CAP\\"] = \\"0.0076\\"\\nos.environ[\\"BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP\\"] = \\"0.00375\\"\\n\\nos.environ[\\"BIOHUB_ILP_DIVISION_WEIGHT\\"] = \\"1.2\\"     \\nos.environ[\\"BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN\\"] = \\"4\\"\\nos.environ[\\"BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB\\"] = \\"0.88\\"\\nos.environ[\\"BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM\\"] = \\"3.0\\"\\nos.environ[\\"BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC\\"] = \\"0.012\\"\\nos.environ[\\"BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS\\"] = \\"120\\"\\nos.environ[\\"BIOHUB_USE_DEEPCENTER_VETO\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_REQUIRE_DEEPCENTER_VETO\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_DEEPCENTER_EXPECTED_EPOCH\\"] = \\"2\\"\\nos.environ[\\"BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM\\"] = \\"8.5\\"\\nos.environ[\\"BIOHUB_DEEPCENTER_CHECKPOINT\\"] = \\"/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt\\"\\nos.environ[\\"BIOHUB_DEEPCENTER_GAP_VETO\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_DEEPCENTER_GAP_THRESHOLD\\"] = \\"0.25\\"\\nos.environ[\\"BIOHUB_DEEPCENTER_SAFE_DIV_VETO\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_RUN_OUTPUT_DIAGNOSTICS\\"] = \\"0\\"\\nos.environ[\\"BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT\\"] = \\"0.15\\"\\nos.environ[\\"BIOHUB_BIDIRECTIONAL_FUSION_MODE\\"] = \\"harmonic_probability\\"\\nos.environ[\\"BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION\\"] = \\"0.90\\"\\nos.environ[\\"BIOHUB_DIAGNOSTIC_ARM\\"] = \\"harmonic_association_production\\"\\nos.environ[\\"BIOHUB_VALIDATOR_N_PER_TYPE\\"] = \\"4\\"\\nos.environ[\\"BIOHUB_PPSWEEP_SELECT_MARGIN\\"] = \\"0.001\\"\\nos.environ[\\"BIOHUB_PPSWEEP_MAX_ADJ_LOSS\\"] = \\"0.0005\\"\\n\\nos.environ[\\"BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD\\"] = \\"0.25\\"\\nos.environ[\\"BIOHUB_DEEPCENTER_TTA\\"] = \\"1\\"\\nprint(\\"BIOHUB_PRESET:\\", BIOHUB_PRESET)\\nprint(\\"BIOHUB_SCORE_AXIS:\\", BIOHUB_SCORE_AXIS)"], [3, "\\nimport json as _guard_json\\nimport math as _guard_math\\nimport os as _guard_os\\n\\n_EXPECTED_NUMERIC = {\\n    \\"BIOHUB_ILP_DIVISION_WEIGHT\\": 1.2,\\n    \\"BIOHUB_DET_THRESHOLD\\": 0.965,\\n    \\"BIOHUB_ILP_APPEARANCE_WEIGHT\\": 0.0,\\n    \\"BIOHUB_ILP_DISAPPEARANCE_WEIGHT\\": 2,\\n    \\"BIOHUB_GAP_CLOSE_UM\\": 5.0,\\n    \\"BIOHUB_OUTPUT_MIN_TRACK_LEN\\": 6.0,\\n    \\"BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT\\": 0.15,\\n}\\n\\n_EXPECTED_TEXT = {\\n    \\"BIOHUB_OUTPUT_MOTION_RELINK\\": \\"0\\",\\n    \\"BIOHUB_BIDIRECTIONAL_FUSION_MODE\\": \\"harmonic_probability\\",\\n    \\"BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION\\": \\"0.90\\",\\n}\\n\\n_drift = {}\\nfor _key, _want in _EXPECTED_NUMERIC.items():\\n    _raw = _guard_os.environ.get(_key)\\n    if _raw is None:\\n        _drift[_key] = \\"missing\\"\\n        continue\\n    _got = float(_raw)\\n    if not _guard_math.isclose(_got, _want, rel_tol=0.0, abs_tol=1e-12):\\n        _drift[_key] = {\\"expected\\": _want, \\"actual\\": _got}\\n\\nfor _key, _want in _EXPECTED_TEXT.items():\\n    _got = _guard_os.environ.get(_key)\\n    if _got != _want:\\n        _drift[_key] = {\\"expected\\": _want, \\"actual\\": _got}\\n\\nif _drift:\\n    raise RuntimeError(\\n        \\"Configuration drift detected: \\" + _guard_json.dumps(_drift, sort_keys=True)\\n    )\\n\\nprint(\\"Configuration guard: PASS\\")\\nprint(\\"HF-FP-01: frozen v30 base, no train/proxy parameter selection\\")\\n"], [4, "from __future__ import annotations\\n\\nimport csv\\nimport importlib.util\\nimport json\\nimport math\\nimport os\\nimport shutil\\nimport subprocess\\nimport tempfile\\nimport zipfile\\nimport sys\\nimport time\\nfrom pathlib import Path\\n\\nimport pandas as pd\\nfrom IPython.display import display\\n\\nCOMPETITION = \\"biohub-cell-tracking-during-development\\"\\nCOMP_DIR_CANDIDATES = [\\n    Path(f\\"/kaggle/input/competitions/{COMPETITION}\\"),\\n    Path(f\\"/kaggle/input/{COMPETITION}\\"),\\n]\\nCOMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])\\n\\nTEST_DIR = COMP_DIR / \\"test\\"\\n\\nWORKING_DIR = Path(\\"/kaggle/working\\") if Path(\\"/kaggle/working\\").exists() else Path(\\".\\")\\nREPO_DIR = WORKING_DIR / \\"tracking_repo\\"\\nSUBMISSION_PATH = WORKING_DIR / \\"submission.csv\\"\\nRUN_STATS_PATH = WORKING_DIR / \\"run_stats.csv\\"\\n\\nMETHOD = \\"unet_transformer\\"\\nWEIGHTS_RELATIVE = f\\"weights/{METHOD}/split_0/edge_predictor_best.pth\\"\\nEXPERIMENT_TAG = \\"selected_101_dual_seed_near_balanced_center_confirmed_synthetic_gap\\"\\nTARGET_ARTIFACT_SLUG = os.environ.get(\\"BIOHUB_TARGET_ARTIFACT_SLUG\\", \\"biohub-tracking-support-pack-50ep-v1\\")\\nPRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(\\n    \\"BIOHUB_PRIMARY_ARTIFACT_MANIFEST\\",\\n    \\"/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json\\",\\n))\\nALLOW_ARTIFACT_FALLBACK = os.environ.get(\\"BIOHUB_ALLOW_ARTIFACT_FALLBACK\\", \\"0\\") != \\"0\\"\\n\\nDET_THRESHOLD = float(os.environ.get(\\"BIOHUB_DET_THRESHOLD\\", \\"0.99\\"))\\nUNET_BATCH_SIZE = int(os.environ.get(\\"BIOHUB_UNET_BATCH_SIZE\\", \\"4\\"))\\nUSE_ILP = os.environ.get(\\"BIOHUB_USE_ILP\\", \\"1\\") != \\"0\\"\\nILP_EDGE_WEIGHT = float(os.environ.get(\\"BIOHUB_ILP_EDGE_WEIGHT\\", \\"-1.0\\"))\\nILP_APPEARANCE_WEIGHT = float(os.environ.get(\\"BIOHUB_ILP_APPEARANCE_WEIGHT\\", \\"0.1\\"))\\nILP_DISAPPEARANCE_WEIGHT = float(os.environ.get(\\"BIOHUB_ILP_DISAPPEARANCE_WEIGHT\\", \\"0.1\\"))\\nILP_DIVISION_WEIGHT = float(os.environ.get(\\"BIOHUB_ILP_DIVISION_WEIGHT\\", \\"1.0\\"))\\n\\n\\nSLICE = \\"\\"\\n\\n\\n\\nALLOW_PIP_INSTALL = os.environ.get(\\"BIOHUB_ALLOW_PIP_INSTALL\\", \\"0\\") != \\"0\\"\\nRUN_OUTPUT_DIAGNOSTICS = os.environ.get(\\"BIOHUB_RUN_OUTPUT_DIAGNOSTICS\\", \\"1\\") != \\"0\\"\\n\\n\\nOUTPUT_EDGE_MAX_UM = float(os.environ.get(\\"BIOHUB_OUTPUT_EDGE_MAX_UM\\", \\"14.0\\"))\\nOUTPUT_ENFORCE_NEXT_FRAME = os.environ.get(\\"BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME\\", \\"1\\") != \\"0\\"\\nOUTPUT_SINGLE_PARENT_REPAIR = os.environ.get(\\"BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR\\", \\"1\\") != \\"0\\"\\nOUTPUT_SINGLE_CHILD_REPAIR = os.environ.get(\\"BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR\\", \\"0\\") != \\"0\\"\\nOUTPUT_PRUNE_ISOLATED = os.environ.get(\\"BIOHUB_OUTPUT_PRUNE_ISOLATED\\", \\"1\\") != \\"0\\"\\nOUTPUT_MOTION_RELINK = os.environ.get(\\"BIOHUB_OUTPUT_MOTION_RELINK\\", \\"1\\") != \\"0\\"\\nMOTION_RELINK_TIGHT_UM = float(os.environ.get(\\"BIOHUB_MOTION_RELINK_TIGHT_UM\\", \\"6.0\\"))\\nMOTION_RELINK_RELAXED_UM = float(os.environ.get(\\"BIOHUB_MOTION_RELINK_RELAXED_UM\\", \\"10.0\\"))\\nMOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get(\\"BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT\\", \\"0.5\\"))\\nMOTION_RELINK_LEARNED_BONUS = float(os.environ.get(\\"BIOHUB_MOTION_RELINK_LEARNED_BONUS\\", \\"0.75\\"))\\nMOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get(\\"BIOHUB_MOTION_RELINK_MAX_FRAME_NODES\\", \\"2600\\"))\\n\\nOUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get(\\"BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER\\", \\"0\\") != \\"0\\"\\nDIV_PARENT_MAX_UM = float(os.environ.get(\\"BIOHUB_DIV_PARENT_MAX_UM\\", \\"10.5\\"))\\nDIV_SISTER_MAX_UM = float(os.environ.get(\\"BIOHUB_DIV_SISTER_MAX_UM\\", \\"8.0\\"))\\nDIV_DROP_TO_SINGLE_IF_BAD = os.environ.get(\\"BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD\\", \\"1\\") != \\"0\\"\\nOUTPUT_GAP_CLOSE = os.environ.get(\\"BIOHUB_OUTPUT_GAP_CLOSE\\", \\"1\\") != \\"0\\"\\nGAP_CLOSE_MAX_GAP = int(os.environ.get(\\"BIOHUB_GAP_CLOSE_MAX_GAP\\", \\"1\\"))\\nGAP_CLOSE_UM = float(os.environ.get(\\"BIOHUB_GAP_CLOSE_UM\\", \\"6.0\\"))\\nGAP_DENSITY_ADAPTIVE = os.environ.get(\\"BIOHUB_GAP_DENSITY_ADAPTIVE\\", \\"0\\") != \\"0\\"\\nGAP_DENSITY_REFERENCE_UM = float(os.environ.get(\\"BIOHUB_GAP_DENSITY_REFERENCE_UM\\", \\"6.5\\"))\\nGAP_DENSITY_GAIN = float(os.environ.get(\\"BIOHUB_GAP_DENSITY_GAIN\\", \\"0.040\\"))\\nGAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get(\\"BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM\\", \\"0.125\\"))\\nGAP_DENSITY_NEIGHBORS = int(os.environ.get(\\"BIOHUB_GAP_DENSITY_NEIGHBORS\\", \\"3\\"))\\nGAP_CLOSE_REUSE_EXISTING = os.environ.get(\\"BIOHUB_GAP_CLOSE_REUSE_EXISTING\\", \\"1\\") != \\"0\\"\\nGAP_CLOSE_REUSE_UM = float(os.environ.get(\\"BIOHUB_GAP_CLOSE_REUSE_UM\\", \\"3.2\\"))\\nGAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get(\\"BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC\\", \\"0.05\\"))\\nGAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get(\\"BIOHUB_GAP_CLOSE_MAX_ADDED_ABS\\", \\"2000\\"))\\nGAP_REFINE_SYNTHETIC = os.environ.get(\\"BIOHUB_GAP_REFINE_SYNTHETIC\\", \\"1\\") != \\"0\\"\\nGAP_REFINE_WIN_Z = int(os.environ.get(\\"BIOHUB_GAP_REFINE_WIN_Z\\", \\"1\\"))\\nGAP_REFINE_WIN_YX = int(os.environ.get(\\"BIOHUB_GAP_REFINE_WIN_YX\\", \\"3\\"))\\nGAP_REFINE_MAX_SHIFT_UM = float(os.environ.get(\\"BIOHUB_GAP_REFINE_MAX_SHIFT_UM\\", \\"3.2\\"))\\n\\nOUTPUT_FILTER_SHORT_TRACKS = os.environ.get(\\"BIOHUB_OUTPUT_FILTER_SHORT_TRACKS\\", \\"1\\") != \\"0\\"\\nOUTPUT_MIN_TRACK_LEN = int(os.environ.get(\\"BIOHUB_OUTPUT_MIN_TRACK_LEN\\", \\"6\\"))\\nOUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get(\\"BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS\\", \\"1\\") != \\"0\\"\\nADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get(\\"BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE\\", \\"0\\") != \\"0\\"\\nSHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get(\\"BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC\\", \\"0.10\\"))\\nSHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get(\\"BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN\\", \\"4\\"))\\nSHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get(\\"BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB\\", \\"0.82\\"))\\nSHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get(\\"BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM\\", \\"3.25\\"))\\nSHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get(\\"BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC\\", \\"0.018\\"))\\nSHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get(\\"BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS\\", \\"180\\"))\\n\\nOUTPUT_LINEFIT_SMOOTH = os.environ.get(\\"BIOHUB_OUTPUT_LINEFIT_SMOOTH\\", \\"1\\") != \\"0\\"\\nOUTPUT_LINEFIT_WEIGHT = float(os.environ.get(\\"BIOHUB_OUTPUT_LINEFIT_WEIGHT\\", \\"0.8\\"))\\nOUTPUT_LINEFIT_WINDOW = int(os.environ.get(\\"BIOHUB_OUTPUT_LINEFIT_WINDOW\\", \\"2\\"))\\n\\nOUTPUT_GAP2_RECOVERY = os.environ.get(\\"BIOHUB_OUTPUT_GAP2_RECOVERY\\", \\"0\\") != \\"0\\"\\nGAP2_MAX_TOTAL_UM = float(os.environ.get(\\"BIOHUB_GAP2_MAX_TOTAL_UM\\", \\"10.2\\"))\\nGAP2_MAX_STEP_UM = float(os.environ.get(\\"BIOHUB_GAP2_MAX_STEP_UM\\", \\"4.4\\"))\\nGAP2_MAX_LINKS_FRAC = float(os.environ.get(\\"BIOHUB_GAP2_MAX_LINKS_FRAC\\", \\"0.0045\\"))\\nGAP2_MAX_LINKS_ABS = int(os.environ.get(\\"BIOHUB_GAP2_MAX_LINKS_ABS\\", \\"180\\"))\\nGAP2_REQUIRE_CONTEXT = os.environ.get(\\"BIOHUB_GAP2_REQUIRE_CONTEXT\\", \\"1\\") != \\"0\\"\\nGAP2_FRAME_FRAC_CAP = float(os.environ.get(\\"BIOHUB_GAP2_FRAME_FRAC_CAP\\", \\"0.006\\"))\\n\\nOUTPUT_SAFE_DIVISIONS = os.environ.get(\\"BIOHUB_OUTPUT_SAFE_DIVISIONS\\", \\"1\\") != \\"0\\"\\nSAFE_DIV_MAX_UM = float(os.environ.get(\\"BIOHUB_SAFE_DIV_MAX_UM\\", \\"4.7\\"))\\nSAFE_DIV_SISTER_MAX_UM = float(os.environ.get(\\"BIOHUB_SAFE_DIV_SISTER_MAX_UM\\", \\"7.2\\"))\\nSAFE_DIV_SISTER_SYMMETRY_TAU = float(os.environ.get(\\"BIOHUB_SAFE_DIV_SISTER_SYMMETRY_TAU\\", \\"0.0\\"))\\nSAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get(\\"BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM\\", \\"7.8\\"))\\nSAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get(\\"BIOHUB_SAFE_DIV_FRAME_FRAC_CAP\\", \\"0.008\\"))\\nSAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get(\\"BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP\\", \\"0.004\\"))\\n\\n\\nSAFE_DIV_DIVERGE_UM = float(os.environ.get(\\"BIOHUB_SAFE_DIV_DIVERGE_UM\\", \\"2.25\\"))\\nSAFE_DIV_REQUIRE_DIVERGENCE = os.environ.get(\\"BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE\\", \\"1\\") != \\"0\\"\\nSAFE_DIV_REQUIRE_MUTUAL_NN = os.environ.get(\\"BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN\\", \\"1\\") != \\"0\\"\\n\\n\\nUSE_DEEPCENTER_VETO = os.environ.get(\\"BIOHUB_USE_DEEPCENTER_VETO\\", \\"1\\") != \\"0\\"\\nREQUIRE_DEEPCENTER_VETO = os.environ.get(\\"BIOHUB_REQUIRE_DEEPCENTER_VETO\\", \\"1\\") != \\"0\\"\\nDEEPCENTER_MANIFEST_DEFAULT = os.environ.get(\\n    \\"BIOHUB_DEEPCENTER_MANIFEST_DEFAULT\\",\\n    \\"/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json\\",\\n)\\nDEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(\\n    \\"BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT\\",\\n    \\"/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt\\",\\n)\\nDEEPCENTER_RELATIVE = os.environ.get(\\"BIOHUB_DEEPCENTER_RELATIVE\\", \\"weights/full_frame_center/best.pt\\")\\nDEEPCENTER_GAP_VETO = os.environ.get(\\"BIOHUB_DEEPCENTER_GAP_VETO\\", \\"1\\") != \\"0\\"\\nDEEPCENTER_SAFE_DIV_VETO = os.environ.get(\\"BIOHUB_DEEPCENTER_SAFE_DIV_VETO\\", \\"1\\") != \\"0\\"\\nDEEPCENTER_GAP_THRESHOLD = float(os.environ.get(\\"BIOHUB_DEEPCENTER_GAP_THRESHOLD\\", \\"0.10\\"))\\nDEEPCENTER_EXPECTED_EPOCH = int(os.environ.get(\\"BIOHUB_DEEPCENTER_EXPECTED_EPOCH\\", \\"0\\"))\\nDEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get(\\"BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM\\", \\"0\\"))\\nDEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get(\\"BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD\\", \\"0.12\\"))\\nDEEPCENTER_SCORE_WIN_Z = int(os.environ.get(\\"BIOHUB_DEEPCENTER_SCORE_WIN_Z\\", \\"1\\"))\\nDEEPCENTER_SCORE_WIN_YX = int(os.environ.get(\\"BIOHUB_DEEPCENTER_SCORE_WIN_YX\\", \\"2\\"))\\nDEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get(\\"BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES\\", \\"8\\"))\\n\\nCONFIG_DISPLAY = {\\n    \\"experiment_tag\\": EXPERIMENT_TAG,\\n    \\"method\\": METHOD,\\n    \\"weights\\": WEIGHTS_RELATIVE,\\n    \\"target_artifact_slug\\": TARGET_ARTIFACT_SLUG,\\n    \\"primary_artifact_manifest\\": str(PRIMARY_ARTIFACT_MANIFEST),\\n    \\"allow_artifact_fallback\\": ALLOW_ARTIFACT_FALLBACK,\\n    \\"det_threshold\\": DET_THRESHOLD,\\n    \\"unet_batch_size\\": UNET_BATCH_SIZE,\\n    \\"use_ilp\\": USE_ILP,\\n    \\"ilp_edge_weight\\": ILP_EDGE_WEIGHT,\\n    \\"ilp_appearance_weight\\": ILP_APPEARANCE_WEIGHT,\\n    \\"ilp_disappearance_weight\\": ILP_DISAPPEARANCE_WEIGHT,\\n    \\"ilp_division_weight\\": ILP_DIVISION_WEIGHT,\\n    \\"slice\\": SLICE,\\n    \\"allow_pip_install\\": ALLOW_PIP_INSTALL,\\n    \\"output_edge_max_um\\": OUTPUT_EDGE_MAX_UM,\\n    \\"output_enforce_next_frame\\": OUTPUT_ENFORCE_NEXT_FRAME,\\n    \\"output_single_parent_repair\\": OUTPUT_SINGLE_PARENT_REPAIR,\\n    \\"output_single_child_repair\\": OUTPUT_SINGLE_CHILD_REPAIR,\\n    \\"output_prune_isolated\\": OUTPUT_PRUNE_ISOLATED,\\n    \\"output_motion_relink\\": OUTPUT_MOTION_RELINK,\\n    \\"motion_relink_tight_um\\": MOTION_RELINK_TIGHT_UM,\\n    \\"motion_relink_relaxed_um\\": MOTION_RELINK_RELAXED_UM,\\n    \\"motion_relink_velocity_weight\\": MOTION_RELINK_VELOCITY_WEIGHT,\\n    \\"motion_relink_learned_bonus\\": MOTION_RELINK_LEARNED_BONUS,\\n    \\"motion_relink_max_frame_nodes\\": MOTION_RELINK_MAX_FRAME_NODES,\\n    \\"output_division_geometry_filter\\": OUTPUT_DIVISION_GEOMETRY_FILTER,\\n    \\"div_parent_max_um\\": DIV_PARENT_MAX_UM,\\n    \\"div_sister_max_um\\": DIV_SISTER_MAX_UM,\\n    \\"div_drop_to_single_if_bad\\": DIV_DROP_TO_SINGLE_IF_BAD,\\n    \\"output_gap_close\\": OUTPUT_GAP_CLOSE,\\n    \\"gap_close_max_gap\\": GAP_CLOSE_MAX_GAP,\\n    \\"gap_close_effective_max_gap\\": min(GAP_CLOSE_MAX_GAP, 1),\\n    \\"gap_close_um\\": GAP_CLOSE_UM,\\n    \\"gap_density_adaptive\\": GAP_DENSITY_ADAPTIVE,\\n    \\"gap_density_reference_um\\": GAP_DENSITY_REFERENCE_UM,\\n    \\"gap_density_gain\\": GAP_DENSITY_GAIN,\\n    \\"gap_density_max_step_delta_um\\": GAP_DENSITY_MAX_STEP_DELTA_UM,\\n    \\"gap_density_neighbors\\": GAP_DENSITY_NEIGHBORS,\\n    \\"gap_close_reuse_existing\\": GAP_CLOSE_REUSE_EXISTING,\\n    \\"gap_close_reuse_um\\": GAP_CLOSE_REUSE_UM,\\n    \\"gap_close_max_added_frac\\": GAP_CLOSE_MAX_ADDED_FRAC,\\n    \\"gap_close_max_added_abs\\": GAP_CLOSE_MAX_ADDED_ABS,\\n    \\"gap_refine_synthetic\\": GAP_REFINE_SYNTHETIC,\\n    \\"gap_refine_win_z\\": GAP_REFINE_WIN_Z,\\n    \\"gap_refine_win_yx\\": GAP_REFINE_WIN_YX,\\n    \\"gap_refine_max_shift_um\\": GAP_REFINE_MAX_SHIFT_UM,\\n    \\"output_filter_short_tracks\\": OUTPUT_FILTER_SHORT_TRACKS,\\n    \\"output_min_track_len\\": OUTPUT_MIN_TRACK_LEN,\\n    \\"output_keep_division_components\\": OUTPUT_KEEP_DIVISION_COMPONENTS,\\n    \\"adaptive_short_track_rescue\\": ADAPTIVE_SHORT_TRACK_RESCUE,\\n    \\"short_track_rescue_trigger_removed_frac\\": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,\\n    \\"short_track_rescue_min_len\\": SHORT_TRACK_RESCUE_MIN_LEN,\\n    \\"short_track_rescue_min_mean_edge_prob\\": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,\\n    \\"short_track_rescue_max_mean_edge_dist_um\\": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,\\n    \\"short_track_rescue_max_nodes_frac\\": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,\\n    \\"short_track_rescue_max_nodes_abs\\": SHORT_TRACK_RESCUE_MAX_NODES_ABS,\\n    \\"output_linefit_smooth\\": OUTPUT_LINEFIT_SMOOTH,\\n    \\"output_linefit_weight\\": OUTPUT_LINEFIT_WEIGHT,\\n    \\"output_linefit_window\\": OUTPUT_LINEFIT_WINDOW,\\n    \\"output_gap2_recovery\\": OUTPUT_GAP2_RECOVERY,\\n    \\"gap2_max_total_um\\": GAP2_MAX_TOTAL_UM,\\n    \\"gap2_max_step_um\\": GAP2_MAX_STEP_UM,\\n    \\"gap2_max_links_frac\\": GAP2_MAX_LINKS_FRAC,\\n    \\"gap2_max_links_abs\\": GAP2_MAX_LINKS_ABS,\\n    \\"gap2_require_context\\": GAP2_REQUIRE_CONTEXT,\\n    \\"gap2_frame_frac_cap\\": GAP2_FRAME_FRAC_CAP,\\n    \\"output_safe_divisions\\": OUTPUT_SAFE_DIVISIONS,\\n    \\"safe_div_max_um\\": SAFE_DIV_MAX_UM,\\n    \\"safe_div_sister_max_um\\": SAFE_DIV_SISTER_MAX_UM,\\n    \\"safe_div_existing_child_max_um\\": SAFE_DIV_EXISTING_CHILD_MAX_UM,\\n    \\"safe_div_frame_frac_cap\\": SAFE_DIV_FRAME_FRAC_CAP,\\n    \\"safe_div_global_frac_cap\\": SAFE_DIV_GLOBAL_FRAC_CAP,\\n    \\"use_deepcenter_add_only_gate\\": USE_DEEPCENTER_VETO,\\n    \\"deepcenter_gap_add_gate\\": DEEPCENTER_GAP_VETO,\\n    \\"deepcenter_safe_div_add_gate\\": DEEPCENTER_SAFE_DIV_VETO,\\n    \\"deepcenter_gap_threshold\\": DEEPCENTER_GAP_THRESHOLD,\\n    \\"deepcenter_expected_epoch\\": DEEPCENTER_EXPECTED_EPOCH,\\n    \\"deepcenter_gap_confirm_min_span_um\\": DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM,\\n    \\"deepcenter_safe_div_threshold\\": DEEPCENTER_SAFE_DIV_THRESHOLD,\\n    \\"deepcenter_checkpoint_default\\": DEEPCENTER_CHECKPOINT_DEFAULT,\\n}\\n\\nprint(\\"Biohub learned UNet + node-transformer + ILP submission\\")\\nprint(\\"COMP_DIR:\\", COMP_DIR, \\"exists:\\", COMP_DIR.exists())\\nprint(\\"TEST_DIR:\\", TEST_DIR, \\"exists:\\", TEST_DIR.exists())\\nprint(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))"], [5, "import re\\n\\nos.environ.setdefault(\\"POLARS_PREFER_PKG\\", \\"32\\")\\n\\nPACKAGE_SPECS = {\\n    \\"tracksdata\\": (\\"tracksdata\\", \\"tracksdata\\"),\\n    \\"zarr\\": (\\"zarr\\", \\"zarr>=3.0.10,<4\\"),\\n    \\"pyscipopt\\": (\\"pyscipopt\\", \\"pyscipopt\\"),\\n    \\"geff\\": (\\"geff\\", \\"geff>=1.1.3.1.1\\"),\\n    \\"geff_spec\\": (\\"geff_spec\\", \\"geff-spec<1.2\\"),\\n    \\"ilpy\\": (\\"ilpy\\", \\"ilpy>=0.5.1\\"),\\n    \\"polars\\": (\\"polars\\", \\"polars>=1.36\\"),\\n    \\"blosc2\\": (\\"blosc2\\", \\"blosc2\\"),\\n    \\"dask\\": (\\"dask\\", \\"dask\\"),\\n    \\"imagecodecs\\": (\\"imagecodecs\\", \\"imagecodecs\\"),\\n    \\"skimage\\": (\\"skimage\\", \\"scikit-image>=0.24\\"),\\n    \\"pyarrow\\": (\\"pyarrow\\", \\"pyarrow\\"),\\n    \\"rustworkx\\": (\\"rustworkx\\", \\"rustworkx>=0.17.1\\"),\\n    \\"sqlalchemy\\": (\\"sqlalchemy\\", \\"sqlalchemy>=2\\"),\\n    \\"numcodecs\\": (\\"numcodecs\\", \\"numcodecs>=0.13,<0.16\\"),\\n    \\"donfig\\": (\\"donfig\\", \\"donfig>=0.8\\"),\\n    \\"google_crc32c\\": (\\"google_crc32c\\", \\"google-crc32c>=1.5\\"),\\n    \\"bidict\\": (\\"bidict\\", \\"bidict>=0.23.1\\"),\\n    \\"psygnal\\": (\\"psygnal\\", \\"psygnal>=0.14\\"),\\n    \\"rich\\": (\\"rich\\", \\"rich\\"),\\n    \\"networkx\\": (\\"networkx\\", \\"networkx>=3.2.1\\"),\\n    \\"pydantic\\": (\\"pydantic\\", \\"pydantic>=2.11\\"),\\n    \\"pydantic_core\\": (\\"pydantic_core\\", \\"pydantic-core\\"),\\n    \\"annotated_types\\": (\\"annotated_types\\", \\"annotated-types\\"),\\n    \\"typing_extensions\\": (\\"typing_extensions\\", \\"typing-extensions>=4.13\\"),\\n    \\"typing_inspection\\": (\\"typing_inspection\\", \\"typing-inspection\\"),\\n    \\"markdown_it\\": (\\"markdown_it\\", \\"markdown-it-py\\"),\\n    \\"pygments\\": (\\"pygments\\", \\"pygments\\"),\\n    \\"click\\": (\\"click\\", \\"click\\"),\\n    \\"cloudpickle\\": (\\"cloudpickle\\", \\"cloudpickle\\"),\\n    \\"fsspec\\": (\\"fsspec\\", \\"fsspec\\"),\\n    \\"partd\\": (\\"partd\\", \\"partd\\"),\\n    \\"locket\\": (\\"locket\\", \\"locket\\"),\\n    \\"toolz\\": (\\"toolz\\", \\"toolz\\"),\\n    \\"yaml\\": (\\"yaml\\", \\"pyyaml\\"),\\n    \\"ndindex\\": (\\"ndindex\\", \\"ndindex\\"),\\n    \\"msgpack\\": (\\"msgpack\\", \\"msgpack\\"),\\n    \\"numexpr\\": (\\"numexpr\\", \\"numexpr\\"),\\n    \\"deprecated\\": (\\"deprecated\\", \\"deprecated\\"),\\n    \\"wrapt\\": (\\"wrapt\\", \\"wrapt\\"),\\n    \\"imageio\\": (\\"imageio\\", \\"imageio\\"),\\n    \\"PIL\\": (\\"PIL\\", \\"pillow\\"),\\n    \\"tifffile\\": (\\"tifffile\\", \\"tifffile\\"),\\n    \\"lazy_loader\\": (\\"lazy_loader\\", \\"lazy-loader\\"),\\n    \\"tqdm\\": (\\"tqdm\\", \\"tqdm\\"),\\n}\\nEXTRA_SPECS_BY_NAME = {\\n    \\"tracksdata\\": [\\"bidict>=0.23.1\\", \\"psygnal>=0.14\\", \\"rich\\"],\\n    \\"zarr\\": [\\"donfig>=0.8\\", \\"google-crc32c>=1.5\\", \\"numcodecs>=0.13,<0.16\\"],\\n    \\"geff\\": [\\"geff-spec<1.2\\", \\"networkx>=3.2.1\\", \\"pydantic>=2.11\\", \\"numcodecs>=0.13,<0.16\\"],\\n    \\"geff_spec\\": [\\"pydantic>=2.11\\", \\"annotated-types\\", \\"pydantic-core\\", \\"typing-inspection\\"],\\n    \\"polars\\": [\\"polars-runtime-32\\"],\\n    \\"dask\\": [\\"click\\", \\"cloudpickle\\", \\"fsspec\\", \\"partd\\", \\"pyyaml\\", \\"toolz\\"],\\n    \\"partd\\": [\\"locket\\"],\\n    \\"blosc2\\": [\\"ndindex\\", \\"msgpack\\", \\"numexpr\\"],\\n    \\"numcodecs\\": [\\"deprecated\\", \\"msgpack\\", \\"wrapt\\"],\\n    \\"rich\\": [\\"markdown-it-py\\", \\"pygments\\"],\\n    \\"pydantic\\": [\\"annotated-types\\", \\"pydantic-core\\", \\"typing-extensions>=4.13\\", \\"typing-inspection\\"],\\n    \\"skimage\\": [\\"imageio\\", \\"pillow\\", \\"tifffile\\", \\"lazy-loader\\", \\"networkx\\"],\\n}\\nPIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]\\nREQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}\\nFALLBACK_ARTIFACT_SLUGS = [\\"biohub-tracking-support-pack-v1\\"]\\n\\n\\n\\nALLOW_PIP_INSTALL = os.environ.get(\\"BIOHUB_ALLOW_PIP_INSTALL\\", \\"0\\") != \\"0\\"\\n\\n\\ndef module_missing(module_name: str) -> bool:\\n    return importlib.util.find_spec(module_name) is None\\n\\n\\ndef has_model_artifact(path: Path) -> bool:\\n    has_repo_dir = (path / \\"repo\\").exists()\\n    has_weights_dir = (path / \\"weights\\" / METHOD / \\"split_0\\" / \\"edge_predictor_best.pth\\").exists()\\n    has_repo_zip = (path / \\"repo.zip\\").exists()\\n    has_weights_zip = (path / \\"weights.zip\\").exists()\\n    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)\\n\\n\\ndef artifact_manifest(path: Path) -> dict:\\n    manifest = path / \\"ARTIFACT_MANIFEST.json\\"\\n    if not manifest.exists():\\n        return {}\\n    try:\\n        return json.loads(manifest.read_text())\\n    except Exception:\\n        return {}\\n\\n\\ndef artifact_matches_target(path: Path) -> bool:\\n    if ALLOW_ARTIFACT_FALLBACK:\\n        return True\\n    manifest = artifact_manifest(path)\\n    artifact_name = str(manifest.get(\\"artifact_name\\", \\"\\"))\\n    path_text = str(path)\\n    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text\\n\\n\\ndef candidate_roots_for_slug(slug: str) -> list[Path]:\\n    return [\\n        Path(f\\"/kaggle/input/datasets/pilkwang/{slug}\\"),\\n        Path(f\\"/kaggle/input/{slug}\\"),\\n        Path(f\\"/kaggle/input/{slug}/{slug}\\"),\\n        Path(f\\"PublicNotebook/{slug}\\"),\\n    ]\\n\\n\\ndef find_artifacts_root() -> Path:\\n    candidates: list[Path] = []\\n    for env_name in [\\"BIOHUB_MODEL_ARTIFACTS\\", \\"BIOHUB_ARTIFACTS\\"]:\\n        explicit = os.environ.get(env_name, \\"\\").strip()\\n        if explicit:\\n            candidates.append(Path(explicit))\\n\\n    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)\\n    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))\\n\\n    if ALLOW_ARTIFACT_FALLBACK:\\n        for slug in FALLBACK_ARTIFACT_SLUGS:\\n            candidates.extend(candidate_roots_for_slug(slug))\\n\\n    input_root = Path(\\"/kaggle/input\\")\\n    if input_root.exists():\\n        for child in input_root.iterdir():\\n            if not child.is_dir():\\n                continue\\n            child_text = str(child)\\n            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:\\n                candidates.append(child)\\n                candidates.append(child / child.name)\\n                for grandchild in child.iterdir():\\n                    if grandchild.is_dir():\\n                        candidates.append(grandchild)\\n\\n    seen: set[Path] = set()\\n    for candidate in candidates:\\n        candidate = candidate.expanduser()\\n        if candidate in seen:\\n            continue\\n        seen.add(candidate)\\n        if has_model_artifact(candidate) and artifact_matches_target(candidate):\\n            return candidate\\n    checked = \\"\\\\n\\".join(str(path) for path in candidates[:80])\\n    raise FileNotFoundError(\\n        \\"Could not find the required model artifact. \\"\\n        f\\"Expected slug: {TARGET_ARTIFACT_SLUG}\\\\n\\"\\n        \\"Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\\\\n\\"\\n        \\"To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\\\\n\\"\\n        \\"Checked:\\\\n\\" + checked\\n    )\\n\\n\\ndef _has_package_file(path: Path) -> bool:\\n    if not path.exists() or not path.is_dir():\\n        return False\\n    patterns = (\\"*.whl\\", \\"*.tar.gz\\", \\"*.zip\\")\\n    return any(any(path.glob(pattern)) for pattern in patterns)\\n\\n\\ndef find_offline_package_dirs(artifacts: Path) -> list[Path]:\\n    candidates: list[Path] = [\\n        artifacts / \\"wheels\\",\\n        artifacts,\\n        Path(\\"/kaggle/working\\"),\\n        Path(\\"/kaggle/working/wheels\\"),\\n    ]\\n    input_root = Path(\\"/kaggle/input\\")\\n    if input_root.exists():\\n        for child in input_root.iterdir():\\n            if child.is_dir():\\n                candidates.extend([child / \\"wheels\\", child])\\n                for grandchild in child.iterdir():\\n                    if grandchild.is_dir():\\n                        candidates.extend([grandchild / \\"wheels\\", grandchild])\\n\\n    out: list[Path] = []\\n    seen: set[Path] = set()\\n    for candidate in candidates:\\n        candidate = candidate.expanduser()\\n        if candidate in seen:\\n            continue\\n        seen.add(candidate)\\n        if _has_package_file(candidate):\\n            out.append(candidate)\\n    return out\\n\\n\\ndef purge_imported_modules(package_names: list[str]) -> None:\\n    roots = {\\"tracksdata\\"}\\n    for name in package_names:\\n        if name in PACKAGE_SPECS:\\n            module = PACKAGE_SPECS[name][0]\\n            roots.add(module.split(\\".\\")[0])\\n        if name == \\"polars\\":\\n            roots.add(\\"polars\\")\\n    for root in roots:\\n        for module_name in list(sys.modules):\\n            if module_name == root or module_name.startswith(root + \\".\\"):\\n                sys.modules.pop(module_name, None)\\n\\n\\ndef polars_runtime_ready() -> bool:\\n    try:\\n        import polars as _pl\\n        from polars._plr import PySeries as _PySeries\\n\\n        _ = _PySeries\\n        return hasattr(_pl, \\"Float16\\") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64\\n    except Exception:\\n        return False\\n\\n\\ndef packages_requiring_refresh() -> list[str]:\\n    refresh: list[str] = []\\n    if not module_missing(\\"polars\\") and not polars_runtime_ready():\\n        refresh.append(\\"polars\\")\\n\\n    if not module_missing(\\"zarr\\"):\\n        try:\\n            import zarr as _zarr\\n            version_text = str(getattr(_zarr, \\"__version__\\", \\"0\\"))\\n            major = int(version_text.split(\\".\\", 1)[0])\\n            if major < 3:\\n                refresh.append(\\"zarr\\")\\n        except Exception:\\n            refresh.append(\\"zarr\\")\\n    return refresh\\n\\n\\ndef dependency_specs_for(missing: list[str]) -> list[str]:\\n    specs: list[str] = []\\n    seen: set[str] = set()\\n\\n    def add(spec: str) -> None:\\n        key = spec.lower()\\n        if key not in seen:\\n            seen.add(key)\\n            specs.append(spec)\\n\\n    for name in missing:\\n        if name in PACKAGE_SPECS:\\n            add(PACKAGE_SPECS[name][1])\\n        for spec in EXTRA_SPECS_BY_NAME.get(name, []):\\n            add(spec)\\n    return specs\\n\\n\\ndef import_failures() -> dict[str, str]:\\n    failures: dict[str, str] = {}\\n    for name, module_name in REQUIRED_MODULES.items():\\n        try:\\n            importlib.import_module(module_name)\\n        except Exception as exc:\\n            failures[name] = f\\"{type(exc).__name__}: {exc}\\"\\n    return failures\\n\\n\\ndef missing_names_from_failures(failures: dict[str, str]) -> list[str]:\\n    names: list[str] = []\\n    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}\\n    for message in failures.values():\\n        match = re.search(r\\"No module named [\'\\\\\\"]([^\'\\\\\\"]+)[\'\\\\\\"]\\", message)\\n        if match:\\n            module = match.group(1).split(\\".\\")[0]\\n        else:\\n            match = re.search(r\\"module [\'\\\\\\"]([^\'\\\\\\"]+)[\'\\\\\\"] has no attribute\\", message)\\n            if not match:\\n                continue\\n            module = match.group(1).split(\\".\\")[0]\\n        name = module_to_name.get(module)\\n        if name and name not in names:\\n            names.append(name)\\n    return names\\n\\n\\ndef install_missing_dependencies(missing: list[str], artifacts: Path) -> None:\\n    specs = dependency_specs_for(missing)\\n    force_reinstall = bool({\\"polars\\", \\"zarr\\"} & set(missing))\\n    if not specs:\\n        return\\n\\n    package_dirs = find_offline_package_dirs(artifacts)\\n    if package_dirs:\\n        offline_cmd = [sys.executable, \\"-m\\", \\"pip\\", \\"install\\", \\"--no-index\\", \\"--no-deps\\"]\\n        if force_reinstall:\\n            offline_cmd.append(\\"--force-reinstall\\")\\n        for package_dir in package_dirs:\\n            offline_cmd.extend([\\"--find-links\\", str(package_dir)])\\n        offline_cmd.extend(specs)\\n        print(\\"Installing missing packages from offline package dirs:\\", missing)\\n        print(\\"Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.\\")\\n        print(\\"Offline package dirs:\\", [str(path) for path in package_dirs])\\n        result = subprocess.run(offline_cmd, text=True, capture_output=True)\\n        if result.returncode == 0:\\n            purge_imported_modules(missing)\\n            print(\\"Offline dependency install succeeded.\\")\\n            return\\n        print(\\"Offline dependency install failed. Last pip output:\\")\\n        print((result.stdout or \\"\\")[-2000:])\\n        print((result.stderr or \\"\\")[-2000:])\\n\\n    if ALLOW_PIP_INSTALL:\\n        online_cmd = [sys.executable, \\"-m\\", \\"pip\\", \\"install\\", \\"--no-deps\\"]\\n        if force_reinstall:\\n            online_cmd.append(\\"--force-reinstall\\")\\n        online_cmd.extend(specs)\\n        print(\\"Installing missing packages from PyPI:\\", missing)\\n        result = subprocess.run(online_cmd, text=True, capture_output=True)\\n        if result.returncode == 0:\\n            purge_imported_modules(missing)\\n            print(\\"PyPI dependency install succeeded.\\")\\n            return\\n        print(\\"PyPI dependency install failed. Last pip output:\\")\\n        print((result.stdout or \\"\\")[-2000:])\\n        print((result.stderr or \\"\\")[-2000:])\\n\\n    command = \\"pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs\\"\\n    raise ImportError(\\n        \\"Missing required packages or dependency wheels: \\" + \\", \\".join(missing) + \\"\\\\n\\"\\n        \\"Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\\\\n\\"\\n        + command + \\"\\\\n\\"\\n        \\"Do not quote zarr>=3.0.10,<4 in Kaggle dependency input.\\"\\n    )\\n\\n\\ndef ensure_dependencies(artifacts: Path) -> None:\\n    for _ in range(5):\\n        refresh = packages_requiring_refresh()\\n        if refresh:\\n            install_missing_dependencies(refresh, artifacts)\\n            continue\\n\\n        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]\\n        if missing:\\n            install_missing_dependencies(missing, artifacts)\\n            continue\\n\\n        failures = import_failures()\\n        if not failures:\\n            print(\\"Required graph/Zarr/ILP packages import successfully.\\")\\n            return\\n\\n        missing_from_import = missing_names_from_failures(failures)\\n        if missing_from_import:\\n            install_missing_dependencies(missing_from_import, artifacts)\\n            continue\\n\\n        raise ImportError(\\n            \\"Required packages are present but failed to import. \\"\\n            \\"This may indicate a binary dependency mismatch in the live notebook kernel. \\"\\n            \\"Keep Kaggle dependency input empty and attach the wheels artifact.\\\\n\\"\\n            + json.dumps(failures, indent=2)\\n        )\\n\\n    failures = import_failures()\\n    raise ImportError(\\n        \\"Dependency recovery did not converge after repeated offline installs. \\"\\n        \\"The attached support artifact may be missing wheels.\\\\n\\"\\n        + json.dumps(failures, indent=2)\\n    )\\n\\n\\ndef remove_path(path: Path) -> None:\\n    if path.is_symlink() or path.is_file():\\n        path.unlink()\\n    elif path.exists():\\n        shutil.rmtree(path)\\n\\n\\ndef copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:\\n    remove_path(dst)\\n    if src_dir.exists() and src_dir.is_dir():\\n        shutil.copytree(src_dir, dst)\\n        return\\n    if src_zip.exists() and src_zip.is_file():\\n        dst.mkdir(parents=True, exist_ok=True)\\n        with zipfile.ZipFile(src_zip) as zf:\\n            zf.extractall(dst)\\n        return\\n    raise FileNotFoundError(f\\"Missing source tree or zip: {src_dir} / {src_zip}\\")\\n\\n\\ndef link_or_copy_tree(src: Path, dst: Path) -> None:\\n    remove_path(dst)\\n    try:\\n        os.symlink(src, dst, target_is_directory=True)\\n    except Exception:\\n        shutil.copytree(src, dst)\\n\\n\\ndef materialize_inference_repo(artifacts: Path) -> None:\\n    copy_or_extract_tree(artifacts / \\"repo\\", artifacts / \\"repo.zip\\", REPO_DIR)\\n\\n    weights_src = artifacts / \\"weights\\"\\n    weights_zip = artifacts / \\"weights.zip\\"\\n    weights_dst = REPO_DIR / \\"weights\\"\\n    if weights_src.exists() and weights_src.is_dir():\\n        link_or_copy_tree(weights_src, weights_dst)\\n    elif weights_zip.exists() and weights_zip.is_file():\\n        remove_path(weights_dst)\\n        weights_dst.mkdir(parents=True, exist_ok=True)\\n        with zipfile.ZipFile(weights_zip) as zf:\\n            zf.extractall(weights_dst)\\n    else:\\n        raise FileNotFoundError(f\\"Missing weights tree or zip under {artifacts}\\")\\n\\n    required = [\\n        REPO_DIR / \\"scripts\\" / \\"predict_unet_transformer.py\\",\\n        REPO_DIR / WEIGHTS_RELATIVE,\\n    ]\\n    missing = [str(path) for path in required if not path.exists()]\\n    if missing:\\n        raise FileNotFoundError(\\"Materialized inference repo is incomplete:\\\\n\\" + \\"\\\\n\\".join(missing))\\n    print(\\"Inference repo:\\", REPO_DIR)\\n    print(\\"Weights:\\", REPO_DIR / WEIGHTS_RELATIVE)\\n\\n\\nARTIFACTS = find_artifacts_root()\\nprint(\\"ARTIFACTS:\\", ARTIFACTS)\\nprint(\\"Has offline wheels:\\", (ARTIFACTS / \\"wheels\\").exists())\\nmanifest_info = artifact_manifest(ARTIFACTS)\\nif manifest_info:\\n    print(\\"Artifact name:\\", manifest_info.get(\\"artifact_name\\"))\\n    print(\\"Weight sha256:\\", manifest_info.get(\\"model\\", {}).get(\\"weight_sha256\\"))\\n    print(\\"Weight path:\\", manifest_info.get(\\"model\\", {}).get(\\"weight_path\\"))\\n    _expected_primary_sha256 = \\"12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771\\"\\n    _actual_primary_sha256 = str(manifest_info.get(\\"model\\", {}).get(\\"weight_sha256\\", \\"\\"))\\n    if _actual_primary_sha256 != _expected_primary_sha256:\\n        raise RuntimeError(\\n            \\"Primary model checksum mismatch: \\"\\n            f\\"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or \'missing\'}\\"\\n        )\\n\\nensure_dependencies(ARTIFACTS)\\nmaterialize_inference_repo(ARTIFACTS)\\n\\n\\n\\n\\nimport hashlib as _integrity_hashlib\\n\\n_support_expected_sha256 = {\\n    \\"scripts/augmentations.py\\": \\"13db09817bf492f8d0f710a0a4d09776320b262060167055090a303fc6057f4e\\",\\n    \\"scripts/dataspec.py\\": \\"e69bf952fb985477ac50ff8598a35020c95d20a035a09b81ab4056e655dd311f\\",\\n    \\"scripts/evaluate.py\\": \\"614813cc51c3581c6ccda4bb20725a19da8ecac4a27620654bfca58319cffa3c\\",\\n    \\"scripts/predict_unet_transformer.py\\": \\"c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9\\",\\n    \\"scripts/train_unet_transformer.py\\": \\"c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea\\",\\n    \\"src/biohub_tracking/__init__.py\\": \\"26a18d8da84e40da73281a48ebc3017d847a2e57431ab63e8629d2109e6e8571\\",\\n    \\"src/biohub_tracking/division_metrics.py\\": \\"d1cf1e0a43009d02174f1699ce2aa28458a2220ac4b521731d3bcf31cf8c76be\\",\\n    \\"src/biohub_tracking/img_proc.py\\": \\"00e8ef0adc8b39f1aaaa547ea6197b906bf9e8c009e339d3e95f8f8dbf31be3f\\",\\n    \\"src/biohub_tracking/io.py\\": \\"efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd\\",\\n    \\"src/biohub_tracking/metrics.py\\": \\"31baf45b54c78f68bab4f65dd8f4b38bca702abb644171c6df7c46cdeef55d83\\",\\n    \\"src/biohub_tracking/models/__init__.py\\": \\"ab7587ef79856bae50d24b62e5805092d0459ee1c586522b763f9ef70c093e1d\\",\\n    \\"src/biohub_tracking/models/simple_node_transformer.py\\": \\"b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e\\",\\n    \\"src/biohub_tracking/models/temporal_unet.py\\": \\"d809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac\\"\\n}\\n_support_expected_manifest_sha256 = \\"978b626d1fd1e7397435a437dfe68691defe1572fc3c20e61012d7c9b52ed029\\"\\n_primary_expected_sha256 = \\"12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771\\"\\n_deepcenter_expected_sha256 = \\"8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0\\"  \\n\\n\\ndef _integrity_sha256_file(path: Path) -> str:\\n    digest = _integrity_hashlib.sha256()\\n    with path.open(\\"rb\\") as stream:\\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\n_support_materialized_paths = {\\n    path.relative_to(REPO_DIR).as_posix(): path\\n    for path in REPO_DIR.rglob(\\"*.py\\")\\n}\\n_support_actual_names = set(_support_materialized_paths)\\n_support_expected_names = set(_support_expected_sha256)\\nif _support_actual_names != _support_expected_names:\\n    raise RuntimeError({\\n        \\"support_repo_python_files_missing\\": sorted(\\n            _support_expected_names - _support_actual_names\\n        ),\\n        \\"support_repo_python_files_extra\\": sorted(\\n            _support_actual_names - _support_expected_names\\n        ),\\n    })\\n_support_actual_sha256 = {\\n    relative: _integrity_sha256_file(_support_materialized_paths[relative])\\n    for relative in sorted(_support_materialized_paths)\\n}\\nif _support_actual_sha256 != _support_expected_sha256:\\n    raise RuntimeError({\\n        \\"support_repo_python_checksum_mismatch\\": {\\n            relative: {\\n                \\"expected\\": _support_expected_sha256[relative],\\n                \\"actual\\": _support_actual_sha256[relative],\\n            }\\n            for relative in sorted(_support_expected_sha256)\\n            if _support_actual_sha256[relative]\\n            != _support_expected_sha256[relative]\\n        }\\n    })\\n_support_manifest_bytes = \\"\\".join(\\n    f\\"{_support_actual_sha256[relative]}  {relative}\\\\n\\"\\n    for relative in sorted(_support_actual_sha256)\\n).encode(\\"utf-8\\")\\n_support_actual_manifest_sha256 = _integrity_hashlib.sha256(\\n    _support_manifest_bytes\\n).hexdigest()\\nif _support_actual_manifest_sha256 != _support_expected_manifest_sha256:\\n    raise RuntimeError(\\n        \\"Support repo manifest checksum mismatch: \\"\\n        f\\"expected {_support_expected_manifest_sha256}, \\"\\n        f\\"got {_support_actual_manifest_sha256}\\"\\n    )\\n\\n_primary_materialized_path = REPO_DIR / WEIGHTS_RELATIVE\\n_primary_actual_sha256 = _integrity_sha256_file(_primary_materialized_path)\\nif _primary_actual_sha256 != _primary_expected_sha256:\\n    raise RuntimeError(\\n        \\"Materialized primary model checksum mismatch: \\"\\n        f\\"expected {_primary_expected_sha256}, got {_primary_actual_sha256}\\"\\n    )\\n\\n_deepcenter_candidate_strings = [\\n    os.environ.get(\\"BIOHUB_DEEPCENTER_CHECKPOINT\\", \\"\\").strip(),\\n    \\"/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/\\"\\n    \\"full_frame_center/best.pt\\",\\n    \\"/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/\\"\\n    \\"weights/full_frame_center/best.pt\\",\\n]\\n_deepcenter_candidates = []\\nfor _candidate_string in _deepcenter_candidate_strings:\\n    if not _candidate_string:\\n        continue\\n    _candidate_path = Path(_candidate_string)\\n    if _candidate_path not in _deepcenter_candidates:\\n        _deepcenter_candidates.append(_candidate_path)\\n_deepcenter_materialized_path = next(\\n    (path for path in _deepcenter_candidates if path.is_file()),\\n    None,\\n)\\nif _deepcenter_materialized_path is None:\\n    raise FileNotFoundError({\\n        \\"missing_deepcenter_checkpoint\\": [str(path) for path in _deepcenter_candidates]\\n    })\\n_deepcenter_actual_sha256 = _integrity_sha256_file(\\n    _deepcenter_materialized_path\\n)\\nif _deepcenter_actual_sha256 != _deepcenter_expected_sha256:\\n    raise RuntimeError(\\n        \\"DeepCenter checkpoint checksum mismatch: \\"\\n        f\\"expected {_deepcenter_expected_sha256}, \\"\\n        f\\"got {_deepcenter_actual_sha256}\\"\\n    )\\nos.environ[\\"BIOHUB_DEEPCENTER_CHECKPOINT\\"] = str(\\n    _deepcenter_materialized_path\\n)\\n\\nprint(\\"Support repo Python manifest SHA256:\\", _support_actual_manifest_sha256)\\nprint(\\"Primary materialized SHA256:\\", _primary_actual_sha256)\\nprint(\\"DeepCenter materialized SHA256:\\", _deepcenter_actual_sha256)\\n\\n\\n\\nimport hashlib as _hashlib\\n\\n_secondary_manifest_explicit = Path(os.environ.get(\\n    \\"BIOHUB_SECONDARY_ARTIFACT_MANIFEST\\",\\n    \\"/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/ARTIFACT_MANIFEST.json\\",\\n))\\n_secondary_expected_sha256 = \\"9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f\\"\\n_secondary_slug = \\"biohub-temporal-unet3d-seed314159-v1\\"\\n\\n\\ndef _find_secondary_artifact_root() -> tuple[Path, dict]:\\n    candidates = [\\n        _secondary_manifest_explicit,\\n        Path(f\\"/kaggle/input/{_secondary_slug}/ARTIFACT_MANIFEST.json\\"),\\n        Path(f\\"/kaggle/input/datasets/pilkwang/{_secondary_slug}/ARTIFACT_MANIFEST.json\\"),\\n    ]\\n    input_root = Path(\\"/kaggle/input\\")\\n    if input_root.exists():\\n        candidates.extend(input_root.rglob(\\"ARTIFACT_MANIFEST.json\\"))\\n\\n    seen = set()\\n    for manifest_path in candidates:\\n        manifest_path = manifest_path.expanduser()\\n        if manifest_path in seen or not manifest_path.is_file():\\n            continue\\n        seen.add(manifest_path)\\n        try:\\n            info = json.loads(manifest_path.read_text())\\n        except Exception:\\n            continue\\n        sha256 = str(info.get(\\"model\\", {}).get(\\"weight_sha256\\", \\"\\"))\\n        if sha256 == _secondary_expected_sha256:\\n            return manifest_path.parent, info\\n    raise FileNotFoundError(\\n        \\"Could not find the independent-seed artifact with weight SHA256 \\"\\n        + _secondary_expected_sha256\\n    )\\n\\n\\nSECONDARY_ARTIFACTS, secondary_manifest_info = _find_secondary_artifact_root()\\nSECONDARY_WEIGHTS_ROOT = WORKING_DIR / \\"secondary_seed_weights\\"\\ncopy_or_extract_tree(\\n    SECONDARY_ARTIFACTS / \\"weights\\",\\n    SECONDARY_ARTIFACTS / \\"weights.zip\\",\\n    SECONDARY_WEIGHTS_ROOT,\\n)\\nSECONDARY_WEIGHTS_PATH = (\\n    SECONDARY_WEIGHTS_ROOT\\n    / \\"unet_transformer\\"\\n    / \\"split_0\\"\\n    / \\"edge_predictor_best.pth\\"\\n)\\nSECONDARY_CONFIG_PATH = SECONDARY_WEIGHTS_PATH.parent / \\"config.json\\"\\nfor _required_secondary_path in (SECONDARY_WEIGHTS_PATH, SECONDARY_CONFIG_PATH):\\n    if not _required_secondary_path.is_file():\\n        raise FileNotFoundError(f\\"Missing secondary model file: {_required_secondary_path}\\")\\n\\n\\ndef _sha256_file(path: Path) -> str:\\n    digest = _hashlib.sha256()\\n    with path.open(\\"rb\\") as handle:\\n        for block in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\n_secondary_actual_sha256 = _sha256_file(SECONDARY_WEIGHTS_PATH)\\nif _secondary_actual_sha256 != _secondary_expected_sha256:\\n    raise RuntimeError(\\n        \\"Secondary model checksum mismatch: \\"\\n        f\\"expected {_secondary_expected_sha256}, got {_secondary_actual_sha256}\\"\\n    )\\n\\nos.environ[\\"BIOHUB_SECONDARY_WEIGHTS\\"] = str(SECONDARY_WEIGHTS_PATH)\\nos.environ[\\"BIOHUB_SECONDARY_EDGE_WEIGHT\\"] = \\"0.15\\"\\nprint(\\"Secondary artifact:\\", SECONDARY_ARTIFACTS)\\nprint(\\"Secondary weight:\\", SECONDARY_WEIGHTS_PATH)\\nprint(\\"Secondary SHA256:\\", _secondary_actual_sha256)\\nprint(\\"Secondary edge-logit weight:\\", os.environ[\\"BIOHUB_SECONDARY_EDGE_WEIGHT\\"])\\n\\nos.environ[\\"BIOHUB_SECONDARY_DETECTION_WEIGHT\\"] = \\"0.80\\"  \\nos.environ[\\"BIOHUB_SECONDARY_LINK_MODE\\"] = \\"low_margin_consensus\\"\\nos.environ[\\"BIOHUB_SECONDARY_MIX_TEMPERATURE\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_SECONDARY_LOW_MARGIN_MAX\\"] = \\"0.35\\"\\nos.environ[\\"BIOHUB_DUAL_SEED_EDGE_THRESHOLD\\"] = \\"0.48\\"\\n\\n_runtime_integrity_receipt = {\\n    \\"status\\": \\"complete_label_free_runtime_integrity\\",\\n    \\"verified_before_dynamic_source_patch\\": True,\\n    \\"support_repo_python_file_count\\": len(_support_actual_sha256),\\n    \\"support_repo_python_sha256\\": _support_actual_sha256,\\n    \\"support_repo_python_manifest_sha256\\": _support_actual_manifest_sha256,\\n    \\"checkpoint_sha256\\": {\\n        \\"primary\\": _primary_actual_sha256,\\n        \\"secondary\\": _secondary_actual_sha256,\\n        \\"deepcenter\\": _deepcenter_actual_sha256,\\n    },\\n    \\"materialized_paths\\": {\\n        \\"primary\\": str(_primary_materialized_path),\\n        \\"secondary\\": str(SECONDARY_WEIGHTS_PATH),\\n        \\"deepcenter\\": str(_deepcenter_materialized_path),\\n    },\\n    \\"ground_truth_accessed\\": False,\\n}\\n_runtime_integrity_receipt_path = (\\n    WORKING_DIR / \\"bidirectional_production_runtime_integrity.json\\"\\n)\\n_runtime_integrity_receipt_path.write_text(\\n    json.dumps(_runtime_integrity_receipt, indent=2, sort_keys=True) + \\"\\\\n\\",\\n    encoding=\\"utf-8\\",\\n)\\nprint(\\"Runtime integrity receipt:\\", _runtime_integrity_receipt_path)\\n"], [6, "\\nimport torch as _torch\\n\\nif not _torch.cuda.is_available():\\n    raise RuntimeError(\\n        \\"CUDA GPU is required for this notebook. Enable a Kaggle GPU accelerator and commit again.\\"\\n    )\\nprint(\\"CUDA device:\\", _torch.cuda.get_device_name(0))\\n\\n\\n_ps = REPO_DIR / \\"scripts\\" / \\"predict_unet_transformer.py\\"\\n_s = _ps.read_text()\\n_old = \\"\\"\\"        if cfg.det_tta:\\n            tta_flips = [(-1,), (-2,), (-2, -1)]\\n            for dims in tta_flips:\\n                imgs_flip = imgs.flip(dims)\\n                _, det_flip = model.encode(imgs_flip)\\n                for f in range(W):\\n                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)\\n                del imgs_flip, det_flip\\n            for f in range(W):\\n                det_logits[f] = det_logits[f] / 4\\"\\"\\"\\n_new = \\"\\"\\"        if cfg.det_tta:\\n            _nv = 1\\n            for dims in [(-1,), (-2,), (-2, -1)]:\\n                imgs_flip = imgs.flip(dims)\\n                _, det_flip = model.encode(imgs_flip)\\n                for f in range(W):\\n                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)\\n                del imgs_flip, det_flip\\n                _nv += 1\\n            for _k in (1, 3):\\n                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\\n                _, det_rot = model.encode(imgs_rot)\\n                for f in range(W):\\n                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))\\n                del imgs_rot, det_rot\\n                _nv += 1\\n            imgs_t = imgs.transpose(-1, -2)\\n            _, det_t = model.encode(imgs_t)\\n            for f in range(W):\\n                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)\\n            del imgs_t, det_t\\n            _nv += 1\\n            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)\\n            _, det_at = model.encode(imgs_at)\\n            for f in range(W):\\n                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))\\n            del imgs_at, det_at\\n            _nv += 1\\n            for f in range(W):\\n                det_logits[f] = det_logits[f] / _nv\\"\\"\\"\\nif _old in _s:\\n    _ps.write_text(_s.replace(_old, _new))\\n    print(\\"TTA patch applied (400ep spatial D4-style)\\")\\nelse:\\n    print(\\"TTA WARNING: block not found - using default 4-way\\")\\n\\n\\n_s = _ps.read_text()\\n_ensemble_replacements = [\\n    (\'    downsample: tuple[int, ...] = (1, 4, 4),\\\\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:\', \'    downsample: tuple[int, ...] = (1, 4, 4),\\\\n    secondary_model: UNetNodeTransformer | None = None,\\\\n    secondary_edge_weight: float = 0.0,\\\\n    secondary_detection_weight: float = 0.0,\\\\n    secondary_link_mode: str = \\"raw\\",\\\\n    secondary_mix_temperature: float = 1.0,\\\\n    secondary_low_margin_max: float = 0.2,\\\\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:\'),\\n    (\'            for f in range(W):\\\\n                det_logits[f] = det_logits[f] / _nv\\\\n\\\\n        del imgs\', \'            for f in range(W):\\\\n                det_logits[f] = det_logits[f] / _nv\\\\n\\\\n        secondary_unet_out = None\\\\n        if secondary_model is not None:\\\\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\\\\n\\\\n            if secondary_detection_weight > 0.0:\\\\n                if cfg.det_tta:\\\\n                    _secondary_nv = 1\\\\n                    for dims in [(-1,), (-2,), (-2, -1)]:\\\\n                        secondary_imgs_flip = imgs.flip(dims)\\\\n                        _, secondary_det_flip = secondary_model.encode(secondary_imgs_flip)\\\\n                        for f in range(W):\\\\n                            secondary_det_logits[f] = (\\\\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\\\\n                            )\\\\n                        del secondary_imgs_flip, secondary_det_flip\\\\n                        _secondary_nv += 1\\\\n                    for _k in (1, 3):\\\\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\\\\n                        _, secondary_det_rot = secondary_model.encode(secondary_imgs_rot)\\\\n                        for f in range(W):\\\\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\\\\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\\\\n                            )\\\\n                        del secondary_imgs_rot, secondary_det_rot\\\\n                        _secondary_nv += 1\\\\n                    secondary_imgs_t = imgs.transpose(-1, -2)\\\\n                    _, secondary_det_t = secondary_model.encode(secondary_imgs_t)\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = (\\\\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\\\\n                        )\\\\n                    del secondary_imgs_t, secondary_det_t\\\\n                    _secondary_nv += 1\\\\n                    secondary_imgs_at = torch.rot90(\\\\n                        imgs, 1, dims=(-2, -1)\\\\n                    ).transpose(-1, -2)\\\\n                    _, secondary_det_at = secondary_model.encode(secondary_imgs_at)\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\\\\n                            secondary_det_at[f].transpose(-1, -2),\\\\n                            -1,\\\\n                            dims=(-2, -1),\\\\n                        )\\\\n                    del secondary_imgs_at, secondary_det_at\\\\n                    _secondary_nv += 1\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\\\\n\\\\n                for f in range(W):\\\\n                    primary_det = det_logits[f]\\\\n                    secondary_det = secondary_det_logits[f]\\\\n                    primary_mean = primary_det.mean()\\\\n                    secondary_mean = secondary_det.mean()\\\\n                    primary_scale = primary_det.float().std(unbiased=False).clamp_min(1e-4)\\\\n                    secondary_scale = secondary_det.float().std(unbiased=False).clamp_min(1e-4)\\\\n                    scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\\\\n                    secondary_det_aligned = (\\\\n                        (secondary_det - secondary_mean) * scale_ratio + primary_mean\\\\n                    )\\\\n                    det_logits[f] = (\\\\n                        (1.0 - secondary_detection_weight) * primary_det\\\\n                        + secondary_detection_weight * secondary_det_aligned\\\\n                    )\\\\n\\\\n            del secondary_det_logits\\\\n\\\\n        del imgs\'),\\n    (\'            edge_logits_pair = model.predict_edges(\\\\n                unet_feat_src, unet_feat_tgt,\\\\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\\\\n                p_pos_src, p_pos_tgt,\\\\n                p_mask_src, p_mask_tgt,\\\\n            )  # (1, n_src, n_tgt)\\\\n\\\\n            raw = edge_logits_pair[0]\', \'            edge_logits_pair = model.predict_edges(\\\\n                unet_feat_src, unet_feat_tgt,\\\\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\\\\n                p_pos_src, p_pos_tgt,\\\\n                p_mask_src, p_mask_tgt,\\\\n            )  # (1, n_src, n_tgt)\\\\n\\\\n            if secondary_model is not None:\\\\n                if secondary_unet_out is None:\\\\n                    raise RuntimeError(\\"Secondary model is loaded but its feature map is missing\\")\\\\n                secondary_feat_src = secondary_model._index_features(\\\\n                    secondary_unet_out[:, f_idx], p_coords_src, p_mask_src,\\\\n                )\\\\n                secondary_feat_tgt = secondary_model._index_features(\\\\n                    secondary_unet_out[:, f_idx + 1], p_coords_tgt, p_mask_tgt,\\\\n                )\\\\n                secondary_logits_pair = secondary_model.predict_edges(\\\\n                    secondary_feat_src, secondary_feat_tgt,\\\\n                    p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\\\\n                    p_pos_src, p_pos_tgt,\\\\n                    p_mask_src, p_mask_tgt,\\\\n                )\\\\n\\\\n                if secondary_link_mode == \\"raw\\":\\\\n                    secondary_for_mix = secondary_logits_pair\\\\n                    blend_weight = secondary_edge_weight\\\\n                elif secondary_link_mode in {\\\\n                    \\"calibrated\\", \\"adaptive\\", \\"low_margin_consensus\\"\\\\n                }:\\\\n                    primary_center = edge_logits_pair.mean(dim=1, keepdim=True)\\\\n                    primary_scale = edge_logits_pair.float().std(\\\\n                        dim=1, keepdim=True, unbiased=False\\\\n                    ).clamp_min(1e-4)\\\\n                    secondary_center = secondary_logits_pair.mean(dim=1, keepdim=True)\\\\n                    secondary_scale = secondary_logits_pair.float().std(\\\\n                        dim=1, keepdim=True, unbiased=False\\\\n                    ).clamp_min(1e-4)\\\\n                    secondary_scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\\\\n                    secondary_for_mix = (\\\\n                        (secondary_logits_pair - secondary_center) * secondary_scale_ratio\\\\n                        + primary_center\\\\n                    )\\\\n                    if secondary_link_mode == \\"calibrated\\":\\\\n                        blend_weight = secondary_edge_weight\\\\n                    elif secondary_link_mode == \\"adaptive\\":\\\\n                        if n_src >= 2:\\\\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\\\\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\\\\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\\\\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\\\\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\\\\n                            secondary_margin = secondary_top2.values[0] - secondary_top2.values[1]\\\\n                            local_weight = (\\\\n                                secondary_edge_weight + secondary_margin - primary_margin\\\\n                            ).clamp(0.15, 0.75)\\\\n                            same_parent = primary_top2.indices[0].eq(\\\\n                                secondary_top2.indices[0]\\\\n                            )\\\\n                            local_weight = torch.where(\\\\n                                same_parent,\\\\n                                torch.maximum(\\\\n                                    local_weight,\\\\n                                    torch.full_like(local_weight, secondary_edge_weight),\\\\n                                ),\\\\n                                local_weight,\\\\n                            )\\\\n                            blend_weight = local_weight.view(1, 1, -1)\\\\n                        else:\\\\n                            blend_weight = secondary_edge_weight\\\\n                    else:\\\\n                        if n_src >= 2:\\\\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\\\\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\\\\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\\\\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\\\\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\\\\n                            same_parent = primary_top2.indices[0].eq(\\\\n                                secondary_top2.indices[0]\\\\n                            )\\\\n                            uncertainty = (\\\\n                                (secondary_low_margin_max - primary_margin)\\\\n                                / secondary_low_margin_max\\\\n                            ).clamp(0.0, 1.0)\\\\n                            local_weight = secondary_edge_weight * uncertainty\\\\n                            local_weight = torch.where(\\\\n                                same_parent,\\\\n                                local_weight,\\\\n                                torch.zeros_like(local_weight),\\\\n                            )\\\\n                            blend_weight = local_weight.view(1, 1, -1)\\\\n                        else:\\\\n                            blend_weight = 0.0\\\\n                else:\\\\n                    raise ValueError(f\\"Unsupported secondary link mode: {secondary_link_mode}\\")\\\\n\\\\n                edge_logits_pair = (\\\\n                    (1.0 - blend_weight) * edge_logits_pair\\\\n                    + blend_weight * secondary_for_mix\\\\n                )\\\\n                if secondary_mix_temperature != 1.0:\\\\n                    mixed_center = edge_logits_pair.mean(dim=1, keepdim=True)\\\\n                    edge_logits_pair = mixed_center + (\\\\n                        edge_logits_pair - mixed_center\\\\n                    ) / secondary_mix_temperature\\\\n\\\\n            raw = edge_logits_pair[0]\'),\\n    (\'        del unet_out\\\\n\', \'        del unet_out\\\\n        if secondary_unet_out is not None:\\\\n            del secondary_unet_out\\\\n\'),\\n    (\'    model, window_size, downsample = load_model(weights_path, device)\\\\n    print(\', \'    model, window_size, downsample = load_model(weights_path, device)\\\\n\\\\n    secondary_model = None\\\\n    secondary_weights_text = os.environ.get(\\"BIOHUB_SECONDARY_WEIGHTS\\", \\"\\").strip()\\\\n    secondary_edge_weight = float(os.environ.get(\\"BIOHUB_SECONDARY_EDGE_WEIGHT\\", \\"0\\"))\\\\n    secondary_detection_weight = float(\\\\n        os.environ.get(\\"BIOHUB_SECONDARY_DETECTION_WEIGHT\\", \\"0\\")\\\\n    )\\\\n    secondary_link_mode = os.environ.get(\\"BIOHUB_SECONDARY_LINK_MODE\\", \\"raw\\").strip()\\\\n    secondary_mix_temperature = float(\\\\n        os.environ.get(\\"BIOHUB_SECONDARY_MIX_TEMPERATURE\\", \\"1\\")\\\\n    )\\\\n    secondary_low_margin_max = float(\\\\n        os.environ.get(\\"BIOHUB_SECONDARY_LOW_MARGIN_MAX\\", \\"0.2\\")\\\\n    )\\\\n    edge_candidate_threshold = float(\\\\n        os.environ.get(\\"BIOHUB_DUAL_SEED_EDGE_THRESHOLD\\", str(cfg.threshold))\\\\n    )\\\\n    if secondary_weights_text:\\\\n        if not 0.0 < secondary_edge_weight < 1.0:\\\\n            raise ValueError(\\"BIOHUB_SECONDARY_EDGE_WEIGHT must be strictly between 0 and 1\\")\\\\n        if not 0.0 <= secondary_detection_weight < 1.0:\\\\n            raise ValueError(\\\\n                \\"BIOHUB_SECONDARY_DETECTION_WEIGHT must be in the half-open interval [0, 1)\\"\\\\n            )\\\\n        if secondary_link_mode not in {\\\\n            \\"raw\\", \\"calibrated\\", \\"adaptive\\", \\"low_margin_consensus\\"\\\\n        }:\\\\n            raise ValueError(\\\\n                \\"BIOHUB_SECONDARY_LINK_MODE must be raw, calibrated, adaptive, \\"\\\\n                \\"or low_margin_consensus\\"\\\\n            )\\\\n        if not 0.5 <= secondary_mix_temperature <= 2.0:\\\\n            raise ValueError(\\"BIOHUB_SECONDARY_MIX_TEMPERATURE must be in [0.5, 2.0]\\")\\\\n        if not 0.0 < edge_candidate_threshold < 1.0:\\\\n            raise ValueError(\\"BIOHUB_DUAL_SEED_EDGE_THRESHOLD must be strictly between 0 and 1\\")\\\\n        if not 0.0 < secondary_low_margin_max <= 1.0:\\\\n            raise ValueError(\\"BIOHUB_SECONDARY_LOW_MARGIN_MAX must be in (0, 1]\\")\\\\n        secondary_model, secondary_window_size, secondary_downsample = load_model(\\\\n            Path(secondary_weights_text), device,\\\\n        )\\\\n        if secondary_window_size != window_size or secondary_downsample != downsample:\\\\n            raise ValueError(\\\\n                \\"Primary and secondary models have incompatible inference grids: \\"\\\\n                f\\"primary=(window={window_size}, downsample={downsample}), \\"\\\\n                f\\"secondary=(window={secondary_window_size}, downsample={secondary_downsample})\\"\\\\n            )\\\\n        cfg.threshold = edge_candidate_threshold\\\\n        print(\\\\n            f\\"Secondary model: {secondary_weights_text} | \\"\\\\n            f\\"edge weight={secondary_edge_weight:.3f} | \\"\\\\n            f\\"detection weight={secondary_detection_weight:.3f} | \\"\\\\n            f\\"link mode={secondary_link_mode} | \\"\\\\n            f\\"temperature={secondary_mix_temperature:.3f} | \\"\\\\n            f\\"low-margin max={secondary_low_margin_max:.3f} | \\"\\\\n            f\\"edge threshold={cfg.threshold:.3f}\\",\\\\n            flush=True,\\\\n        )\\\\n\\\\n    print(\'),\\n    (\'                unet_batch_size=unet_batch_size,\\\\n                downsample=downsample,\\\\n            )\', \'                unet_batch_size=unet_batch_size,\\\\n                downsample=downsample,\\\\n                secondary_model=secondary_model,\\\\n                secondary_edge_weight=secondary_edge_weight,\\\\n                secondary_detection_weight=secondary_detection_weight,\\\\n                secondary_link_mode=secondary_link_mode,\\\\n                secondary_mix_temperature=secondary_mix_temperature,\\\\n                secondary_low_margin_max=secondary_low_margin_max,\\\\n            )\'),\\n]\\nfor _patch_index, (_ensemble_old, _ensemble_new) in enumerate(\\n    _ensemble_replacements, start=1\\n):\\n    _ensemble_count = _s.count(_ensemble_old)\\n    if _ensemble_count != 1:\\n        raise RuntimeError(\\n            f\'Calibrated dual-seed patch {_patch_index} expected one match, \'\\n            f\'found {_ensemble_count}\'\\n        )\\n    _s = _s.replace(_ensemble_old, _ensemble_new, 1)\\ncompile(_s, str(_ps), \'exec\')\\n_ps.write_text(_s)\\nprint(\'Calibrated dual-seed runtime patch applied\')\\n\\n\\n\\nos.environ[\\"BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION\\"] = \\"0.90\\"\\nfor _guard_old_log in WORKING_DIR.glob(\\"retention_guard_*.jsonl\\"):\\n    _guard_old_log.unlink()\\n\\n_s = _ps.read_text()\\n_guard_old = \\"\\"\\"                    det_logits[f] = (\\n                        (1.0 - secondary_detection_weight) * primary_det\\n                        + secondary_detection_weight * secondary_det_aligned\\n                    )\\"\\"\\"\\n_guard_new = \\"\\"\\"                    blended_det = (\\n                        (1.0 - secondary_detection_weight) * primary_det\\n                        + secondary_detection_weight * secondary_det_aligned\\n                    )\\n                    primary_candidates = len(_detect_cells_pooled(\\n                        primary_det[0],\\n                        int(frame_indices[f]),\\n                        cfg.det_threshold,\\n                        pool_k,\\n                    ))\\n                    blended_candidates = len(_detect_cells_pooled(\\n                        blended_det[0],\\n                        int(frame_indices[f]),\\n                        cfg.det_threshold,\\n                        pool_k,\\n                    ))\\n                    minimum_retention = float(os.environ.get(\\n                        \\"BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION\\",\\n                        \\"0.90\\",\\n                    ))\\n                    candidate_retention = (\\n                        blended_candidates / primary_candidates\\n                        if primary_candidates\\n                        else 1.0\\n                    )\\n                    use_primary_detection = bool(\\n                        primary_candidates > 0\\n                        and candidate_retention < minimum_retention\\n                    )\\n                    det_logits[f] = (\\n                        primary_det if use_primary_detection else blended_det\\n                    )\\n                    if int(frame_indices[f]) not in seen_frames:\\n                        shard = os.environ.get(\\n                            \\"BIOHUB_GPU_SHARD\\", \\"single\\"\\n                        ).replace(\\"/\\", \\"_\\")\\n                        guard_log = (\\n                            Path(\\"/kaggle/working\\")\\n                            / f\\"retention_guard_{shard}.jsonl\\"\\n                        )\\n                        guard_record = {\\n                            \\"dataset\\": ds_path.stem,\\n                            \\"frame\\": int(frame_indices[f]),\\n                            \\"primary_candidates\\": int(primary_candidates),\\n                            \\"blended_candidates\\": int(blended_candidates),\\n                            \\"retention\\": float(candidate_retention),\\n                            \\"minimum_retention\\": float(minimum_retention),\\n                            \\"use_primary\\": bool(use_primary_detection),\\n                        }\\n                        with guard_log.open(\\"a\\") as guard_handle:\\n                            guard_handle.write(\\n                                json.dumps(guard_record, sort_keys=True)\\n                                + \\"\\\\\\\\n\\"\\n                            )\\n                        if use_primary_detection:\\n                            print(\\n                                \\"BIOHUB_RETENTION_GUARD \\"\\n                                + json.dumps(guard_record, sort_keys=True),\\n                                flush=True,\\n                            )\\"\\"\\"\\n_guard_matches = _s.count(_guard_old)\\nif _guard_matches != 1:\\n    raise RuntimeError(\\n        f\\"Retention guard expected one blend block, found {_guard_matches}\\"\\n    )\\n_s = _s.replace(_guard_old, _guard_new, 1)\\ncompile(_s, str(_ps), \\"exec\\")\\n_ps.write_text(_s)\\nprint(\\n    \\"Frozen frame retention guard applied at \\"\\n    + os.environ[\\"BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION\\"]\\n)\\n\\n\\n\\nimport math as _bidirectional_math\\n\\n_bidirectional_weight_guard = float(\\n    os.environ.get(\\"BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT\\", \\"0\\")\\n)\\nif not _bidirectional_math.isclose(\\n    _bidirectional_weight_guard, 0.15, rel_tol=0.0, abs_tol=1e-12\\n):\\n    raise ValueError({\\n        \\"expected_bidirectional_weight\\": 0.15,\\n        \\"actual_bidirectional_weight\\": _bidirectional_weight_guard,\\n    })\\n\\n_s = _ps.read_text()\\n_bi_old = \'            edge_logits_pair = model.predict_edges(\\\\n                unet_feat_src, unet_feat_tgt,\\\\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\\\\n                p_pos_src, p_pos_tgt,\\\\n                p_mask_src, p_mask_tgt,\\\\n            )  # (1, n_src, n_tgt)\\\\n\\\\n            if secondary_model is not None:\\\\n\'\\n_bi_new = \'            edge_logits_pair = model.predict_edges(\\\\n                unet_feat_src, unet_feat_tgt,\\\\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\\\\n                p_pos_src, p_pos_tgt,\\\\n                p_mask_src, p_mask_tgt,\\\\n            )  # (1, n_src, n_tgt)\\\\n\\\\n            _bidirectional_weight = float(\\\\n                os.environ.get(\\"BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT\\", \\"0\\")\\\\n            )\\\\n            if _bidirectional_weight > 0.0:\\\\n                reverse_logits_native = model.predict_edges(\\\\n                    unet_feat_tgt, unet_feat_src,\\\\n                    p_coords_tgt * ds_arr_t, p_coords_src * ds_arr_t,\\\\n                    p_pos_tgt, p_pos_src,\\\\n                    p_mask_tgt, p_mask_src,\\\\n                )  # (1, n_tgt, n_src)\\\\n                reverse_logits_pair = reverse_logits_native.transpose(1, 2)\\\\n\\\\n                forward_center = edge_logits_pair.mean(dim=1, keepdim=True)\\\\n                forward_scale = edge_logits_pair.float().std(\\\\n                    dim=1, keepdim=True, unbiased=False\\\\n                ).clamp_min(1e-4)\\\\n                reverse_center = reverse_logits_pair.mean(dim=1, keepdim=True)\\\\n                reverse_scale = reverse_logits_pair.float().std(\\\\n                    dim=1, keepdim=True, unbiased=False\\\\n                ).clamp_min(1e-4)\\\\n                reverse_scale_ratio = (forward_scale / reverse_scale).clamp(0.5, 2.0)\\\\n                reverse_scale_ratio = reverse_scale_ratio.to(reverse_logits_pair.dtype)\\\\n                reverse_aligned = (\\\\n                    (reverse_logits_pair - reverse_center) * reverse_scale_ratio\\\\n                    + forward_center\\\\n                )\\\\n                # Biohub 145: require mutual forward/reverse support in probability space.\\\\n                # The harmonic mean penalizes a candidate when either temporal direction\\\\n                # assigns it very low probability, while calibration preserves the forward\\\\n                # logit scale used by the unchanged downstream candidate threshold and ILP.\\\\n                forward_prob = torch.softmax(edge_logits_pair.float(), dim=1).clamp_min(1e-8)\\\\n                reverse_prob = torch.softmax(reverse_aligned.float(), dim=1).clamp_min(1e-8)\\\\n                harmonic_prob = 1.0 / (\\\\n                    (1.0 - _bidirectional_weight) / forward_prob\\\\n                    + _bidirectional_weight / reverse_prob\\\\n                )\\\\n                harmonic_prob = harmonic_prob / harmonic_prob.sum(\\\\n                    dim=1, keepdim=True\\\\n                ).clamp_min(1e-8)\\\\n                harmonic_logits = torch.log(harmonic_prob.clamp_min(1e-8))\\\\n                harmonic_center = harmonic_logits.mean(dim=1, keepdim=True)\\\\n                harmonic_scale = harmonic_logits.std(\\\\n                    dim=1, keepdim=True, unbiased=False\\\\n                ).clamp_min(1e-4)\\\\n                harmonic_scale_ratio = (forward_scale / harmonic_scale).clamp(0.5, 2.0)\\\\n                edge_logits_pair = (\\\\n                    (harmonic_logits - harmonic_center) * harmonic_scale_ratio\\\\n                    + forward_center\\\\n                ).to(reverse_aligned.dtype)\\\\n                del (\\\\n                    reverse_logits_native,\\\\n                    reverse_logits_pair,\\\\n                    reverse_aligned,\\\\n                    forward_prob,\\\\n                    reverse_prob,\\\\n                    harmonic_prob,\\\\n                    harmonic_logits,\\\\n                )\\\\n            if secondary_model is not None:\\\\n\'\\n_bi_count = _s.count(_bi_old)\\nif _bi_count != 1:\\n    raise RuntimeError(\\n        f\\"Bidirectional edge patch expected one transformed block, found {_bi_count}\\"\\n    )\\n_s = _s.replace(_bi_old, _bi_new, 1)\\n\\n_coordinate_manifest_old = \'    coords = coords.astype(np.int16)\\\\n    return coords, all_edges\'\\n_coordinate_manifest_new = \'    coords = coords.astype(np.int16)\\\\n\\\\n    # Label-free, pre-ILP detector-coordinate manifest. This executes inside\\\\n    # predict_video, before build_graph and the ILP call in predict().\\\\n    _coordinate_manifest_arm = os.environ.get(\\\\n        \\"BIOHUB_DIAGNOSTIC_ARM\\", \\"\\"\\\\n    ).strip()\\\\n    if _coordinate_manifest_arm:\\\\n        import hashlib as _coordinate_hashlib\\\\n\\\\n        _coordinate_shard = os.environ.get(\\\\n            \\"BIOHUB_GPU_SHARD\\", \\"single\\"\\\\n        ).replace(\\"/\\", \\"_\\")\\\\n        _coordinate_array = np.ascontiguousarray(\\\\n            coords.astype(\\"<i2\\", copy=False)\\\\n        )\\\\n        _coordinate_frame_counts = [\\\\n            [int(_coordinate_t), int((_coordinate_array[:, 0] == _coordinate_t).sum())]\\\\n            for _coordinate_t in np.unique(_coordinate_array[:, 0])\\\\n        ]\\\\n        _coordinate_record = {\\\\n            \\"columns\\": [\\"t\\", \\"z\\", \\"y\\", \\"x\\"],\\\\n            \\"coordinate_sha256\\": _coordinate_hashlib.sha256(\\\\n                _coordinate_array.tobytes(order=\\"C\\")\\\\n            ).hexdigest(),\\\\n            \\"dataset\\": ds_path.stem,\\\\n            \\"dtype\\": \\"<i2\\",\\\\n            \\"frame_counts\\": _coordinate_frame_counts,\\\\n            \\"rows\\": int(len(_coordinate_array)),\\\\n            \\"stage\\": \\"post_detection_pre_graph_pre_ilp\\",\\\\n        }\\\\n        _coordinate_manifest_path = (\\\\n            Path(\\"/kaggle/working\\")\\\\n            / f\\"detector_coordinates_{_coordinate_manifest_arm}_\\"\\\\n            f\\"{_coordinate_shard}.jsonl\\"\\\\n        )\\\\n        with _coordinate_manifest_path.open(\\"a\\") as _coordinate_handle:\\\\n            _coordinate_handle.write(\\\\n                json.dumps(_coordinate_record, sort_keys=True) + \\"\\\\\\\\n\\"\\\\n            )\\\\n\\\\n    return coords, all_edges\'\\n_coordinate_manifest_count = _s.count(_coordinate_manifest_old)\\nif _coordinate_manifest_count != 1:\\n    raise RuntimeError(\\n        \\"Coordinate-manifest patch expected one pre-return block, found \\"\\n        f\\"{_coordinate_manifest_count}\\"\\n    )\\n_s = _s.replace(\\n    _coordinate_manifest_old, _coordinate_manifest_new, 1\\n)\\ncompile(_s, str(_ps), \\"exec\\")\\n_ps.write_text(_s)\\nprint(\\n    \\"Bidirectional harmonic-probability association fusion applied | weight=\\",\\n    _bidirectional_weight_guard,\\n)\\nprint(\\"Pre-ILP detector-coordinate manifest hook applied\\")\\n\\n\\n_et_s = _ps.read_text()\\n_et_old = \'        if cfg.det_tta:\\\\n            _nv = 1\\\\n            for dims in [(-1,), (-2,), (-2, -1)]:\\\\n                imgs_flip = imgs.flip(dims)\\\\n                _, det_flip = model.encode(imgs_flip)\\\\n                for f in range(W):\\\\n                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)\\\\n                del imgs_flip, det_flip\\\\n                _nv += 1\\\\n            for _k in (1, 3):\\\\n                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\\\\n                _, det_rot = model.encode(imgs_rot)\\\\n                for f in range(W):\\\\n                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))\\\\n                del imgs_rot, det_rot\\\\n                _nv += 1\\\\n            imgs_t = imgs.transpose(-1, -2)\\\\n            _, det_t = model.encode(imgs_t)\\\\n            for f in range(W):\\\\n                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)\\\\n            del imgs_t, det_t\\\\n            _nv += 1\\\\n            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)\\\\n            _, det_at = model.encode(imgs_at)\\\\n            for f in range(W):\\\\n                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))\\\\n            del imgs_at, det_at\\\\n            _nv += 1\\\\n            for f in range(W):\\\\n                det_logits[f] = det_logits[f] / _nv\\\\n\'\\n_et_new = \\"        if cfg.det_tta:\\\\n            _edge_tta = os.environ.get(\'BIOHUB_EDGE_FEATURE_TTA\', \'0\') != \'0\'\\\\n            _unet_acc = unet_out.clone() if _edge_tta else None\\\\n            _nv = 1\\\\n            for dims in [(-1,), (-2,), (-2, -1)]:\\\\n                imgs_flip = imgs.flip(dims)\\\\n                _u_flip, det_flip = model.encode(imgs_flip)\\\\n                for f in range(W):\\\\n                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)\\\\n                if _edge_tta:\\\\n                    _unet_acc = _unet_acc + _u_flip.flip(dims)\\\\n                del imgs_flip, det_flip, _u_flip\\\\n                _nv += 1\\\\n            for _k in (1, 3):\\\\n                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\\\\n                _u_rot, det_rot = model.encode(imgs_rot)\\\\n                for f in range(W):\\\\n                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))\\\\n                if _edge_tta:\\\\n                    _unet_acc = _unet_acc + torch.rot90(_u_rot, -_k, dims=(-2, -1))\\\\n                del imgs_rot, det_rot, _u_rot\\\\n                _nv += 1\\\\n            imgs_t = imgs.transpose(-1, -2)\\\\n            _u_t, det_t = model.encode(imgs_t)\\\\n            for f in range(W):\\\\n                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)\\\\n            if _edge_tta:\\\\n                _unet_acc = _unet_acc + _u_t.transpose(-1, -2)\\\\n            del imgs_t, det_t, _u_t\\\\n            _nv += 1\\\\n            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)\\\\n            _u_at, det_at = model.encode(imgs_at)\\\\n            for f in range(W):\\\\n                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))\\\\n            if _edge_tta:\\\\n                _unet_acc = _unet_acc + torch.rot90(_u_at.transpose(-1, -2), -1, dims=(-2, -1))\\\\n            del imgs_at, det_at, _u_at\\\\n            _nv += 1\\\\n            for f in range(W):\\\\n                det_logits[f] = det_logits[f] / _nv\\\\n            if _edge_tta:\\\\n                if _unet_acc.shape != unet_out.shape:\\\\n                    raise RuntimeError(\'EDGE-TTA SHAPE MISMATCH: %s vs %s\'\\\\n                                       % (tuple(_unet_acc.shape), tuple(unet_out.shape)))\\\\n                _delta = float((_unet_acc / _nv - unet_out).abs().mean())\\\\n                if _delta == 0.0:\\\\n                    raise RuntimeError(\'EDGE-TTA NO-OP: averaged features bit-identical to the \'\\\\n                                       \'single-pass features, so the augmented encodes \'\\\\n                                       \'contributed nothing and this arm would read as a \'\\\\n                                       \'false null\')\\\\n                unet_out = _unet_acc / _nv\\\\n                print(\'EDGE_TTA_ACTIVE views=\', _nv, \'mean_abs_feat_delta=\', round(_delta, 6), flush=True)\\\\n                del _unet_acc\\\\n\\"\\nif _et_s.count(_et_old) != 1:\\n    raise RuntimeError(\'edge-TTA anchor block not unique: %d\' % _et_s.count(_et_old))\\n_et_s = _et_s.replace(_et_old, _et_new, 1)\\ncompile(_et_s, str(_ps), \'exec\')\\n_ps.write_text(_et_s)\\nif \'EDGE_TTA_ACTIVE\' not in _ps.read_text():\\n    raise RuntimeError(\'EDGE-TTA PATCH DID NOT PERSIST\')\\nos.environ[\'BIOHUB_EDGE_FEATURE_TTA\'] = \'1\'\\nprint(\'EDGE_TTA patch installed and enabled in\', _ps)\\n\\n_secondary_tta_source = _ps.read_text()\\n_secondary_tta_old = \'        secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\\\\n\\\\n            if secondary_detection_weight > 0.0:\\\\n                if cfg.det_tta:\\\\n                    _secondary_nv = 1\\\\n                    for dims in [(-1,), (-2,), (-2, -1)]:\\\\n                        secondary_imgs_flip = imgs.flip(dims)\\\\n                        _, secondary_det_flip = secondary_model.encode(secondary_imgs_flip)\\\\n                        for f in range(W):\\\\n                            secondary_det_logits[f] = (\\\\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\\\\n                            )\\\\n                        del secondary_imgs_flip, secondary_det_flip\\\\n                        _secondary_nv += 1\\\\n                    for _k in (1, 3):\\\\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\\\\n                        _, secondary_det_rot = secondary_model.encode(secondary_imgs_rot)\\\\n                        for f in range(W):\\\\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\\\\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\\\\n                            )\\\\n                        del secondary_imgs_rot, secondary_det_rot\\\\n                        _secondary_nv += 1\\\\n                    secondary_imgs_t = imgs.transpose(-1, -2)\\\\n                    _, secondary_det_t = secondary_model.encode(secondary_imgs_t)\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = (\\\\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\\\\n                        )\\\\n                    del secondary_imgs_t, secondary_det_t\\\\n                    _secondary_nv += 1\\\\n                    secondary_imgs_at = torch.rot90(\\\\n                        imgs, 1, dims=(-2, -1)\\\\n                    ).transpose(-1, -2)\\\\n                    _, secondary_det_at = secondary_model.encode(secondary_imgs_at)\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\\\\n                            secondary_det_at[f].transpose(-1, -2),\\\\n                            -1,\\\\n                            dims=(-2, -1),\\\\n                        )\\\\n                    del secondary_imgs_at, secondary_det_at\\\\n                    _secondary_nv += 1\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\\\\n\\\\n                for f in range(W):\'\\n_secondary_tta_new = \'        secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\\\\n            _secondary_edge_tta = os.environ.get(\\\\n                \\"BIOHUB_SECONDARY_EDGE_FEATURE_TTA\\", \\"0\\"\\\\n            ) != \\"0\\"\\\\n            _secondary_unet_acc = (\\\\n                secondary_unet_out.clone() if _secondary_edge_tta else None\\\\n            )\\\\n\\\\n            if secondary_detection_weight > 0.0:\\\\n                if cfg.det_tta:\\\\n                    _secondary_nv = 1\\\\n                    for dims in [(-1,), (-2,), (-2, -1)]:\\\\n                        secondary_imgs_flip = imgs.flip(dims)\\\\n                        _secondary_u_flip, secondary_det_flip = secondary_model.encode(\\\\n                            secondary_imgs_flip\\\\n                        )\\\\n                        for f in range(W):\\\\n                            secondary_det_logits[f] = (\\\\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\\\\n                            )\\\\n                        if _secondary_edge_tta:\\\\n                            _secondary_unet_acc = _secondary_unet_acc + _secondary_u_flip.flip(dims)\\\\n                        del secondary_imgs_flip, secondary_det_flip, _secondary_u_flip\\\\n                        _secondary_nv += 1\\\\n                    for _k in (1, 3):\\\\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\\\\n                        _secondary_u_rot, secondary_det_rot = secondary_model.encode(\\\\n                            secondary_imgs_rot\\\\n                        )\\\\n                        for f in range(W):\\\\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\\\\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\\\\n                            )\\\\n                        if _secondary_edge_tta:\\\\n                            _secondary_unet_acc = _secondary_unet_acc + torch.rot90(\\\\n                                _secondary_u_rot, -_k, dims=(-2, -1)\\\\n                            )\\\\n                        del secondary_imgs_rot, secondary_det_rot, _secondary_u_rot\\\\n                        _secondary_nv += 1\\\\n                    secondary_imgs_t = imgs.transpose(-1, -2)\\\\n                    _secondary_u_t, secondary_det_t = secondary_model.encode(secondary_imgs_t)\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = (\\\\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\\\\n                        )\\\\n                    if _secondary_edge_tta:\\\\n                        _secondary_unet_acc = _secondary_unet_acc + _secondary_u_t.transpose(-1, -2)\\\\n                    del secondary_imgs_t, secondary_det_t, _secondary_u_t\\\\n                    _secondary_nv += 1\\\\n                    secondary_imgs_at = torch.rot90(\\\\n                        imgs, 1, dims=(-2, -1)\\\\n                    ).transpose(-1, -2)\\\\n                    _secondary_u_at, secondary_det_at = secondary_model.encode(\\\\n                        secondary_imgs_at\\\\n                    )\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\\\\n                            secondary_det_at[f].transpose(-1, -2),\\\\n                            -1,\\\\n                            dims=(-2, -1),\\\\n                        )\\\\n                    if _secondary_edge_tta:\\\\n                        _secondary_unet_acc = _secondary_unet_acc + torch.rot90(\\\\n                            _secondary_u_at.transpose(-1, -2), -1, dims=(-2, -1)\\\\n                        )\\\\n                    del secondary_imgs_at, secondary_det_at, _secondary_u_at\\\\n                    _secondary_nv += 1\\\\n                    for f in range(W):\\\\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\\\\n                    if _secondary_edge_tta:\\\\n                        if _secondary_unet_acc.shape != secondary_unet_out.shape:\\\\n                            raise RuntimeError(\\"SECONDARY_EDGE_TTA_SHAPE_MISMATCH\\")\\\\n                        _secondary_delta = float(\\\\n                            (_secondary_unet_acc / _secondary_nv - secondary_unet_out).abs().mean()\\\\n                        )\\\\n                        if _secondary_delta == 0.0:\\\\n                            raise RuntimeError(\\"SECONDARY_EDGE_TTA_NO_OP\\")\\\\n                        _secondary_edge_tta_weight = float(os.environ.get(\\\\n                            \\"BIOHUB_SECONDARY_EDGE_FEATURE_TTA_WEIGHT\\", \\"1.0\\"\\\\n                        ))\\\\n                        if not 0.0 < _secondary_edge_tta_weight <= 1.0:\\\\n                            raise RuntimeError(\\"SECONDARY_EDGE_TTA_BAD_WEIGHT\\")\\\\n                        _secondary_tta_mean = _secondary_unet_acc / _secondary_nv\\\\n                        secondary_unet_out = (\\\\n                            (1.0 - _secondary_edge_tta_weight) * secondary_unet_out\\\\n                            + _secondary_edge_tta_weight * _secondary_tta_mean\\\\n                        )\\\\n                        print(\\\\n                            \\"SECONDARY_EDGE_TTA_ACTIVE views=\\",\\\\n                            _secondary_nv,\\\\n                            \\"weight=\\",\\\\n                            _secondary_edge_tta_weight,\\\\n                            \\"mean_abs_feat_delta=\\",\\\\n                            round(_secondary_delta, 6),\\\\n                            flush=True,\\\\n                        )\\\\n                        del _secondary_unet_acc\\\\n\\\\n                for f in range(W):\'\\n_secondary_tta_count = _secondary_tta_source.count(_secondary_tta_old)\\nif _secondary_tta_count != 1:\\n    raise RuntimeError(\\n        \\"secondary edge-TTA anchor expected one match, found \\"\\n        + str(_secondary_tta_count)\\n    )\\n_secondary_tta_source = _secondary_tta_source.replace(\\n    _secondary_tta_old, _secondary_tta_new, 1\\n)\\ncompile(_secondary_tta_source, str(_ps), \\"exec\\")\\n_ps.write_text(_secondary_tta_source)\\nif \\"SECONDARY_EDGE_TTA_ACTIVE\\" not in _ps.read_text():\\n    raise RuntimeError(\\"secondary edge-TTA patch did not persist\\")\\nos.environ[\\"BIOHUB_SECONDARY_EDGE_FEATURE_TTA\\"] = \\"1\\"\\nos.environ[\\"BIOHUB_SECONDARY_EDGE_FEATURE_TTA_WEIGHT\\"] = \\"0.75\\"\\nprint(\\"secondary edge-feature TTA patch installed and enabled\\", flush=True)\\n\\n\\ndef list_test_stems() -> list[str]:\\n    if not TEST_DIR.exists():\\n        raise FileNotFoundError(f\\"Test directory does not exist: {TEST_DIR}\\")\\n    stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(\\".zarr\\"))\\n    if not stems:\\n        raise FileNotFoundError(f\\"No test .zarr files found in {TEST_DIR}\\")\\n    return stems\\n\\n\\ntest_stems = list_test_stems()\\nprint(f\\"Found {len(test_stems)} test videos\\")\\nprint(test_stems[:10])\\n\\nsplits_path = REPO_DIR / \\"kaggle_test_splits_50ep.json\\"\\nsplits_path.parent.mkdir(parents=True, exist_ok=True)\\nsplits_path.write_text(json.dumps([{\\"split\\": 0, \\"train\\": [], \\"test\\": test_stems}], indent=2))\\n\\npredict_cmd = [\\n    sys.executable,\\n    \\"scripts/predict_unet_transformer.py\\",\\n    \\"--data-dir\\",\\n    str(TEST_DIR),\\n    \\"--splits\\",\\n    str(splits_path.name),\\n    \\"--split\\",\\n    \\"0\\",\\n    \\"--weights\\",\\n    WEIGHTS_RELATIVE,\\n    \\"--unet-batch-size\\",\\n    str(UNET_BATCH_SIZE),\\n    \\"--det-threshold\\",\\n    str(DET_THRESHOLD),\\n    \\"--ilp-edge-weight\\",\\n    str(ILP_EDGE_WEIGHT),\\n    \\"--ilp-appearance-weight\\",\\n    str(ILP_APPEARANCE_WEIGHT),\\n    \\"--ilp-disappearance-weight\\",\\n    str(ILP_DISAPPEARANCE_WEIGHT),\\n    \\"--ilp-division-weight\\",\\n    str(ILP_DIVISION_WEIGHT),\\n]\\nif USE_ILP:\\n    predict_cmd.append(\\"--use-ilp\\")\\nif SLICE:\\n    predict_cmd.extend([\\"--slice\\", SLICE])\\n\\ndef _visible_cuda_tokens(count: int) -> list[str]:\\n    raw = os.environ.get(\\"CUDA_VISIBLE_DEVICES\\", \\"\\").strip()\\n    if raw and raw != \\"-1\\":\\n        tokens = [token.strip() for token in raw.split(\\",\\") if token.strip()]\\n        if len(tokens) < count:\\n            raise RuntimeError(\\n                f\\"torch reports {count} CUDA devices but CUDA_VISIBLE_DEVICES={raw!r}\\"\\n            )\\n        return tokens[:count]\\n    return [str(index) for index in range(count)]\\n\\n\\ndef _prediction_dir_for_method(method: str) -> Path:\\n    matches = sorted((REPO_DIR / \\"predictions\\").glob(f\\"*/{method}/split_0\\"))\\n    if len(matches) != 1:\\n        raise RuntimeError(\\n            f\\"Expected exactly one prediction directory for {method!r}, found {matches}\\"\\n        )\\n    return matches[0]\\n\\n\\ndef _wait_for_prediction_shards(\\n    processes: dict[int, subprocess.Popen],\\n    commands: dict[int, list[str]],\\n) -> None:\\n    while processes:\\n        failed: tuple[int, int] | None = None\\n        for shard_index, process in list(processes.items()):\\n            return_code = process.poll()\\n            if return_code is None:\\n                continue\\n            del processes[shard_index]\\n            if return_code != 0:\\n                failed = (shard_index, return_code)\\n                break\\n        if failed is None:\\n            if processes:\\n                time.sleep(1.0)\\n            continue\\n\\n        failed_index, failed_code = failed\\n        for process in processes.values():\\n            if process.poll() is None:\\n                process.terminate()\\n        for process in processes.values():\\n            try:\\n                process.wait(timeout=30)\\n            except subprocess.TimeoutExpired:\\n                process.kill()\\n                process.wait()\\n        raise subprocess.CalledProcessError(failed_code, commands[failed_index])\\n\\n\\ndef _merge_prediction_shards(worker_count: int) -> Path:\\n    shard_dirs: list[Path] = []\\n    seen: set[str] = set()\\n    expected_all = set(test_stems)\\n\\n    for shard_index in range(worker_count):\\n        shard_method = f\\"{METHOD}_gpu{shard_index}\\"\\n        shard_dir = _prediction_dir_for_method(shard_method)\\n        expected = set(test_stems[shard_index::worker_count])\\n        shard_paths = sorted(shard_dir.glob(\\"*.geff\\"))\\n        found = {path.stem for path in shard_paths}\\n        if found != expected:\\n            raise RuntimeError(\\n                f\\"GPU shard {shard_index} output mismatch: \\"\\n                f\\"missing={sorted(expected - found)}, extra={sorted(found - expected)}\\"\\n            )\\n        overlap = seen & found\\n        if overlap:\\n            raise RuntimeError(f\\"Duplicate datasets across GPU shards: {sorted(overlap)}\\")\\n        seen.update(found)\\n        shard_dirs.append(shard_dir)\\n\\n    if seen != expected_all:\\n        raise RuntimeError(\\n            f\\"Merged GPU shards do not cover the test set: \\"\\n            f\\"missing={sorted(expected_all - seen)}, extra={sorted(seen - expected_all)}\\"\\n        )\\n\\n    username_roots = {shard_dir.parents[1] for shard_dir in shard_dirs}\\n    if len(username_roots) != 1:\\n        raise RuntimeError(f\\"GPU shards used inconsistent prediction roots: {username_roots}\\")\\n    import shutil as _shutil\\n\\n    final_root = next(iter(username_roots)) / METHOD\\n    final_dir = final_root / \\"split_0\\"\\n    staging_dir = final_root / \\"split_0_dual_gpu_staging\\"\\n    if staging_dir.exists():\\n        if staging_dir.is_dir():\\n            _shutil.rmtree(staging_dir)\\n        else:\\n            staging_dir.unlink()\\n    staging_dir.mkdir(parents=True, exist_ok=False)\\n\\n    for shard_dir in shard_dirs:\\n        for source in sorted(shard_dir.glob(\\"*.geff\\")):\\n            destination = staging_dir / source.name\\n            if destination.exists():\\n                raise RuntimeError(f\\"Refusing to overwrite duplicate merged output: {destination}\\")\\n            _shutil.move(str(source), str(destination))\\n\\n    merged = {path.stem for path in staging_dir.glob(\\"*.geff\\")}\\n    if merged != expected_all:\\n        raise RuntimeError(\\n            f\\"Staged prediction directory failed verification: \\"\\n            f\\"missing={sorted(expected_all - merged)}, extra={sorted(merged - expected_all)}\\"\\n        )\\n\\n    if final_dir.exists():\\n        if final_dir.is_dir():\\n            _shutil.rmtree(final_dir)\\n        else:\\n            final_dir.unlink()\\n    staging_dir.rename(final_dir)\\n    for shard_dir in shard_dirs:\\n        _shutil.rmtree(shard_dir.parent)\\n    print(f\\"Merged {len(merged)} prediction graphs into {final_dir}\\")\\n    return final_dir\\n\\n\\nstart_time = time.time()\\navailable_gpu_count = _torch.cuda.device_count()\\nworker_count = min(2, available_gpu_count, len(test_stems))\\n\\nif worker_count >= 2 and not SLICE:\\n    cuda_tokens = _visible_cuda_tokens(worker_count)\\n    processes: dict[int, subprocess.Popen] = {}\\n    commands: dict[int, list[str]] = {}\\n    print(f\\"Launching {worker_count} independent video shards on CUDA devices {cuda_tokens}\\")\\n    for shard_index in range(worker_count):\\n        shard_method = f\\"{METHOD}_gpu{shard_index}\\"\\n        shard_cmd = [\\n            *predict_cmd,\\n            \\"--method\\",\\n            shard_method,\\n            \\"--slice\\",\\n            f\\"{shard_index}::{worker_count}\\",\\n        ]\\n        shard_env = {**os.environ, \\"PYTHONPATH\\": \\"src\\"}\\n        shard_env[\\"CUDA_VISIBLE_DEVICES\\"] = cuda_tokens[shard_index]\\n        shard_env[\\"BIOHUB_GPU_SHARD\\"] = f\\"{shard_index}/{worker_count}\\"\\n        print(\\n            f\\"GPU shard {shard_index}: CUDA_VISIBLE_DEVICES={cuda_tokens[shard_index]} | \\"\\n            + \\" \\".join(shard_cmd),\\n            flush=True,\\n        )\\n        commands[shard_index] = shard_cmd\\n        processes[shard_index] = subprocess.Popen(\\n            shard_cmd,\\n            cwd=REPO_DIR,\\n            env=shard_env,\\n        )\\n    _wait_for_prediction_shards(processes, commands)\\n    _merge_prediction_shards(worker_count)\\nelse:\\n    reason = \\"SLICE is active\\" if SLICE else f\\"only {available_gpu_count} CUDA device(s) available\\"\\n    print(f\\"Using single-process prediction because {reason}.\\")\\n    print(\\" \\".join(predict_cmd))\\n    subprocess.run(\\n        predict_cmd,\\n        cwd=REPO_DIR,\\n        env={**os.environ, \\"PYTHONPATH\\": \\"src\\"},\\n        check=True,\\n    )\\n\\npredict_seconds = time.time() - start_time\\nprint(f\\"Prediction completed in {predict_seconds / 60:.2f} minutes\\")"], [7, "\\"\\"\\"Conservative, label-free locks for existing ILP division motifs.\\n\\nExperiment HF-FP-01. No training, matching to GT, or parameter search.\\nThe entire parent -> fork -> two daughters -> two successors motif is\\nreserved only if each of its five original edges is confident and short.\\n\\"\\"\\"\\nfrom collections import defaultdict\\nfrom math import dist, isfinite\\n\\n\\ndef select_fork_anchors(\\n    nodes, edges, *, min_probability=0.80, max_step_um=6.0,\\n    max_sister_um=8.0, voxel_scale=(1.625, 0.40625, 0.40625),\\n):\\n    \\"\\"\\"Return copied locked edges and deterministic, score-free diagnostics.\\n\\nIDs and ordering never influence acceptance except stable output ordering.\\nOverlapping motifs are allowed only when their original edges agree;\\nthe original graph\'s unique-parent/exact-degree checks guarantee this.\\n\\"\\"\\"\\n    if not 0.0 <= min_probability <= 1.0:\\n        raise ValueError(\\"min_probability must be in [0,1]\\")\\n    if max_step_um <= 0 or max_sister_um <= 0:\\n        raise ValueError(\\"physical gates must be positive\\")\\n    incoming, outgoing = defaultdict(list), defaultdict(list)\\n    lookup = {}\\n    for edge in edges:\\n        key = (int(edge[\\"source_id\\"]), int(edge[\\"target_id\\"]))\\n        if key in lookup:\\n            raise ValueError(f\\"Duplicate raw edge: {key}\\")\\n        lookup[key] = edge\\n        outgoing[key[0]].append(key)\\n        incoming[key[1]].append(key)\\n\\n    stats = {\\n        \\"fork_anchor_raw_forks\\": 0, \\"fork_anchor_accepted\\": 0,\\n        \\"fork_anchor_reject_context\\": 0, \\"fork_anchor_reject_probability\\": 0,\\n        \\"fork_anchor_reject_geometry\\": 0, \\"fork_anchor_locked_edges\\": 0,\\n    }\\n    locked = set()\\n\\n    def xyz(node_id):\\n        node = nodes[node_id]\\n        return tuple(float(node[k]) * scale for k, scale in zip(\\"zyx\\", voxel_scale))\\n\\n    for fork in sorted(outgoing):\\n        daughters = outgoing[fork]\\n        if len(daughters) != 2:\\n            continue\\n        stats[\\"fork_anchor_raw_forks\\"] += 1\\n        parent_edges = incoming[fork]\\n        if len(parent_edges) != 1:\\n            stats[\\"fork_anchor_reject_context\\"] += 1\\n            continue\\n        before = parent_edges[0]\\n        children = sorted(edge[1] for edge in daughters)\\n        if len(outgoing[before[0]]) != 1 or any(\\n            len(incoming[c]) != 1 or len(outgoing[c]) != 1 for c in children\\n        ):\\n            stats[\\"fork_anchor_reject_context\\"] += 1\\n            continue\\n        after = [outgoing[c][0] for c in children]\\n        motif = [before, *daughters, *after]\\n        ids = {v for pair in motif for v in pair}\\n        if len(ids) != 6 or any(v not in nodes for v in ids) or any(\\n            len(incoming[pair[1]]) != 1 for pair in after\\n        ):\\n            stats[\\"fork_anchor_reject_context\\"] += 1\\n            continue\\n        if any(int(nodes[t][\\"t\\"]) != int(nodes[s][\\"t\\"]) + 1 for s, t in motif):\\n            stats[\\"fork_anchor_reject_context\\"] += 1\\n            continue\\n        probabilities = []\\n        for pair in motif:\\n            try:\\n                value = float(lookup[pair].get(\\"edge_prob\\"))\\n            except (TypeError, ValueError):\\n                value = float(\\"nan\\")\\n            probabilities.append(value)\\n        if any(not isfinite(p) or not min_probability <= p <= 1.0 for p in probabilities):\\n            stats[\\"fork_anchor_reject_probability\\"] += 1\\n            continue\\n        points = {v: xyz(v) for v in ids}\\n        if any(not isfinite(x) for p in points.values() for x in p) or any(\\n            dist(points[s], points[t]) > max_step_um for s, t in motif\\n        ) or dist(points[children[0]], points[children[1]]) > max_sister_um:\\n            stats[\\"fork_anchor_reject_geometry\\"] += 1\\n            continue\\n        locked.update(motif)\\n        stats[\\"fork_anchor_accepted\\"] += 1\\n\\n    result = [{**lookup[pair], \\"fork_anchor_locked\\": 1} for pair in sorted(locked)]\\n    stats[\\"fork_anchor_locked_edges\\"] = len(result)\\n    return result, stats\\n"], [8, "import tracksdata as td\\nimport numpy as np\\nimport blosc2\\nfrom scipy.optimize import linear_sum_assignment\\nfrom scipy.spatial import cKDTree\\n\\nSUBMISSION_COLUMNS = [\\"dataset\\", \\"row_type\\", \\"node_id\\", \\"t\\", \\"z\\", \\"y\\", \\"x\\", \\"source_id\\", \\"target_id\\"]\\nCSV_COLUMNS = [\\"id\\", *SUBMISSION_COLUMNS]\\nVOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)\\n\\n\\ndef graph_from_geff(path: Path):\\n    graph = td.graph.IndexedRXGraph.from_geff(path)\\n    return graph[0] if isinstance(graph, tuple) else graph\\n\\n\\ndef edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:\\n    dz = (float(source[\\"z\\"]) - float(target[\\"z\\"])) * VOXEL_SCALE_UM[0]\\n    dy = (float(source[\\"y\\"]) - float(target[\\"y\\"])) * VOXEL_SCALE_UM[1]\\n    dx = (float(source[\\"x\\"]) - float(target[\\"x\\"])) * VOXEL_SCALE_UM[2]\\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\\n\\n\\ndef point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:\\n    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]\\n    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]\\n    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]\\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\\n\\n\\ndef node_point(node: dict[str, object]) -> tuple[float, float, float]:\\n    return (float(node[\\"z\\"]), float(node[\\"y\\"]), float(node[\\"x\\"]))\\n\\n\\ndef edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:\\n    prob = edge.get(\\"edge_prob\\")\\n    prob_value = float(prob) if prob is not None else 0.0\\n    return prob_value, -float(edge[\\"distance_um\\"])\\n\\n\\ndef _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:\\n    return max(nodes_by_id) + 1 if nodes_by_id else 1\\n\\n\\n\\ndef read_test_frame(dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:\\n    if t in frame_cache:\\n        return frame_cache[t]\\n    zarr_path = TEST_DIR / f\\"{dataset}.zarr\\"\\n    meta = json.loads((zarr_path / \\"0\\" / \\"zarr.json\\").read_text())\\n    shape = tuple(int(v) for v in meta[\\"shape\\"])\\n    dtype = np.dtype(meta[\\"data_type\\"])\\n    frame_shape = shape[1:]\\n    chunk_path = zarr_path / \\"0\\" / \\"c\\" / str(t) / \\"0\\" / \\"0\\" / \\"0\\"\\n    try:\\n        raw = chunk_path.read_bytes()\\n        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)\\n        if arr.size == int(np.prod(frame_shape)):\\n            frame = arr.reshape(frame_shape).copy()\\n            frame_cache[t] = frame\\n            return frame\\n    except Exception:\\n        pass\\n    import zarr\\n    frame = np.asarray(zarr.open(zarr_path / \\"0\\", mode=\\"r\\")[t])\\n    frame_cache[t] = frame\\n    return frame\\n\\n\\ndef refine_synthetic_midpoint(\\n    dataset: str | None,\\n    t: int,\\n    midpoint: tuple[float, float, float],\\n    frame_cache: dict[int, np.ndarray],\\n    stats: dict[str, int],\\n) -> tuple[float, float, float]:\\n    if not GAP_REFINE_SYNTHETIC or dataset is None:\\n        return midpoint\\n    try:\\n        frame = read_test_frame(dataset, t, frame_cache)\\n        z, y, x = [int(round(v)) for v in midpoint]\\n        z0 = max(0, z - GAP_REFINE_WIN_Z)\\n        z1 = min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)\\n        y0 = max(0, y - GAP_REFINE_WIN_YX)\\n        y1 = min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)\\n        x0 = max(0, x - GAP_REFINE_WIN_YX)\\n        x1 = min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)\\n        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)\\n        if patch.size == 0:\\n            stats[\\"gap_refine_failed\\"] += 1\\n            return midpoint\\n        baseline = float(np.percentile(patch, 20.0))\\n        weights = np.maximum(patch - baseline, 0.0)\\n        total = float(weights.sum())\\n        if total <= 0:\\n            stats[\\"gap_refine_failed\\"] += 1\\n            return midpoint\\n        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]\\n        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]\\n        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]\\n        refined = (\\n            float((weights * zz).sum() / total),\\n            float((weights * yy).sum() / total),\\n            float((weights * xx).sum() / total),\\n        )\\n        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:\\n            stats[\\"gap_refine_rejected_shift\\"] += 1\\n            return midpoint\\n        stats[\\"gap_refined_synthetic\\"] += 1\\n        return refined\\n    except Exception:\\n        stats[\\"gap_refine_failed\\"] += 1\\n        return midpoint\\n\\n\\n\\ndef _dc_pool_frame_xy(volume: np.ndarray, factor: int) -> np.ndarray:\\n    if factor <= 1:\\n        return volume.astype(np.float32, copy=False)\\n    z, y, x = volume.shape\\n    y2 = (y // factor) * factor\\n    x2 = (x // factor) * factor\\n    cropped = volume[:, :y2, :x2].astype(np.float32, copy=False)\\n    return cropped.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))\\n\\n\\ndef _dc_normalize_dynamic_range(volume: np.ndarray, cfg: object) -> np.ndarray:\\n    vol = np.asarray(volume, dtype=np.float32)\\n    lo = float(np.percentile(vol, float(getattr(cfg, \\"norm_lo_pct\\", 50.0))))\\n    hi = float(np.percentile(vol, float(getattr(cfg, \\"norm_hi_pct\\", 99.5))))\\n    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:\\n        return np.zeros_like(vol, dtype=np.float32)\\n    ratio = (vol - lo) / (hi - lo)\\n    return np.clip(\\n        ratio,\\n        float(getattr(cfg, \\"norm_clip_lo\\", -0.5)),\\n        float(getattr(cfg, \\"norm_clip_hi\\", 6.0)),\\n    ).astype(np.float32)\\n\\n\\ndef _dc_manifest_weight_paths(manifest_path: Path) -> list[Path]:\\n    if not manifest_path.exists():\\n        return []\\n    try:\\n        manifest = json.loads(manifest_path.read_text())\\n    except Exception as exc:\\n        print(\\"Could not read DeepCenter manifest:\\", manifest_path, type(exc).__name__, exc)\\n        return []\\n    root = manifest_path.parent\\n    sections: list[dict[str, object]] = []\\n    for section in [\\n        manifest.get(\\"model\\", {}),\\n        manifest.get(\\"models\\", {}).get(\\"full_frame_center\\", {}) if isinstance(manifest.get(\\"models\\", {}), dict) else {},\\n        manifest.get(\\"full_frame_center\\", {}),\\n    ]:\\n        if isinstance(section, dict):\\n            sections.append(section)\\n    candidates: list[Path] = []\\n    for section in sections:\\n        for key in (\\"weight_path\\", \\"path\\"):\\n            rel = section.get(key)\\n            if isinstance(rel, str) and rel:\\n                candidates.append(root / rel)\\n        for key in (\\"last_checkpoint\\", \\"best_checkpoint\\"):\\n            item = section.get(key)\\n            if isinstance(item, dict):\\n                rel = item.get(\\"path\\")\\n                if isinstance(rel, str) and rel:\\n                    candidates.append(root / rel)\\n    for name in (\\"checkpoint_last.pt\\", \\"best.pt\\", \\"last.pt\\"):\\n        candidates.append(root / \\"weights\\" / \\"full_frame_center\\" / name)\\n        candidates.append(root / name)\\n    candidates.append(root / DEEPCENTER_RELATIVE)\\n    return candidates\\n\\n\\ndef _dc_checkpoint_candidates() -> list[Path]:\\n    candidates: list[Path] = []\\n    explicit = os.environ.get(\\"BIOHUB_DEEPCENTER_CHECKPOINT\\", DEEPCENTER_CHECKPOINT_DEFAULT).strip()\\n    if explicit:\\n        candidates.append(Path(explicit))\\n    manifest_explicit = os.environ.get(\\"BIOHUB_DEEPCENTER_MANIFEST\\", DEEPCENTER_MANIFEST_DEFAULT).strip()\\n    if manifest_explicit:\\n        candidates.extend(_dc_manifest_weight_paths(Path(manifest_explicit)))\\n\\n    input_root = Path(\\"/kaggle/input\\")\\n    preferred_dirs = [\\n        Path(\\"/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1\\"),\\n        Path(\\"/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1\\"),\\n    ]\\n    for directory in preferred_dirs:\\n        candidates.extend(_dc_manifest_weight_paths(directory / \\"ARTIFACT_MANIFEST.json\\"))\\n        for name in (\\"checkpoint_last.pt\\", \\"best.pt\\", \\"last.pt\\"):\\n            candidates.append(directory / \\"weights\\" / \\"full_frame_center\\" / name)\\n            candidates.append(directory / name)\\n    if input_root.exists():\\n        for name in (\\"checkpoint_last.pt\\", \\"best.pt\\", \\"last.pt\\"):\\n            candidates.extend(sorted(input_root.glob(f\\"**/full_frame_center/**/{name}\\")))\\n\\n    seen: set[Path] = set()\\n    out: list[Path] = []\\n    for path in candidates:\\n        path = path.expanduser()\\n        try:\\n            key = path.resolve() if path.exists() else path\\n        except Exception:\\n            key = path\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        out.append(path)\\n    return out\\n\\n\\ntry:\\n    import torch\\nexcept Exception as _dc_torch_error:\\n    torch = None\\n\\n\\nif torch is not None:\\n    class _DCConvBlock3d(torch.nn.Module):\\n        def __init__(self, in_channels: int, out_channels: int) -> None:\\n            super().__init__()\\n            groups = min(8, out_channels)\\n            self.block = torch.nn.Sequential(\\n                torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),\\n                torch.nn.GroupNorm(groups, out_channels),\\n                torch.nn.SiLU(inplace=True),\\n                torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),\\n                torch.nn.GroupNorm(groups, out_channels),\\n                torch.nn.SiLU(inplace=True),\\n            )\\n\\n        def forward(self, x):\\n            return self.block(x)\\n\\n\\n    class _DCDeepCenterUNet3D(torch.nn.Module):\\n        def __init__(self, in_channels: int = 1, base_channels: int = 24) -> None:\\n            super().__init__()\\n            c = int(base_channels)\\n            self.enc1 = _DCConvBlock3d(in_channels, c)\\n            self.down1 = torch.nn.MaxPool3d(2, 2)\\n            self.enc2 = _DCConvBlock3d(c, c * 2)\\n            self.down2 = torch.nn.MaxPool3d(2, 2)\\n            self.enc3 = _DCConvBlock3d(c * 2, c * 4)\\n            self.down3 = torch.nn.MaxPool3d(2, 2)\\n            self.bottleneck = _DCConvBlock3d(c * 4, c * 8)\\n            self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)\\n            self.dec3 = _DCConvBlock3d(c * 8, c * 4)\\n            self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)\\n            self.dec2 = _DCConvBlock3d(c * 4, c * 2)\\n            self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)\\n            self.dec1 = _DCConvBlock3d(c * 2, c)\\n            self.head = torch.nn.Conv3d(c, 1, 1)\\n\\n        def forward(self, x):\\n            e1 = self.enc1(x)\\n            e2 = self.enc2(self.down1(e1))\\n            e3 = self.enc3(self.down2(e2))\\n            b = self.bottleneck(self.down3(e3))\\n            d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))\\n            d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))\\n            d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))\\n            return self.head(d1)\\nelse:\\n    _DCConvBlock3d = None\\n    _DCDeepCenterUNet3D = None\\n\\ndef load_deepcenter_veto_detector() -> dict[str, object] | None:\\n    if not USE_DEEPCENTER_VETO:\\n        print(\\"DeepCenter add-only repair gate disabled by configuration.\\")\\n        return None\\n    if torch is None:\\n        if REQUIRE_DEEPCENTER_VETO:\\n            raise ImportError(\\"torch is required for DeepCenter add-only repair gate\\")\\n        print(\\"DeepCenter add-only repair gate skipped because torch is unavailable.\\")\\n        return None\\n    from types import SimpleNamespace\\n\\n    device = torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n    load_errors: list[str] = []\\n    for checkpoint_path in _dc_checkpoint_candidates():\\n        if not checkpoint_path.exists():\\n            continue\\n        try:\\n            print(\\"Trying DeepCenter add-only gate checkpoint:\\", checkpoint_path)\\n            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\\n            if not isinstance(checkpoint, dict) or \\"model_state\\" not in checkpoint:\\n                raise ValueError(\\"checkpoint has no model_state\\")\\n            checkpoint_epoch = int(checkpoint.get(\\"epoch\\", -1))\\n            if DEEPCENTER_EXPECTED_EPOCH > 0 and checkpoint_epoch != DEEPCENTER_EXPECTED_EPOCH:\\n                raise ValueError(\\n                    f\\"expected DeepCenter epoch {DEEPCENTER_EXPECTED_EPOCH}, got {checkpoint_epoch}\\"\\n                )\\n            cfg = SimpleNamespace(**checkpoint.get(\\"config\\", {}))\\n            model = _DCDeepCenterUNet3D(base_channels=int(getattr(cfg, \\"base_channels\\", 24)))\\n            model.load_state_dict(checkpoint[\\"model_state\\"])\\n            model.to(device)\\n            model.eval()\\n            print(\\"Loaded DeepCenter add-only gate checkpoint:\\", checkpoint_path)\\n            print(\\"DeepCenter checkpoint epoch:\\", checkpoint.get(\\"epoch\\"), \\"best_score:\\", checkpoint.get(\\"best_score\\"))\\n            return {\\n                \\"model\\": model,\\n                \\"cfg\\": cfg,\\n                \\"device\\": device,\\n                \\"path\\": checkpoint_path,\\n                \\"torch\\": torch,\\n            }\\n        except Exception as exc:\\n            load_errors.append(f\\"{checkpoint_path}: {type(exc).__name__}: {exc}\\")\\n            print(\\"Skipping incompatible DeepCenter checkpoint:\\", checkpoint_path, \\"|\\", type(exc).__name__, exc)\\n    message = \\"No usable DeepCenter checkpoint found for add-only repair gate.\\"\\n    if REQUIRE_DEEPCENTER_VETO:\\n        checked = \\"\\\\n\\".join(str(p) for p in _dc_checkpoint_candidates()[:80])\\n        errors = \\"\\\\n\\".join(load_errors[-20:])\\n        raise FileNotFoundError(message + \\"\\\\nChecked:\\\\n\\" + checked + (\\"\\\\nLoad errors:\\\\n\\" + errors if errors else \\"\\"))\\n    print(message)\\n    return None\\n\\n\\ndef _dc_cache_trim(cache: dict[tuple[str, int], np.ndarray]) -> None:\\n    limit = max(1, int(DEEPCENTER_SCORE_CACHE_MAX_FRAMES))\\n    while len(cache) > limit:\\n        cache.pop(next(iter(cache)))\\n\\n\\ndef deepcenter_heatmap_for_frame(\\n    dataset: str,\\n    t: int,\\n    detector_bundle: dict[str, object] | None,\\n    frame_cache: dict[int, np.ndarray],\\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\\n) -> np.ndarray | None:\\n    if detector_bundle is None:\\n        return None\\n    key = (dataset, int(t))\\n    cached = heatmap_cache.get(key)\\n    if cached is not None:\\n        return cached\\n    model = detector_bundle[\\"model\\"]\\n    cfg = detector_bundle[\\"cfg\\"]\\n    device = detector_bundle[\\"device\\"]\\n    torch_mod = detector_bundle[\\"torch\\"]\\n    pool_factor = int(getattr(cfg, \\"pool_factor\\", 4))\\n    volume = read_test_frame(dataset, int(t), frame_cache)\\n    pooled = _dc_pool_frame_xy(volume, pool_factor)\\n    image = _dc_normalize_dynamic_range(pooled, cfg)\\n    with torch_mod.no_grad():\\n        tensor = torch_mod.from_numpy(image[None, None, ...]).to(device=device, dtype=torch_mod.float32)\\n        logits = model(tensor)\\n        \\n        \\n        \\n        \\n        \\n        if os.environ.get(\\"BIOHUB_DEEPCENTER_TTA\\", \\"0\\") != \\"0\\":\\n            acc = logits.clone(); nv = 1\\n            for dims in [(-1,), (-2,), (-2, -1)]:\\n                acc = acc + model(tensor.flip(dims)).flip(dims); nv += 1\\n            if tensor.shape[-1] == tensor.shape[-2]:\\n                for k in (1, 3):\\n                    acc = acc + torch_mod.rot90(model(torch_mod.rot90(tensor, k, dims=(-2, -1))), -k, dims=(-2, -1)); nv += 1\\n                acc = acc + model(tensor.transpose(-1, -2)).transpose(-1, -2); nv += 1\\n                at = torch_mod.rot90(tensor, 1, dims=(-2, -1)).transpose(-1, -2)\\n                acc = acc + torch_mod.rot90(model(at).transpose(-1, -2), -1, dims=(-2, -1)); nv += 1\\n            delta = float((acc / nv - logits).abs().mean())\\n            if delta == 0.0:\\n                raise RuntimeError(\\"DEEPCENTER_TTA_NO_OP: averaged veto logits identical to the single view\\")\\n            if not getattr(deepcenter_heatmap_for_frame, \\"_tta_announced\\", False):\\n                print(\\"DEEPCENTER_TTA_ACTIVE views=\\", nv, \\"mean_abs_logit_delta=\\", round(delta, 6), flush=True)\\n                deepcenter_heatmap_for_frame._tta_announced = True\\n            logits = acc / nv\\n        heatmap = torch_mod.sigmoid(logits)[0, 0].detach().cpu().numpy().astype(np.float32, copy=False)\\n    heatmap_cache[key] = heatmap\\n    _dc_cache_trim(heatmap_cache)\\n    return heatmap\\n\\n\\ndef deepcenter_score_point(\\n    dataset: str | None,\\n    t: int,\\n    point: tuple[float, float, float],\\n    detector_bundle: dict[str, object] | None,\\n    frame_cache: dict[int, np.ndarray],\\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\\n) -> float | None:\\n    if not USE_DEEPCENTER_VETO or detector_bundle is None or dataset is None:\\n        return None\\n    heatmap = deepcenter_heatmap_for_frame(dataset, int(t), detector_bundle, frame_cache, heatmap_cache)\\n    if heatmap is None or heatmap.size == 0:\\n        return None\\n    cfg = detector_bundle[\\"cfg\\"]\\n    pool_factor = int(getattr(cfg, \\"pool_factor\\", 4))\\n    z = int(round(float(point[0])))\\n    y = int(round(float(point[1]) / max(pool_factor, 1)))\\n    x = int(round(float(point[2]) / max(pool_factor, 1)))\\n    z0, z1 = max(0, z - DEEPCENTER_SCORE_WIN_Z), min(heatmap.shape[0], z + DEEPCENTER_SCORE_WIN_Z + 1)\\n    y0, y1 = max(0, y - DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[1], y + DEEPCENTER_SCORE_WIN_YX + 1)\\n    x0, x1 = max(0, x - DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[2], x + DEEPCENTER_SCORE_WIN_YX + 1)\\n    patch = heatmap[z0:z1, y0:y1, x0:x1]\\n    if patch.size == 0:\\n        return None\\n    score = float(np.max(patch))\\n    return score if np.isfinite(score) else None\\n\\n\\ndef deepcenter_accept_repair_point(\\n    dataset: str | None,\\n    t: int,\\n    point: tuple[float, float, float],\\n    detector_bundle: dict[str, object] | None,\\n    frame_cache: dict[int, np.ndarray],\\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\\n    stats: dict[str, int],\\n    prefix: str,\\n    threshold: float,\\n) -> bool:\\n    if not USE_DEEPCENTER_VETO:\\n        return True\\n    if detector_bundle is None or dataset is None:\\n        stats[f\\"deepcenter_{prefix}_missing\\"] += 1\\n        return True\\n    stats[f\\"deepcenter_{prefix}_checked\\"] += 1\\n    score = deepcenter_score_point(dataset, int(t), point, detector_bundle, frame_cache, heatmap_cache)\\n    if score is None:\\n        stats[f\\"deepcenter_{prefix}_missing\\"] += 1\\n        return True\\n    if score < float(threshold):\\n        stats[f\\"deepcenter_{prefix}_rejected\\"] += 1\\n        return False\\n    stats[f\\"deepcenter_{prefix}_accepted\\"] += 1\\n    return True\\n\\ndef _position_um(node: dict[str, object]) -> np.ndarray:\\n    return np.array(\\n        [float(node[\\"z\\"]) * VOXEL_SCALE_UM[0], float(node[\\"y\\"]) * VOXEL_SCALE_UM[1], float(node[\\"x\\"]) * VOXEL_SCALE_UM[2]],\\n        dtype=np.float64,\\n    )\\n\\n\\ndef motion_relink_edges(\\n    nodes_by_id: dict[int, dict[str, object]],\\n    stats: dict[str, int],\\n    learned_edge_probs: dict[tuple[int, int], float] | None = None,\\n    fixed_edges: list[dict[str, object]] | None = None,\\n) -> list[dict[str, object]]:\\n    if not OUTPUT_MOTION_RELINK or not nodes_by_id:\\n        return []\\n\\n    learned_edge_probs = learned_edge_probs or {}\\n\\n    def learned_prob(source_id: int, target_id: int) -> float:\\n        value = learned_edge_probs.get((source_id, target_id), 0.0)\\n        try:\\n            value = float(value)\\n        except (TypeError, ValueError):\\n            return 0.0\\n        if not np.isfinite(value):\\n            return 0.0\\n        if value < 0.0 or value > 1.0:\\n            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))\\n        return float(np.clip(value, 0.0, 1.0))\\n\\n    ids_by_t: dict[int, list[int]] = {}\\n    for node_id, node in nodes_by_id.items():\\n        ids_by_t.setdefault(int(node[\\"t\\"]), []).append(node_id)\\n    for ids in ids_by_t.values():\\n        ids.sort()\\n\\n    frame_sizes = [len(ids) for ids in ids_by_t.values()]\\n    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:\\n        stats[\\"motion_relink_skipped_large_frame\\"] = 1\\n        return []\\n\\n    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}\\n    fixed_edges = fixed_edges or []\\n    fixed_out = {int(e[\'source_id\']) for e in fixed_edges}\\n    fixed_in = {int(e[\'target_id\']) for e in fixed_edges}\\n    predecessor_position_um = {\\n        int(e[\'target_id\']): position_um[int(e[\'source_id\'])] for e in fixed_edges\\n    }\\n    selected_edges = [dict(e) for e in fixed_edges]\\n\\n    def assign_pass(\\n        source_ids: list[int],\\n        target_ids: list[int],\\n        gate_um: float,\\n    ) -> list[tuple[int, int, float, float, float]]:\\n        if not source_ids or not target_ids:\\n            return []\\n        big = gate_um * 1000.0 + 1.0\\n        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)\\n        raw_dist = np.full_like(cost, np.inf)\\n        motion_dist = np.full_like(cost, np.inf)\\n        prob_matrix = np.zeros_like(cost)\\n        for i, source_id in enumerate(source_ids):\\n            source_pos = position_um[source_id]\\n            prev_pos = predecessor_position_um.get(source_id)\\n            if prev_pos is None:\\n                predicted = source_pos\\n            else:\\n                predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)\\n            for j, target_id in enumerate(target_ids):\\n                target_pos = position_um[target_id]\\n                raw = float(np.linalg.norm(target_pos - source_pos))\\n                if raw > gate_um:\\n                    continue\\n                motion = float(np.linalg.norm(target_pos - predicted))\\n                prob = learned_prob(source_id, target_id)\\n                raw_dist[i, j] = raw\\n                motion_dist[i, j] = motion\\n                prob_matrix[i, j] = prob\\n                cost[i, j] = motion + 0.05 * raw - MOTION_RELINK_LEARNED_BONUS * prob\\n        row_ind, col_ind = linear_sum_assignment(cost)\\n        matches: list[tuple[int, int, float, float, float]] = []\\n        for r, c in zip(row_ind, col_ind):\\n            if cost[r, c] >= big:\\n                continue\\n            matches.append((\\n                source_ids[int(r)],\\n                target_ids[int(c)],\\n                float(raw_dist[r, c]),\\n                float(motion_dist[r, c]),\\n                float(prob_matrix[r, c]),\\n            ))\\n        return matches\\n\\n    times = sorted(ids_by_t)\\n    for t in times:\\n        source_ids = [n for n in ids_by_t.get(t, []) if n not in fixed_out]\\n        target_ids = [n for n in ids_by_t.get(t + 1, []) if n not in fixed_in]\\n        if not source_ids or not target_ids:\\n            continue\\n        unmatched_sources = set(source_ids)\\n        unmatched_targets = set(target_ids)\\n        frame_matches: list[tuple[int, int, float, float, str, float]] = []\\n        for pass_name, gate_um in ((\\"tight\\", MOTION_RELINK_TIGHT_UM), (\\"relaxed\\", MOTION_RELINK_RELAXED_UM)):\\n            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]\\n            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]\\n            matches = assign_pass(pass_sources, pass_targets, gate_um)\\n            for source_id, target_id, raw, motion, prob in matches:\\n                if source_id not in unmatched_sources or target_id not in unmatched_targets:\\n                    continue\\n                unmatched_sources.remove(source_id)\\n                unmatched_targets.remove(target_id)\\n                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))\\n                if pass_name == \\"tight\\":\\n                    stats[\\"motion_relink_tight_edges\\"] += 1\\n                else:\\n                    stats[\\"motion_relink_relaxed_edges\\"] += 1\\n        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:\\n            selected_edges.append({\\n                \\"source_id\\": source_id,\\n                \\"target_id\\": target_id,\\n                \\"edge_prob\\": prob,\\n                \\"distance_um\\": raw,\\n                \\"motion_distance_um\\": motion,\\n                \\"motion_relinked\\": 1,\\n                \\"motion_pass\\": pass_name,\\n            })\\n            predecessor_position_um[target_id] = position_um[source_id]\\n        stats[\\"motion_relink_frames\\"] += 1\\n\\n    stats[\\"motion_relink_edges\\"] = len(selected_edges)\\n    return selected_edges\\n\\ndef close_single_frame_gaps(\\n    nodes_by_id: dict[int, dict[str, object]],\\n    edges: list[dict[str, object]],\\n    stats: dict[str, int],\\n    dataset: str | None = None,\\n    deepcenter_bundle: dict[str, object] | None = None,\\n    frame_cache: dict[int, np.ndarray] | None = None,\\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\\n    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:\\n        return nodes_by_id, edges\\n\\n    outgoing = {int(edge[\\"source_id\\"]) for edge in edges}\\n    incoming = {int(edge[\\"target_id\\"]) for edge in edges}\\n    incident = outgoing | incoming\\n\\n    ends_by_t: dict[int, list[int]] = {}\\n    starts_by_t: dict[int, list[int]] = {}\\n    isolated_by_t: dict[int, list[int]] = {}\\n    all_ids_by_t: dict[int, list[int]] = {}\\n    for node_id, node in nodes_by_id.items():\\n        t = int(node[\\"t\\"])\\n        all_ids_by_t.setdefault(t, []).append(node_id)\\n        if node_id not in outgoing:\\n            ends_by_t.setdefault(t, []).append(node_id)\\n        if node_id not in incoming:\\n            starts_by_t.setdefault(t, []).append(node_id)\\n        if node_id not in incident:\\n            isolated_by_t.setdefault(t, []).append(node_id)\\n\\n    max_synthetic = min(\\n        GAP_CLOSE_MAX_ADDED_ABS,\\n        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,\\n    )\\n    next_id = _next_node_id(nodes_by_id)\\n    frame_cache = frame_cache if frame_cache is not None else {}\\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\\n    used_starts: set[int] = set()\\n    used_isolated: set[int] = set()\\n    synthetic_added = 0\\n    new_edges: list[dict[str, object]] = []\\n\\n    density_cache: dict[int, dict[int, float]] = {}\\n\\n    def frame_local_spacing(t: int) -> dict[int, float]:\\n        cached = density_cache.get(t)\\n        if cached is not None:\\n            return cached\\n\\n        frame_ids = all_ids_by_t.get(t, [])\\n        if len(frame_ids) <= 1:\\n            result = {\\n                node_id: GAP_DENSITY_REFERENCE_UM\\n                for node_id in frame_ids\\n            }\\n            density_cache[t] = result\\n            return result\\n\\n        positions = np.stack(\\n            [_position_um(nodes_by_id[node_id]) for node_id in frame_ids]\\n        )\\n        tree = cKDTree(positions)\\n        query_k = min(\\n            len(frame_ids),\\n            max(2, GAP_DENSITY_NEIGHBORS + 1),\\n        )\\n        distances, _ = tree.query(positions, k=query_k)\\n        if distances.ndim == 1:\\n            distances = distances[:, None]\\n\\n        result: dict[int, float] = {}\\n        for idx, node_id in enumerate(frame_ids):\\n            neighbour_distances = distances[idx, 1:]\\n            neighbour_distances = neighbour_distances[\\n                np.isfinite(neighbour_distances)\\n            ]\\n            spacing = (\\n                float(np.median(neighbour_distances))\\n                if neighbour_distances.size\\n                else GAP_DENSITY_REFERENCE_UM\\n            )\\n            result[node_id] = spacing\\n\\n        density_cache[t] = result\\n        stats[\\"gap_density_nodes_scored\\"] += len(result)\\n        return result\\n\\n    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)\\n    stats[\\"gap_close_effective_max_gap\\"] = effective_gap_max\\n    for gap in range(1, effective_gap_max + 1):\\n        for t, end_ids in sorted(ends_by_t.items()):\\n            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]\\n            if not end_ids or not start_ids:\\n                continue\\n\\n            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]\\n            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]\\n            threshold_um = GAP_CLOSE_UM * (gap + 1)\\n            d = np.zeros(\\n                (len(end_ids), len(start_ids)),\\n                dtype=np.float64,\\n            )\\n            adaptive_threshold = np.full_like(d, threshold_um)\\n\\n            source_spacing = frame_local_spacing(t)\\n            target_spacing = frame_local_spacing(t + gap + 1)\\n\\n            for i, ep in enumerate(end_points):\\n                for j, sp in enumerate(start_points):\\n                    d[i, j] = point_distance_um(ep, sp)\\n\\n                    if GAP_DENSITY_ADAPTIVE:\\n                        local_spacing = 0.5 * (\\n                            source_spacing.get(\\n                                end_ids[i],\\n                                GAP_DENSITY_REFERENCE_UM,\\n                            )\\n                            + target_spacing.get(\\n                                start_ids[j],\\n                                GAP_DENSITY_REFERENCE_UM,\\n                            )\\n                        )\\n                        step_delta = float(\\n                            np.clip(\\n                                GAP_DENSITY_GAIN\\n                                * (\\n                                    local_spacing\\n                                    - GAP_DENSITY_REFERENCE_UM\\n                                ),\\n                                -GAP_DENSITY_MAX_STEP_DELTA_UM,\\n                                GAP_DENSITY_MAX_STEP_DELTA_UM,\\n                            )\\n                        )\\n                        adaptive_threshold[i, j] = (\\n                            threshold_um + step_delta * (gap + 1)\\n                        )\\n                        stats[\\n                            \\"gap_density_step_delta_milli_sum\\"\\n                        ] += int(round(1000.0 * step_delta))\\n\\n            base_allowed = d <= threshold_um\\n            adaptive_allowed = d <= adaptive_threshold\\n\\n            stats[\\"gap_density_candidates_expanded\\"] += int(\\n                (adaptive_allowed & ~base_allowed).sum()\\n            )\\n            stats[\\"gap_density_candidates_restricted\\"] += int(\\n                (base_allowed & ~adaptive_allowed).sum()\\n            )\\n            stats[\\"gap_candidates\\"] += int(adaptive_allowed.sum())\\n\\n            if not np.isfinite(d).any():\\n                continue\\n\\n            max_threshold = float(np.max(adaptive_threshold))\\n            big = max_threshold * 1000.0 + 1.0\\n            cost = np.where(adaptive_allowed, d, big)\\n            row_ind, col_ind = linear_sum_assignment(cost)\\n\\n            for r, c in zip(row_ind, col_ind):\\n                if not adaptive_allowed[r, c]:\\n                    continue\\n                if not base_allowed[r, c]:\\n                    stats[\\n                        \\"gap_density_selected_outside_base\\"\\n                    ] += 1\\n                source_id = end_ids[int(r)]\\n                target_id = start_ids[int(c)]\\n                if source_id in outgoing or target_id in used_starts:\\n                    continue\\n\\n                source = nodes_by_id[source_id]\\n                target = nodes_by_id[target_id]\\n                mid_t = int(source[\\"t\\"]) + gap\\n                mid_point = (\\n                    (float(source[\\"z\\"]) + float(target[\\"z\\"])) / 2.0,\\n                    (float(source[\\"y\\"]) + float(target[\\"y\\"])) / 2.0,\\n                    (float(source[\\"x\\"]) + float(target[\\"x\\"])) / 2.0,\\n                )\\n\\n                middle_id: int | None = None\\n                middle_reused = False\\n                if GAP_CLOSE_REUSE_EXISTING:\\n                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]\\n                    if candidates:\\n                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]\\n                        best_idx = int(np.argmin(distances))\\n                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:\\n                            middle_id = candidates[best_idx]\\n                            middle_reused = True\\n\\n                if middle_id is None:\\n                    if synthetic_added >= max_synthetic:\\n                        stats[\\"gap_skipped_node_cap\\"] += 1\\n                        continue\\n                    middle_id = next_id\\n                    next_id += 1\\n                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)\\n                    nodes_by_id[middle_id] = {\\n                        \\"node_id\\": middle_id,\\n                        \\"t\\": mid_t,\\n                        \\"z\\": refined_point[0],\\n                        \\"y\\": refined_point[1],\\n                        \\"x\\": refined_point[2],\\n                        \\"gap_synthetic\\": 1,\\n                    }\\n                    synthetic_added += 1\\n                    stats[\\"gap_inserted_synthetic\\"] += 1\\n\\n                middle = nodes_by_id[middle_id]\\n                gap_span_um = float(d[r, c])\\n                marginal_gap = gap_span_um >= DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM\\n                synthetic_middle = int(middle.get(\\"gap_synthetic\\", 0)) == 1\\n                requires_center_confirmation = (\\n                    DEEPCENTER_GAP_VETO and marginal_gap and synthetic_middle\\n                )\\n                if DEEPCENTER_GAP_VETO and not marginal_gap:\\n                    stats[\\"deepcenter_gap_bypassed_strong_motion\\"] += 1\\n                elif DEEPCENTER_GAP_VETO and not synthetic_middle:\\n                    stats[\\"deepcenter_gap_bypassed_observed_node\\"] += 1\\n                if requires_center_confirmation and not deepcenter_accept_repair_point(\\n                    dataset,\\n                    mid_t,\\n                    node_point(middle),\\n                    deepcenter_bundle,\\n                    frame_cache,\\n                    deepcenter_cache,\\n                    stats,\\n                    \\"gap\\",\\n                    DEEPCENTER_GAP_THRESHOLD,\\n                ):\\n                    if int(middle.get(\\"gap_synthetic\\", 0)) == 1:\\n                        nodes_by_id.pop(middle_id, None)\\n                        synthetic_added = max(0, synthetic_added - 1)\\n                        stats[\\"gap_inserted_synthetic\\"] = max(0, stats[\\"gap_inserted_synthetic\\"] - 1)\\n                    continue\\n                if middle_reused:\\n                    used_isolated.add(middle_id)\\n                    stats[\\"gap_reused_existing\\"] += 1\\n\\n                e1 = {\\n                    \\"source_id\\": source_id,\\n                    \\"target_id\\": middle_id,\\n                    \\"edge_prob\\": None,\\n                    \\"distance_um\\": edge_distance_um(source, middle),\\n                    \\"gap_closed\\": 1,\\n                }\\n                e2 = {\\n                    \\"source_id\\": middle_id,\\n                    \\"target_id\\": target_id,\\n                    \\"edge_prob\\": None,\\n                    \\"distance_um\\": edge_distance_um(middle, target),\\n                    \\"gap_closed\\": 1,\\n                }\\n                new_edges.extend([e1, e2])\\n                outgoing.add(source_id)\\n                incoming.add(middle_id)\\n                outgoing.add(middle_id)\\n                incoming.add(target_id)\\n                used_starts.add(target_id)\\n                stats[\\"gap_pairs_selected\\"] += 1\\n                stats[\\"gap_added_edges\\"] += 2\\n\\n    if new_edges:\\n        edges = [*edges, *new_edges]\\n    stats[\\"gap_added_nodes\\"] = stats[\\"gap_inserted_synthetic\\"]\\n    return nodes_by_id, edges\\n\\n\\ndef _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:\\n    by_source: dict[int, list[int]] = {}\\n    for edge in edges:\\n        by_source.setdefault(int(edge[\\"source_id\\"]), []).append(int(edge[\\"target_id\\"]))\\n    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}\\n\\n\\ndef _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:\\n    by_target: dict[int, list[int]] = {}\\n    for edge in edges:\\n        by_target.setdefault(int(edge[\\"target_id\\"]), []).append(int(edge[\\"source_id\\"]))\\n    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}\\n\\n\\ndef recover_strict_gap2(\\n    nodes_by_id: dict[int, dict[str, object]],\\n    edges: list[dict[str, object]],\\n    stats: dict[str, int],\\n    dataset: str | None = None,\\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\\n    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:\\n        return nodes_by_id, edges\\n\\n    outgoing = {int(edge[\\"source_id\\"]) for edge in edges}\\n    incoming = {int(edge[\\"target_id\\"]) for edge in edges}\\n    predecessor = _single_predecessor_map(edges)\\n    successor = _single_successor_map(edges)\\n\\n    ends_by_t: dict[int, list[int]] = {}\\n    starts_by_t: dict[int, list[int]] = {}\\n    for node_id, node in nodes_by_id.items():\\n        t = int(node[\\"t\\"])\\n        if node_id not in outgoing:\\n            ends_by_t.setdefault(t, []).append(node_id)\\n        if node_id not in incoming:\\n            starts_by_t.setdefault(t, []).append(node_id)\\n\\n    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))\\n    proposals: list[tuple[float, int, int, int, float]] = []\\n\\n    def pos_um(node_id: int) -> np.ndarray:\\n        node = nodes_by_id[node_id]\\n        return np.array([float(node[\\"z\\"]), float(node[\\"y\\"]), float(node[\\"x\\"])], dtype=np.float64) * np.array(VOXEL_SCALE_UM)\\n\\n    for t, end_ids in sorted(ends_by_t.items()):\\n        start_ids = starts_by_t.get(t + 3, [])\\n        if not end_ids or not start_ids:\\n            continue\\n        for end_id in end_ids:\\n            end_pos = pos_um(end_id)\\n            for start_id in start_ids:\\n                start_pos = pos_um(start_id)\\n                dist = float(np.linalg.norm(start_pos - end_pos))\\n                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:\\n                    continue\\n                step = (start_pos - end_pos) / 3.0\\n                context_penalty = 0.0\\n                if GAP2_REQUIRE_CONTEXT:\\n                    ok_context = False\\n                    prev_id = predecessor.get(end_id)\\n                    if prev_id is not None:\\n                        prev_step = end_pos - pos_um(prev_id)\\n                        prev_norm = float(np.linalg.norm(prev_step))\\n                        step_norm = float(np.linalg.norm(step))\\n                        if prev_norm <= 0.01 or step_norm <= 0.01:\\n                            ok_context = True\\n                        else:\\n                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))\\n                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:\\n                                ok_context = True\\n                            context_penalty += max(0.0, 0.25 - cos)\\n                    next_id = successor.get(start_id)\\n                    if next_id is not None:\\n                        next_step = pos_um(next_id) - start_pos\\n                        next_norm = float(np.linalg.norm(next_step))\\n                        step_norm = float(np.linalg.norm(step))\\n                        if next_norm <= 0.01 or step_norm <= 0.01:\\n                            ok_context = True\\n                        else:\\n                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))\\n                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:\\n                                ok_context = True\\n                            context_penalty += max(0.0, 0.25 - cos)\\n                    if not ok_context:\\n                        continue\\n                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))\\n\\n    proposals.sort(key=lambda item: item[0])\\n    stats[\\"gap2_candidates\\"] = len(proposals)\\n    if not proposals:\\n        return nodes_by_id, edges\\n\\n    selected: list[tuple[float, int, int, int, float]] = []\\n    used_ends: set[int] = set()\\n    used_starts: set[int] = set()\\n    per_frame_count: dict[int, int] = {}\\n    for proposal in proposals:\\n        if len(selected) >= cap:\\n            stats[\\"gap2_skipped_cap\\"] += 1\\n            break\\n        _, end_id, start_id, t, _ = proposal\\n        if end_id in used_ends or start_id in used_starts:\\n            continue\\n        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))\\n        if per_frame_count.get(t, 0) >= frame_cap:\\n            continue\\n        selected.append(proposal)\\n        used_ends.add(end_id)\\n        used_starts.add(start_id)\\n        per_frame_count[t] = per_frame_count.get(t, 0) + 1\\n\\n    if not selected:\\n        return nodes_by_id, edges\\n\\n    next_node_id = _next_node_id(nodes_by_id)\\n    frame_cache: dict[int, np.ndarray] = {}\\n    new_edges: list[dict[str, object]] = []\\n    for _, end_id, start_id, t, _ in selected:\\n        source = nodes_by_id[end_id]\\n        target = nodes_by_id[start_id]\\n        previous_id = end_id\\n        inserted_ids: list[int] = []\\n        for k in (1, 2):\\n            frac = k / 3.0\\n            mid_t = int(source[\\"t\\"]) + k\\n            midpoint = (\\n                float(source[\\"z\\"]) + (float(target[\\"z\\"]) - float(source[\\"z\\"])) * frac,\\n                float(source[\\"y\\"]) + (float(target[\\"y\\"]) - float(source[\\"y\\"])) * frac,\\n                float(source[\\"x\\"]) + (float(target[\\"x\\"]) - float(source[\\"x\\"])) * frac,\\n            )\\n            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)\\n            node_id = next_node_id\\n            next_node_id += 1\\n            nodes_by_id[node_id] = {\\n                \\"node_id\\": node_id,\\n                \\"t\\": mid_t,\\n                \\"z\\": refined_point[0],\\n                \\"y\\": refined_point[1],\\n                \\"x\\": refined_point[2],\\n            }\\n            inserted_ids.append(node_id)\\n            current = nodes_by_id[node_id]\\n            new_edges.append({\\n                \\"source_id\\": previous_id,\\n                \\"target_id\\": node_id,\\n                \\"edge_prob\\": None,\\n                \\"distance_um\\": edge_distance_um(nodes_by_id[previous_id], current),\\n                \\"gap2_recovered\\": 1,\\n            })\\n            previous_id = node_id\\n        new_edges.append({\\n            \\"source_id\\": previous_id,\\n            \\"target_id\\": start_id,\\n            \\"edge_prob\\": None,\\n            \\"distance_um\\": edge_distance_um(nodes_by_id[previous_id], target),\\n            \\"gap2_recovered\\": 1,\\n        })\\n        stats[\\"gap2_pairs_selected\\"] += 1\\n        stats[\\"gap2_added_nodes\\"] += len(inserted_ids)\\n        stats[\\"gap2_added_edges\\"] += 3\\n\\n    return nodes_by_id, [*edges, *new_edges]\\n\\n\\ndef add_safe_divisions_postlink(\\n    nodes_by_id: dict[int, dict[str, object]],\\n    edges: list[dict[str, object]],\\n    stats: dict[str, int],\\n    dataset: str | None = None,\\n    deepcenter_bundle: dict[str, object] | None = None,\\n    frame_cache: dict[int, np.ndarray] | None = None,\\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\\n) -> list[dict[str, object]]:\\n    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:\\n        return edges\\n    frame_cache = frame_cache if frame_cache is not None else {}\\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\\n \\n    out_by_source: dict[int, list[dict[str, object]]] = {}\\n    incoming: set[int] = set()\\n    for edge in edges:\\n        out_by_source.setdefault(int(edge[\\"source_id\\"]), []).append(edge)\\n        incoming.add(int(edge[\\"target_id\\"]))\\n \\n    ids_by_t: dict[int, list[int]] = {}\\n    for node_id, node in nodes_by_id.items():\\n        ids_by_t.setdefault(int(node[\\"t\\"]), []).append(node_id)\\n \\n    existing_edges = {(int(edge[\\"source_id\\"]), int(edge[\\"target_id\\"])) for edge in edges}\\n    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))\\n    added: list[dict[str, object]] = []\\n    used_targets: set[int] = set()\\n    used_sources: set[int] = set()  \\n \\n    for t in sorted(ids_by_t):\\n        child_frame_ids = ids_by_t.get(t + 1, [])\\n        if not child_frame_ids:\\n            continue\\n        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]\\n        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]\\n        if not source_ids or not candidate_ids:\\n            continue\\n \\n        \\n        \\n        \\n        \\n        \\n        candidate_tree = None\\n        if SAFE_DIV_REQUIRE_MUTUAL_NN:\\n            candidate_positions = np.stack([_position_um(nodes_by_id[cid]) for cid in candidate_ids])\\n            candidate_tree = cKDTree(candidate_positions)\\n \\n        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))\\n        proposals: list[tuple[float, int, int, float, float]] = []\\n        for source_id in source_ids:\\n            source = nodes_by_id[source_id]\\n            existing_child_edge = out_by_source[source_id][0]\\n            existing_child_id = int(existing_child_edge[\\"target_id\\"])\\n            existing_child = nodes_by_id.get(existing_child_id)\\n            if existing_child is None or int(existing_child[\\"t\\"]) != t + 1:\\n                continue\\n            child_dist = edge_distance_um(source, existing_child)\\n            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:\\n                continue\\n \\n            \\n            \\n            \\n            \\n            mutual_nn_id = None\\n            if candidate_tree is not None:\\n                _, nn_idx = candidate_tree.query(_position_um(existing_child))\\n                mutual_nn_id = candidate_ids[int(nn_idx)]\\n \\n            for candidate_id in candidate_ids:\\n                if (source_id, candidate_id) in existing_edges:\\n                    continue\\n                candidate = nodes_by_id[candidate_id]\\n                parent_dist = edge_distance_um(source, candidate)\\n                if parent_dist > SAFE_DIV_MAX_UM:\\n                    continue\\n                sister_dist = edge_distance_um(existing_child, candidate)\\n                if sister_dist > SAFE_DIV_SISTER_MAX_UM:\\n                    continue\\n \\n                \\n                if SAFE_DIV_REQUIRE_MUTUAL_NN and candidate_id != mutual_nn_id:\\n                    stats[\\"safe_division_mutual_nn_rejected\\"] += 1\\n                    continue\\n \\n                \\n                \\n                \\n                \\n                if SAFE_DIV_REQUIRE_DIVERGENCE:\\n                    c1_succ = out_by_source.get(existing_child_id, [])\\n                    q_succ = out_by_source.get(candidate_id, [])\\n                    if len(c1_succ) != 1 or len(q_succ) != 1:\\n                        stats[\\"safe_division_divergence_rejected\\"] += 1\\n                        continue\\n                    c1_grandchild = nodes_by_id.get(int(c1_succ[0][\\"target_id\\"]))\\n                    q_grandchild = nodes_by_id.get(int(q_succ[0][\\"target_id\\"]))\\n                    if (\\n                        c1_grandchild is None or q_grandchild is None\\n                        or int(c1_grandchild[\\"t\\"]) != t + 2\\n                        or int(q_grandchild[\\"t\\"]) != t + 2\\n                    ):\\n                        stats[\\"safe_division_divergence_rejected\\"] += 1\\n                        continue\\n                    grandchild_dist = edge_distance_um(c1_grandchild, q_grandchild)\\n                    if grandchild_dist - sister_dist < SAFE_DIV_DIVERGE_UM:\\n                        stats[\\"safe_division_divergence_rejected\\"] += 1\\n                        continue\\n \\n                stats[\\"safe_division_geometric_candidates\\"] += 1\\n                if DEEPCENTER_SAFE_DIV_VETO and not deepcenter_accept_repair_point(\\n                    dataset,\\n                    int(candidate[\\"t\\"]),\\n                    node_point(candidate),\\n                    deepcenter_bundle,\\n                    frame_cache,\\n                    deepcenter_cache,\\n                    stats,\\n                    \\"safe_div\\",\\n                    DEEPCENTER_SAFE_DIV_THRESHOLD,\\n                ):\\n                    continue\\n                \\n                \\n                \\n                \\n                \\n                \\n                if SAFE_DIV_SISTER_SYMMETRY_TAU > 0.0:\\n                    _sym_denom = max((child_dist + parent_dist) / 2.0, 1e-6)\\n                    if abs(child_dist - parent_dist) / _sym_denom > SAFE_DIV_SISTER_SYMMETRY_TAU:\\n                        stats[\\"safe_division_symmetry_rejected\\"] += 1\\n                        continue\\n                score = parent_dist + 0.15 * sister_dist\\n                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))\\n \\n        stats[\\"safe_division_candidates\\"] += len(proposals)\\n        if not proposals:\\n            continue\\n        proposals.sort(key=lambda item: item[0])\\n        added_this_frame = 0\\n        for _, source_id, candidate_id, parent_dist, _ in proposals:\\n            if len(added) >= global_cap:\\n                stats[\\"safe_division_skipped_cap\\"] += 1\\n                break\\n            if added_this_frame >= frame_cap:\\n                break\\n            if candidate_id in used_targets or candidate_id in incoming:\\n                continue\\n            if source_id in used_sources:\\n                continue\\n            candidate = nodes_by_id[candidate_id]\\n            added.append({\\n                \\"source_id\\": source_id,\\n                \\"target_id\\": candidate_id,\\n                \\"edge_prob\\": None,\\n                \\"distance_um\\": parent_dist,\\n                \\"safe_division\\": 1,\\n            })\\n            used_targets.add(candidate_id)\\n            used_sources.add(source_id)\\n            added_this_frame += 1\\n \\n    if added:\\n        stats[\\"safe_divisions_added\\"] = len(added)\\n        return [*edges, *added]\\n    return edges\\n\\n\\ndef filter_short_track_components(\\n    nodes_by_id: dict[int, dict[str, object]],\\n    edges: list[dict[str, object]],\\n    stats: dict[str, int],\\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\\n    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:\\n        return nodes_by_id, edges\\n\\n    parent = {node_id: node_id for node_id in nodes_by_id}\\n\\n    def find(node_id: int) -> int:\\n        while parent[node_id] != node_id:\\n            parent[node_id] = parent[parent[node_id]]\\n            node_id = parent[node_id]\\n        return node_id\\n\\n    def union(a: int, b: int) -> None:\\n        if a not in parent or b not in parent:\\n            return\\n        ra = find(a)\\n        rb = find(b)\\n        if ra != rb:\\n            parent[ra] = rb\\n\\n    out_count: dict[int, int] = {}\\n    for edge in edges:\\n        source_id = int(edge[\\"source_id\\"])\\n        target_id = int(edge[\\"target_id\\"])\\n        union(source_id, target_id)\\n        out_count[source_id] = out_count.get(source_id, 0) + 1\\n\\n    components: dict[int, list[int]] = {}\\n    for node_id in nodes_by_id:\\n        components.setdefault(find(node_id), []).append(node_id)\\n\\n    component_edges: dict[int, list[dict[str, object]]] = {root: [] for root in components}\\n    for edge in edges:\\n        source_id = int(edge[\\"source_id\\"])\\n        target_id = int(edge[\\"target_id\\"])\\n        if source_id in parent and target_id in parent:\\n            component_edges.setdefault(find(source_id), []).append(edge)\\n\\n    keep: set[int] = set()\\n    for root, members in components.items():\\n        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)\\n        if len(members) >= OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):\\n            keep.update(members)\\n\\n    if not keep:\\n        stats[\\"short_track_filter_skipped_all\\"] += 1\\n        return nodes_by_id, edges\\n\\n    removed_before_rescue = len(nodes_by_id) - len(keep)\\n    if removed_before_rescue <= 0:\\n        return nodes_by_id, edges\\n\\n    if ADAPTIVE_SHORT_TRACK_RESCUE:\\n        removed_frac = removed_before_rescue / max(len(nodes_by_id), 1)\\n        if removed_frac >= SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC:\\n            budget = min(\\n                SHORT_TRACK_RESCUE_MAX_NODES_ABS,\\n                max(0, int(round(len(nodes_by_id) * SHORT_TRACK_RESCUE_MAX_NODES_FRAC))),\\n            )\\n            stats[\\"short_track_rescue_triggered\\"] = 1\\n            stats[\\"short_track_rescue_budget\\"] = budget\\n            proposals: list[tuple[float, int, float, int, list[int]]] = []\\n            for root, members in components.items():\\n                if set(members) & keep:\\n                    continue\\n                if len(members) < SHORT_TRACK_RESCUE_MIN_LEN or len(members) >= OUTPUT_MIN_TRACK_LEN:\\n                    continue\\n                c_edges = component_edges.get(root, [])\\n                if not c_edges:\\n                    continue\\n                probs: list[float] = []\\n                dists: list[float] = []\\n                for edge in c_edges:\\n                    try:\\n                        prob = float(edge.get(\\"edge_prob\\", 0.0))\\n                    except (TypeError, ValueError):\\n                        prob = 0.0\\n                    if np.isfinite(prob):\\n                        probs.append(prob)\\n                    try:\\n                        dist = float(edge.get(\\"distance_um\\", np.nan))\\n                    except (TypeError, ValueError):\\n                        dist = np.nan\\n                    if np.isfinite(dist):\\n                        dists.append(dist)\\n                mean_prob = float(np.mean(probs)) if probs else 0.0\\n                mean_dist = float(np.mean(dists)) if dists else float(\\"inf\\")\\n                if mean_prob < SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB:\\n                    continue\\n                if mean_dist > SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM:\\n                    continue\\n                score = mean_prob - 0.02 * mean_dist + 0.004 * len(members)\\n                proposals.append((score, len(members), mean_prob, root, members))\\n            proposals.sort(reverse=True)\\n            rescued_nodes = 0\\n            rescued_components = 0\\n            for _, size, _, _, members in proposals:\\n                if budget <= 0 or rescued_nodes + size > budget:\\n                    continue\\n                keep.update(members)\\n                rescued_nodes += size\\n                rescued_components += 1\\n            stats[\\"short_track_rescue_components\\"] = rescued_components\\n            stats[\\"short_track_rescue_nodes\\"] = rescued_nodes\\n\\n    removed_nodes = len(nodes_by_id) - len(keep)\\n    if removed_nodes <= 0:\\n        return nodes_by_id, edges\\n\\n    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}\\n    kept_edges = [\\n        edge for edge in edges\\n        if int(edge[\\"source_id\\"]) in kept_nodes and int(edge[\\"target_id\\"]) in kept_nodes\\n    ]\\n    stats[\\"short_track_components_removed\\"] = sum(1 for members in components.values() if not (set(members) & keep))\\n    stats[\\"short_track_nodes_removed\\"] = removed_nodes\\n    stats[\\"short_track_edges_removed\\"] = len(edges) - len(kept_edges)\\n    return kept_nodes, kept_edges\\n\\n\\ndef linefit_smooth_output_graph(\\n    nodes_by_id: dict[int, dict[str, object]],\\n    edges: list[dict[str, object]],\\n    stats: dict[str, int],\\n) -> dict[int, dict[str, object]]:\\n    \\"\\"\\"Smooth linear track interiors without changing graph topology.\\"\\"\\"\\n    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:\\n        return nodes_by_id\\n\\n    predecessor: dict[int, list[int]] = {}\\n    successor: dict[int, list[int]] = {}\\n    for edge in edges:\\n        source_id = int(edge[\\"source_id\\"])\\n        target_id = int(edge[\\"target_id\\"])\\n        source = nodes_by_id.get(source_id)\\n        target = nodes_by_id.get(target_id)\\n        if source is None or target is None:\\n            continue\\n        if int(target[\\"t\\"]) != int(source[\\"t\\"]) + 1:\\n            continue\\n        successor.setdefault(source_id, []).append(target_id)\\n        predecessor.setdefault(target_id, []).append(source_id)\\n\\n    original_pos = {\\n        node_id: np.array([float(node[\\"z\\"]), float(node[\\"y\\"]), float(node[\\"x\\"])], dtype=np.float64)\\n        for node_id, node in nodes_by_id.items()\\n    }\\n    updated_pos: dict[int, np.ndarray] = {}\\n    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))\\n\\n    for node_id in sorted(nodes_by_id):\\n        neighbourhood: list[tuple[int, int]] = [(0, node_id)]\\n\\n        current = node_id\\n        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):\\n            prev_ids = predecessor.get(current, [])\\n            if len(prev_ids) != 1:\\n                break\\n            current = prev_ids[0]\\n            if current not in original_pos:\\n                break\\n            neighbourhood.append((-step, current))\\n\\n        current = node_id\\n        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):\\n            next_ids = successor.get(current, [])\\n            if len(next_ids) != 1:\\n                break\\n            current = next_ids[0]\\n            if current not in original_pos:\\n                break\\n            neighbourhood.append((step, current))\\n\\n        if len(neighbourhood) < 3:\\n            stats[\\"linefit_skipped_nodes\\"] += 1\\n            continue\\n\\n        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)\\n        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])\\n        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)\\n        if not np.isfinite(fitted).all():\\n            stats[\\"linefit_skipped_nodes\\"] += 1\\n            continue\\n        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted\\n\\n    for node_id, pos in updated_pos.items():\\n        nodes_by_id[node_id][\\"z\\"] = float(pos[0])\\n        nodes_by_id[node_id][\\"y\\"] = float(pos[1])\\n        nodes_by_id[node_id][\\"x\\"] = float(pos[2])\\n\\n    stats[\\"linefit_smoothed_nodes\\"] = len(updated_pos)\\n    return nodes_by_id\\n\\n\\ndef filter_output_graph(nodes_by_id: dict[int, dict[str, object]], raw_edges: list[dict[str, object]], dataset: str | None=None, deepcenter_bundle: dict[str, object] | None=None) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:\\n    stats = {\'raw_edges\': len(raw_edges), \'dropped_nonconsecutive_edges\': 0, \'dropped_long_edges\': 0, \'dropped_multi_parent_edges\': 0, \'dropped_multi_child_edges\': 0, \'dropped_division_edges\': 0, \'gap_candidates\': 0, \'gap_pairs_selected\': 0, \'gap_reused_existing\': 0, \'gap_inserted_synthetic\': 0, \'gap_added_nodes\': 0, \'gap_added_edges\': 0, \'gap_skipped_node_cap\': 0, \'gap_density_nodes_scored\': 0, \'gap_density_candidates_expanded\': 0, \'gap_density_candidates_restricted\': 0, \'gap_density_selected_outside_base\': 0, \'gap_density_step_delta_milli_sum\': 0, \'gap_refined_synthetic\': 0, \'gap_refine_failed\': 0, \'gap_refine_rejected_shift\': 0, \'pruned_isolated_nodes\': 0, \'motion_relink_edges\': 0, \'motion_relink_tight_edges\': 0, \'motion_relink_relaxed_edges\': 0, \'motion_relink_frames\': 0, \'motion_relink_replaced_raw_edges\': 0, \'motion_relink_fallback_raw\': 0, \'motion_relink_skipped_large_frame\': 0, \'gap2_candidates\': 0, \'gap2_pairs_selected\': 0, \'gap2_added_nodes\': 0, \'gap2_added_edges\': 0, \'gap2_skipped_cap\': 0, \'safe_division_candidates\': 0, \'safe_division_geometric_candidates\': 0, \'safe_divisions_added\': 0, \'safe_division_skipped_cap\': 0, \'safe_division_mutual_nn_rejected\': 0, \'safe_division_divergence_rejected\': 0, \'safe_division_symmetry_rejected\': 0, \'deepcenter_gap_checked\': 0, \'deepcenter_gap_bypassed_strong_motion\': 0, \'deepcenter_gap_bypassed_observed_node\': 0, \'deepcenter_gap_accepted\': 0, \'deepcenter_gap_rejected\': 0, \'deepcenter_gap_missing\': 0, \'deepcenter_safe_div_checked\': 0, \'deepcenter_safe_div_accepted\': 0, \'deepcenter_safe_div_rejected\': 0, \'deepcenter_safe_div_missing\': 0, \'short_track_components_removed\': 0, \'short_track_nodes_removed\': 0, \'short_track_edges_removed\': 0, \'short_track_filter_skipped_all\': 0, \'short_track_rescue_triggered\': 0, \'short_track_rescue_components\': 0, \'short_track_rescue_nodes\': 0, \'short_track_rescue_budget\': 0, \'linefit_smoothed_nodes\': 0, \'linefit_skipped_nodes\': 0}\\n    edges: list[dict[str, object]] = []\\n    for edge in raw_edges:\\n        source = nodes_by_id.get(int(edge[\'source_id\']))\\n        target = nodes_by_id.get(int(edge[\'target_id\']))\\n        if source is None or target is None:\\n            continue\\n        if OUTPUT_ENFORCE_NEXT_FRAME and int(target[\'t\']) != int(source[\'t\']) + 1:\\n            stats[\'dropped_nonconsecutive_edges\'] += 1\\n            continue\\n        distance_um = edge_distance_um(source, target)\\n        edge[\'distance_um\'] = distance_um\\n        if OUTPUT_EDGE_MAX_UM > 0 and distance_um > OUTPUT_EDGE_MAX_UM:\\n            stats[\'dropped_long_edges\'] += 1\\n            continue\\n        edges.append(edge)\\n    if OUTPUT_MOTION_RELINK:\\n        learned_edge_probs: dict[tuple[int, int], float] = {}\\n        for edge in edges:\\n            prob = edge.get(\'edge_prob\')\\n            if prob is None:\\n                continue\\n            try:\\n                prob = float(prob)\\n            except (TypeError, ValueError):\\n                continue\\n            if np.isfinite(prob):\\n                key = (int(edge[\'source_id\']), int(edge[\'target_id\']))\\n                learned_edge_probs[key] = max(learned_edge_probs.get(key, float(\'-inf\')), prob)\\n        anchors, anchor_stats = select_fork_anchors(nodes_by_id, edges, min_probability=HF_ANCHOR_MIN_PROB, max_step_um=MOTION_RELINK_TIGHT_UM, max_sister_um=DIV_SISTER_MAX_UM, voxel_scale=VOXEL_SCALE_UM)\\n        stats.update(anchor_stats)\\n        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs, fixed_edges=anchors if HF_PRESERVE_FORKS else None)\\n        if motion_edges:\\n            stats[\'motion_relink_replaced_raw_edges\'] = len(edges)\\n            edges = motion_edges\\n        else:\\n            stats[\'motion_relink_fallback_raw\'] = 1\\n    if OUTPUT_SINGLE_PARENT_REPAIR and edges:\\n        best_by_target: dict[int, dict[str, object]] = {}\\n        for edge in edges:\\n            target_id = int(edge[\'target_id\'])\\n            prev = best_by_target.get(target_id)\\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\\n                best_by_target[target_id] = edge\\n        kept_ids = {id(edge) for edge in best_by_target.values()}\\n        stats[\'dropped_multi_parent_edges\'] = sum((1 for edge in edges if id(edge) not in kept_ids))\\n        edges = [edge for edge in edges if id(edge) in kept_ids]\\n    if OUTPUT_SINGLE_CHILD_REPAIR and edges:\\n        best_by_source: dict[int, dict[str, object]] = {}\\n        for edge in edges:\\n            source_id = int(edge[\'source_id\'])\\n            prev = best_by_source.get(source_id)\\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\\n                best_by_source[source_id] = edge\\n        kept_ids = {id(edge) for edge in best_by_source.values()}\\n        stats[\'dropped_multi_child_edges\'] = sum((1 for edge in edges if id(edge) not in kept_ids))\\n        edges = [edge for edge in edges if id(edge) in kept_ids]\\n    print(f\'  [{dataset}] after edge-filter+motion-relink: {len(nodes_by_id)} nodes, {len(edges)} edges\')\\n    repair_frame_cache: dict[int, np.ndarray] = {}\\n    deepcenter_heatmap_cache: dict[tuple[str, int], np.ndarray] = {}\\n    nodes_by_id, edges = close_single_frame_gaps(nodes_by_id, edges, stats, dataset=dataset, deepcenter_bundle=deepcenter_bundle, frame_cache=repair_frame_cache, deepcenter_cache=deepcenter_heatmap_cache)\\n    edges = add_safe_divisions_postlink(nodes_by_id, edges, stats, dataset=dataset, deepcenter_bundle=deepcenter_bundle, frame_cache=repair_frame_cache, deepcenter_cache=deepcenter_heatmap_cache)\\n    _geo_cands = stats[\'safe_division_geometric_candidates\']\\n    _post_veto_cands = stats[\'safe_division_candidates\']\\n    _rejected_by_dc = _geo_cands - _post_veto_cands\\n    print(f\\"  [{dataset}] after safe-division repair: {len(nodes_by_id)} nodes, {len(edges)} edges (geometric_candidates={_geo_cands}, deepcenter_rejected={_rejected_by_dc}, post_veto_candidates={_post_veto_cands}, added={stats[\'safe_divisions_added\']}, cap_skipped={stats[\'safe_division_skipped_cap\']}, mutual_nn_rejected={stats[\'safe_division_mutual_nn_rejected\']}, divergence_rejected={stats[\'safe_division_divergence_rejected\']})\\")\\n    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)\\n    print(f\'  [{dataset}] after gap-closing (single-frame + gap2): {len(nodes_by_id)} nodes, {len(edges)} edges\')\\n    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:\\n        by_source: dict[int, list[dict[str, object]]] = {}\\n        for edge in edges:\\n            by_source.setdefault(int(edge[\'source_id\']), []).append(edge)\\n        filtered: list[dict[str, object]] = []\\n        for source_id, source_edges in by_source.items():\\n            if len(source_edges) <= 1:\\n                filtered.extend(source_edges)\\n                continue\\n            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)\\n            source = nodes_by_id[source_id]\\n            top1 = ranked[0]\\n            top2 = ranked[1]\\n            d1 = float(top1[\'distance_um\'])\\n            d2 = float(top2[\'distance_um\'])\\n            sister = edge_distance_um(nodes_by_id[int(top1[\'target_id\'])], nodes_by_id[int(top2[\'target_id\'])])\\n            valid_division = max(d1, d2) <= DIV_PARENT_MAX_UM and sister <= DIV_SISTER_MAX_UM and (int(nodes_by_id[int(top1[\'target_id\'])][\'t\']) == int(source[\'t\']) + 1) and (int(nodes_by_id[int(top2[\'target_id\'])][\'t\']) == int(source[\'t\']) + 1)\\n            if valid_division:\\n                filtered.extend([top1, top2])\\n                stats[\'dropped_division_edges\'] += max(0, len(ranked) - 2)\\n            elif DIV_DROP_TO_SINGLE_IF_BAD:\\n                filtered.append(top1)\\n                stats[\'dropped_division_edges\'] += len(ranked) - 1\\n            else:\\n                filtered.extend(ranked)\\n        edges = filtered\\n    if OUTPUT_PRUNE_ISOLATED:\\n        incident = {int(edge[\'source_id\']) for edge in edges} | {int(edge[\'target_id\']) for edge in edges}\\n        if incident:\\n            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}\\n            stats[\'pruned_isolated_nodes\'] = len(nodes_by_id) - len(kept_nodes)\\n            nodes_by_id = kept_nodes\\n            edges = [edge for edge in edges if int(edge[\'source_id\']) in nodes_by_id and int(edge[\'target_id\']) in nodes_by_id]\\n    print(f\'  [{dataset}] after division-geometry-filter+prune-isolated: {len(nodes_by_id)} nodes, {len(edges)} edges\')\\n    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)\\n    print(f\\"  [{dataset}] after short-track filtering: {len(nodes_by_id)} nodes, {len(edges)} edges (components_removed={stats[\'short_track_components_removed\']})\\")\\n    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)\\n    print(f\'  [{dataset}] FINAL: {len(nodes_by_id)} nodes, {len(edges)} edges\')\\n    final_keys = {(int(e[\'source_id\']), int(e[\'target_id\'])) for e in edges}\\n    stats[\'fork_anchor_final_surviving_edges\'] = sum(((int(e[\'source_id\']), int(e[\'target_id\'])) in final_keys for e in locals().get(\'anchors\', [])))\\n    return (nodes_by_id, edges, stats)\\n\\n\\nDEEPCENTER_VETO_DETECTOR = load_deepcenter_veto_detector()\\n\\ndef write_test_submission(tag: str = \\"base\\") -> None:\\n    \\n    \\n    geffs = sorted((REPO_DIR / \\"predictions\\").glob(f\\"*/{METHOD}/split_0/*.geff\\"))\\n    print(f\\"Found {len(geffs)} prediction graphs\\")\\n    if len(geffs) != len(test_stems):\\n        found = {path.stem for path in geffs}\\n        missing = sorted(set(test_stems) - found)\\n        raise RuntimeError(f\\"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}\\")\\n\\n    stats_rows: list[dict[str, object]] = []\\n    seen_datasets: set[str] = set()\\n    row_id = 0\\n    total_nodes = 0\\n    total_edges = 0\\n\\n    with SUBMISSION_PATH.open(\\"w\\", newline=\\"\\") as f:\\n        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)\\n        writer.writeheader()\\n\\n        for geff_path in geffs:\\n            dataset = geff_path.stem\\n            seen_datasets.add(dataset)\\n            graph = graph_from_geff(geff_path)\\n\\n            nodes_by_id: dict[int, dict[str, object]] = {}\\n            for row in graph.node_attrs().iter_rows(named=True):\\n                node_id = int(row[\\"node_id\\"])\\n                nodes_by_id[node_id] = {\\n                    \\"node_id\\": node_id,\\n                    \\"t\\": int(row[\\"t\\"]),\\n                    \\"z\\": float(row[\\"z\\"]),\\n                    \\"y\\": float(row[\\"y\\"]),\\n                    \\"x\\": float(row[\\"x\\"]),\\n                }\\n\\n            raw_edges: list[dict[str, object]] = []\\n            for row in graph.edge_attrs().iter_rows(named=True):\\n                edge_prob = row.get(\\"edge_prob\\") if hasattr(row, \\"get\\") else None\\n                raw_edges.append({\\n                    \\"source_id\\": int(row[\\"source_id\\"]),\\n                    \\"target_id\\": int(row[\\"target_id\\"]),\\n                    \\"edge_prob\\": None if edge_prob is None else float(edge_prob),\\n                })\\n\\n            raw_node_count = len(nodes_by_id)\\n            nodes_by_id, edges, filter_stats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset, deepcenter_bundle=DEEPCENTER_VETO_DETECTOR)\\n            if not nodes_by_id:\\n                raise AssertionError(f\\"{dataset}: post-processing removed every node\\")\\n\\n            for node_id in sorted(nodes_by_id):\\n                node = nodes_by_id[node_id]\\n                writer.writerow({\\n                    \\"id\\": row_id,\\n                    \\"dataset\\": dataset,\\n                    \\"row_type\\": \\"node\\",\\n                    \\"node_id\\": int(node[\\"node_id\\"]),\\n                    \\"t\\": int(node[\\"t\\"]),\\n                    \\"z\\": max(0, int(round(float(node[\\"z\\"])))),\\n                    \\"y\\": max(0, int(round(float(node[\\"y\\"])))),\\n                    \\"x\\": max(0, int(round(float(node[\\"x\\"])))),\\n                    \\"source_id\\": -1,\\n                    \\"target_id\\": -1,\\n                })\\n                row_id += 1\\n\\n            division_sources: dict[int, int] = {}\\n            for edge in edges:\\n                source_id = int(edge[\\"source_id\\"])\\n                target_id = int(edge[\\"target_id\\"])\\n                if source_id not in nodes_by_id or target_id not in nodes_by_id:\\n                    raise AssertionError(f\\"{dataset}: dangling edge after filtering\\")\\n                writer.writerow({\\n                    \\"id\\": row_id,\\n                    \\"dataset\\": dataset,\\n                    \\"row_type\\": \\"edge\\",\\n                    \\"node_id\\": -1,\\n                    \\"t\\": -1,\\n                    \\"z\\": -1,\\n                    \\"y\\": -1,\\n                    \\"x\\": -1,\\n                    \\"source_id\\": source_id,\\n                    \\"target_id\\": target_id,\\n                })\\n                row_id += 1\\n                division_sources[source_id] = division_sources.get(source_id, 0) + 1\\n\\n            node_count = len(nodes_by_id)\\n            edge_count = len(edges)\\n            total_nodes += node_count\\n            total_edges += edge_count\\n            stats_rows.append({\\n                \\"dataset\\": dataset,\\n                \\"raw_nodes\\": raw_node_count,\\n                \\"nodes\\": node_count,\\n                \\"raw_edges\\": filter_stats[\\"raw_edges\\"],\\n                \\"edges\\": edge_count,\\n                \\"division_like_sources\\": sum(1 for count in division_sources.values() if count >= 2),\\n                \\"edge_to_node_ratio\\": edge_count / max(node_count, 1),\\n                \\"gap_added_nodes_frac\\": filter_stats.get(\\"gap_added_nodes\\", 0) / max(raw_node_count, 1),\\n                **filter_stats,\\n            })\\n\\n    expected_datasets = set(test_stems)\\n    missing_datasets = sorted(expected_datasets - seen_datasets)\\n    extra_datasets = sorted(seen_datasets - expected_datasets)\\n    if missing_datasets or extra_datasets:\\n        raise AssertionError({\\"missing\\": missing_datasets[:10], \\"extra\\": extra_datasets[:10]})\\n    assert row_id == total_nodes + total_edges, \\"Internal row counter mismatch\\"\\n    assert total_nodes > 0, \\"No node rows produced\\"\\n\\n    header = SUBMISSION_PATH.open().readline().strip().split(\\",\\")\\n    assert header == CSV_COLUMNS, f\\"Bad CSV header: {header}\\"\\n\\n    stats = pd.DataFrame(stats_rows).sort_values(\\"dataset\\").reset_index(drop=True)\\n    stats[\\"predict_minutes_total\\"] = predict_seconds / 60.0\\n    stats[\\"experiment_tag\\"] = f\\"{EXPERIMENT_TAG}:{tag}\\"\\n    stats.to_csv(RUN_STATS_PATH, index=False)\\n\\n    print(f\\"Wrote {SUBMISSION_PATH} with {row_id:,} rows\\")\\n    print(f\\"Node rows: {total_nodes:,} | edge rows: {total_edges:,}\\")\\n    print(f\\"Wrote {RUN_STATS_PATH}\\")\\n    display(pd.read_csv(SUBMISSION_PATH, nrows=8))\\n\\n\\n# Both arms are exported explicitly in the next cell; no proxy sweep.\\n\\n"], [9, "import csv as _pair_csv\\nimport hashlib as _pair_hashlib\\nimport itertools as _pair_itertools\\nimport json as _pair_json\\nimport math as _pair_math\\nimport shutil as _pair_shutil\\nimport time as _pair_time\\nfrom collections import Counter as _PairCounter\\n\\ndef _pair_digest(path):\\n    digest = _pair_hashlib.sha256()\\n    with path.open(\\"rb\\") as handle:\\n        for block in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\ndef _pair_graphs(path):\\n    \\"\\"\\"Validate CSV and stream one video\'s graph at a time, not the entire test.\\"\\"\\"\\n    with path.open(newline=\\"\\") as handle:\\n        reader = _pair_csv.DictReader(handle)\\n        if reader.fieldnames != CSV_COLUMNS:\\n            raise AssertionError(f\\"Bad columns: {path}\\")\\n        row_number = 0\\n        seen = set()\\n        for dataset, rows in _pair_itertools.groupby(reader, key=lambda row: row[\\"dataset\\"]):\\n            if dataset in seen:\\n                raise AssertionError(\\"Noncontiguous dataset rows\\")\\n            seen.add(dataset)\\n            nodes, edges = {}, set()\\n            indegree, outdegree = _PairCounter(), _PairCounter()\\n            for row in rows:\\n                if int(row[\\"id\\"]) != row_number:\\n                    raise AssertionError(\\"Noncontiguous row ID\\")\\n                row_number += 1\\n                if row[\\"row_type\\"] == \\"node\\":\\n                    node_id, frame = int(row[\\"node_id\\"]), int(row[\\"t\\"])\\n                    position = tuple(float(row[k]) for k in \\"zyx\\")\\n                    if node_id in nodes or frame < 0 or any(\\n                        not _pair_math.isfinite(x) or x < 0 for x in position\\n                    ):\\n                        raise AssertionError(\\"Invalid/duplicate node\\")\\n                    nodes[node_id] = (frame, *position)\\n                elif row[\\"row_type\\"] == \\"edge\\":\\n                    edge = (int(row[\\"source_id\\"]), int(row[\\"target_id\\"]))\\n                    if edge in edges:\\n                        raise AssertionError(\\"Duplicate edge\\")\\n                    edges.add(edge)\\n                    outdegree[edge[0]] += 1\\n                    indegree[edge[1]] += 1\\n                else:\\n                    raise AssertionError(\\"Unexpected row type\\")\\n            if not nodes:\\n                raise AssertionError(\\"Empty graph\\")\\n            for source, target in edges:\\n                if source not in nodes or target not in nodes or nodes[target][0] != nodes[source][0] + 1:\\n                    raise AssertionError(\\"Dangling or nonconsecutive edge\\")\\n            if max(indegree.values(), default=0) > 1 or max(outdegree.values(), default=0) > 2:\\n                raise AssertionError(\\"Invalid lineage degree\\")\\n            yield dataset, nodes, edges, {n for n, d in outdegree.items() if d == 2}\\n        if seen != set(test_stems):\\n            raise AssertionError(\\"Dataset coverage mismatch\\")\\n\\nHF_PRESERVE_FORKS = False\\nHF_SUBMISSION_ARM = \\"division_before_gap2\\"\\nSUBMISSION_PATH = WORKING_DIR / \\"submission.csv\\"\\nRUN_STATS_PATH = WORKING_DIR / \\"run_stats.csv\\"\\n\\"\\"\\"Read-only stage observers for K3 TEST; no candidate decisions or neural calls.\\"\\"\\"\\nfrom collections import Counter\\nfrom copy import deepcopy\\nimport math\\nimport time\\n\\n\\ndef check_graph(nodes, edges):\\n    incoming, outgoing, pairs = Counter(), Counter(), set()\\n    for identity, node in nodes.items():\\n        if (identity != node[\'node_id\'] or node[\'t\'] < 0\\n                or any(not math.isfinite(float(node[k])) for k in \'zyx\')):\\n            raise ValueError(\'K3 invalid node\')\\n    for edge in edges:\\n        a, b = edge[\'source_id\'], edge[\'target_id\']\\n        if (a not in nodes or b not in nodes or (a, b) in pairs\\n                or nodes[b][\'t\'] != nodes[a][\'t\'] + 1):\\n            raise ValueError(\'K3 invalid edge\')\\n        pairs.add((a, b))\\n        incoming[b] += 1\\n        outgoing[a] += 1\\n    if max(incoming.values(), default=0) > 1 or max(outgoing.values(), default=0) > 2:\\n        raise ValueError(\'K3 invalid lineage degree\')\\n\\n\\ndef observed_test_write(ns, writer):\\n    expected = set(ns[\'test_stems\'])\\n    if not expected or ns[\'OUTPUT_MOTION_RELINK\'] is not False:\\n        raise ValueError(\'K3 TEST/config drift\')\\n    names = (\'add_safe_divisions_postlink\', \'recover_strict_gap2\')\\n    originals = {name: ns[name] for name in names}\\n    receipts = {}\\n\\n    def wrap(name):\\n        def observe(nodes, edges, stats, *args, **kwargs):\\n            dataset = kwargs.get(\'dataset\')\\n            records = receipts.setdefault(dataset, [])\\n            if dataset not in expected or len(records) >= 2 or names[len(records)] != name:\\n                raise ValueError(\'K3 missing/repeated/reordered stage\')\\n            check_graph(nodes, edges)\\n            before_nodes, before_edges = deepcopy(nodes), deepcopy(edges)\\n            started = time.monotonic()\\n            result = originals[name](nodes, edges, stats, *args, **kwargs)\\n            after_nodes, after_edges = (nodes, result) if name == names[0] else result\\n            if (any(after_nodes.get(i) != n for i, n in before_nodes.items())\\n                    or after_edges[:len(before_edges)] != before_edges):\\n                raise ValueError(\'K3 stage changed pre-existing nodes/edges/forks\')\\n            check_graph(after_nodes, after_edges)\\n            records.append({\'stage\': name, \'existing_nodes_edges_preserved\': True,\\n                            \'added_nodes\': len(after_nodes) - len(before_nodes),\\n                            \'added_edges\': len(after_edges) - len(before_edges),\\n                            \'seconds\': time.monotonic() - started})\\n            return result\\n        return observe\\n\\n    try:\\n        for name in names:\\n            ns[name] = wrap(name)\\n        writer()\\n    finally:\\n        ns.update(originals)\\n    if set(receipts) != expected or any([r[\'stage\'] for r in rows] != list(names) for rows in receipts.values()):\\n        raise ValueError(\'K3 incomplete TEST stage coverage\')\\n    return receipts\\n\\n_control_start = _pair_time.monotonic()\\nassert OUTPUT_MOTION_RELINK is False and ILP_DIVISION_WEIGHT == 1.2 and HF_PRESERVE_FORKS is False\\nassert _pair_digest(_ps) == \\"ddca518b0e838b2123f30cfcb31819fb159b4e68fc20ef6b3a42003cc2d528ed\\"\\nassert SAFE_DIV_SISTER_SYMMETRY_TAU == 0.6\\n_k3_receipts = observed_test_write(globals(), lambda: write_test_submission(\\"division_before_gap2\\"))\\n_control_graphs = []\\nfor _name, _nodes, _edges, _forks in _pair_graphs(SUBMISSION_PATH):\\n    _control_graphs.append({\\"dataset\\": _name, \\"nodes\\": len(_nodes),\\n                            \\"edges\\": len(_edges), \\"forks\\": len(_forks)})\\n_control_report = {\\n    \\"experiment\\": \\"HF-DIVISION-ORDER-01\\",\\n    \\"variant\\": \\"division_before_gap2\\",\\n    \\"stage_receipts\\": _k3_receipts,\\n    \\"dispatcher_ast_sha256\\": \\"c99f7c665851bd9870f4ddf28b10a16df3a2e7003f3037e43d4475bea4adbf66\\",\\n    \\"observer_source_sha256\\": \\"68cdcb5f0b71a35ab525bc3ef03dc30cd93f5ab511594f7ab36027c7d3057c54\\",\\n    \\"safe_div_sister_symmetry_tau\\": SAFE_DIV_SISTER_SYMMETRY_TAU,\\n    \\"input_split\\": \\"test\\",\\n    \\"output_motion_relink\\": OUTPUT_MOTION_RELINK,\\n    \\"ilp_division_weight\\": ILP_DIVISION_WEIGHT,\\n    \\"inference_source_sha256\\": _pair_digest(_ps),\\n    \\"weight_sha256\\": {\\"primary\\": _primary_actual_sha256, \\"secondary\\": _secondary_actual_sha256, \\"deepcenter\\": _deepcenter_actual_sha256},\\n    \\"source_kernel\\": \\"flexonafft/biohub-harmonic-fusion\\",\\n    \\"source_version\\": 30,\\n    \\"reference\\": \\"HF-MOTION-OFF-01 version 1\\",\\n    \\"status\\": \\"mechanics_pass_quality_unmeasured\\",\\n    \\"quality_score\\": None,\\n    \\"train_proxy_sweep\\": False,\\n    \\"fork_preserve\\": False,\\n    \\"postprocessing_passes\\": 1,\\n    \\"csv_sha256\\": _pair_digest(SUBMISSION_PATH),\\n    \\"graphs\\": _control_graphs,\\n    \\"stage_stats\\": pd.read_csv(RUN_STATS_PATH).to_dict(orient=\\"records\\"),\\n    \\"postprocessing_seconds\\": _pair_time.monotonic() - _control_start,\\n}\\nfor _s in _control_report[\\"stage_stats\\"]:\\n    for _k in (\\"motion_relink_edges\\", \\"motion_relink_frames\\", \\"motion_relink_replaced_raw_edges\\"):\\n        assert _s[_k] == 0, (_s[\\"dataset\\"], _k, _s[_k])\\n(WORKING_DIR / \\"experiment_diagnostics.json\\").write_text(\\n    _pair_json.dumps(_control_report, indent=2, allow_nan=False) + \\"\\\\n\\"\\n)\\nprint(_pair_json.dumps({k: v for k, v in _control_report.items() if k != \\"stage_stats\\"}, indent=2))\\nprint(\\"K3 order/preservation/CSV contracts passed; public quality unmeasured. No submission API is called inside this notebook.\\")\\n"]]', 'k3_child.py': "import json, os, sys, time\nfrom pathlib import Path\nroot = Path(sys.argv[1])\nns = {'__name__': '__main__'}\nstarted = time.monotonic()\nfor index, source in json.loads(Path('k3_cells.json').read_text()):\n    exec(compile(source, 'frozen_k3_cell_'+str(index), 'exec'), ns)\n    if index == 4:\n        # Filesystem routing only, before model loading or numerical computation.\n        ns.update(WORKING_DIR=root, REPO_DIR=root/'tracking_repo',\n                  SUBMISSION_PATH=root/'submission.csv', RUN_STATS_PATH=root/'run_stats.csv')\nreport = json.loads((root/'experiment_diagnostics.json').read_text())\nreport['combined_child_seconds'] = time.monotonic()-started\nreport['combined_test_stems'] = ns['test_stems']\n(root/'experiment_diagnostics.json').write_text(json.dumps(report, allow_nan=False))\n", 'hengck_child.py': "import json, sys\nfrom pathlib import Path\nmodel = Path('/kaggle/input/datasets/hengck23/hengck23-cell-point-detector-demo')\nsys.path.insert(0, str(model))\nfrom hengck_test_runtime import run\nreport = run(model, sys.argv[1], sys.argv[2])\n(Path(sys.argv[1])/'summary.json').write_text(json.dumps(report, allow_nan=False))\n", 'production.py': '"""Sequential GPU subprocesses; only final submission and aggregate receipts escape tmp."""\nimport hashlib\nimport json\nimport os\nimport re\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom src.k3_hengck_production_adapter import transfer_csv\n\n\ndef run_child(command, directory, seconds, log):\n    with Path(log).open(\'w\') as stream:\n        process = subprocess.Popen(command, cwd=directory, stdout=stream, stderr=subprocess.STDOUT,\n                                   start_new_session=True)\n        try:\n            code = process.wait(timeout=seconds)\n            if code:\n                stream.flush()\n                lines = Path(log).read_text(errors=\'replace\').splitlines()\n                errors = [line[:400] for line in lines if re.match(r\'^(?:[A-Za-z.]+Error|AssertionError|TimeoutError):\', line)]\n                raise RuntimeError(\'Child process failed: \'+Path(command[1]).name+\' exit \'+str(code)+\'; \'+\'; \'.join(errors[-3:]))\n        finally:\n            # Also reap any grandchildren after a failed inference or timeout.\n            try:\n                os.killpg(process.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n            process.wait()\n\n\ndef run(runtime, output, hashes):\n    started = time.monotonic()\n    runtime, output = Path(runtime), Path(output)\n    competition = \'biohub-cell-tracking-during-development\'\n    roots = [Path(\'/kaggle/input/competitions\')/competition, Path(\'/kaggle/input\')/competition]\n    test = next(p/\'test\' for p in roots if (p/\'test\').is_dir())\n    expected = sorted(p.stem for p in test.glob(\'*.zarr\'))\n    assert expected and len(expected) == len(set(expected))\n    hg, k3 = runtime/\'hengck\', runtime/\'k3\'\n    hg.mkdir(); k3.mkdir()\n    def remaining():\n        value = 42900-(time.monotonic()-started)\n        if value <= 0:\n            raise TimeoutError(\'Combined runtime budget exhausted\')\n        return value\n    print(\'STAGE Hengck starting\', flush=True)\n    run_child([sys.executable, \'hengck_child.py\', str(hg), str(test)], runtime, remaining(), runtime/\'hengck.log\')\n    hreport = json.loads((hg/\'summary.json\').read_text())\n    assert set(hreport[\'videos\']) == set(expected)\n    print(\'STAGE Hengck complete; GPU process exited; K3 starting\', flush=True)\n    run_child([sys.executable, \'k3_child.py\', str(k3)], runtime, remaining(), runtime/\'k3.log\')\n    kreport = json.loads((k3/\'experiment_diagnostics.json\').read_text())\n    assert sorted(kreport[\'combined_test_stems\']) == expected\n    print(\'STAGE K3 complete; integer CSV transfer starting\', flush=True)\n    transfer_start = time.monotonic()\n    receipt = transfer_csv(k3/\'submission.csv\', runtime/\'final.csv\', hg, expected)\n    assert receipt[\'csv\'][\'baseline_sha256\'] == kreport[\'csv_sha256\']\n    remaining()\n    report = dict(experiment=\'K3-HENGCK-FORK-TRANSFER-PRODUCTION-01\', complete=True,\n        input_split=\'test\', quality_score=None, training_run=False, gt_read=False,\n        models_sequential=True, all_test_coverage=True, test_stems=expected,\n        runtime_file_sha256=hashes, hengck=hreport, k3=kreport, transfer=receipt,\n        transfer_seconds=time.monotonic()-transfer_start, elapsed_seconds=time.monotonic()-started,\n        csv_sha256=receipt[\'csv\'][\'output_sha256\'])\n    shutil.copyfile(runtime/\'final.csv\', output/\'submission.csv\')\n    assert hashlib.sha256((output/\'submission.csv\').read_bytes()).hexdigest() == report[\'csv_sha256\']\n    return report\n'}
FILE_HASHES = {'hengck_original_helpers.py': '514439b744975ac77d7194b25528a4100f10a9b0a37821afb7fb6933f62d6859', 'hengck_original_config.py': '7d5ff6f1ecd6fae9d41c4442c8e608826d3e3bdfad1fcca82e4d254b8544d32f', 'hengck_plain_checkpoint.py': '38d029768f8cd461a18701aec318cc10226c75a9ff827bf31a581fa841b8a517', 'hengck_frame_cache.py': '17ef5008b89e69b4ff6d64826b4c01c9d199eeab2157f247c5cab48e318d0c79', 'hengck_test_runtime.py': '544c97fde0334fcc166fd1bf8c882da7853eaae2ce151f54764954e2d25be1f8', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/hengck_division_geometry.py': 'c4943d52eb05ef47fe5abade397b4b1d69842d425a68f79b34abf682de7245f2', 'src/k3_hengck_fork_transfer.py': '7274f57d8781293ddef8967f8baec3d5236e02d713b6b2919f45e1ac613057f5', 'src/fork_transfer_csv.py': '3109867b467794cecc64e299b9fe53b6fd73fe1f5f73b61753b4c57d8983d76b', 'src/k3_hengck_production_adapter.py': '11d9f15b20f9675d435a74f756ac317c034ab655e1e7ae19c606b987db3d6146', 'src/hengck_division_opportunity.py': '09dd1fdddcf3e79f96902c07688e8cf5ab65ab431ab6e081331de806b3dff17c', 'k3_cells.json': '5945fd3929e35efedff92b680c6ab834420770a9f5c2b905804c39ed2697ef03', 'k3_child.py': '0add3caa26065689df2ba801f1265a8682b85dfbe15b23a31d847201e7f2508b', 'hengck_child.py': '94d694b66062c2e960047f8df79fa1a84d15a05693382ac05a92b2fa38ecdf6b', 'production.py': '5214db5bd980da2e5d47cc832583bd61884a75f8482c34e2b933b3d6e49d993b'}
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile

os.environ.setdefault('POLARS_PREFER_PKG', '32')
OUTPUT = Path('/kaggle/working')
INITIAL_FILES = {p.relative_to(OUTPUT).as_posix() for p in OUTPUT.rglob('*') if p.is_file()}
SUPPORT = Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1')
PACKAGES = ['tracksdata', 'zarr==3.2.1', 'numcodecs==0.15.1', 'donfig==0.8.1.post1',
 'geff==1.2.0.1.1', 'geff-spec==1.1.1', 'pyscipopt==6.2.1', 'ilpy==0.6.0',
 'rustworkx==0.18.0', 'polars==1.42.0', 'polars-runtime-32==1.42.0', 'bidict==0.23.1', 'imagecodecs==2026.6.26']
assert not (OUTPUT/'submission.csv').exists()
try:
    with tempfile.TemporaryDirectory(prefix='k3_hengck_', dir='/tmp') as temporary:
        RUNTIME = Path(temporary)
        for name, source in FILES.items():
            path = RUNTIME/name
            assert path.resolve().is_relative_to(RUNTIME.resolve())
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(source, encoding='utf-8')
            assert hashlib.sha256(path.read_bytes()).hexdigest() == FILE_HASHES[name]
        with (RUNTIME/'install.log').open('w') as log:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                '--find-links', str(SUPPORT/'wheels'), *PACKAGES], check=True, timeout=180,
                stdout=log, stderr=subprocess.STDOUT)
        sys.path.insert(0, str(RUNTIME))
        from production import run
        report = run(RUNTIME, OUTPUT, FILE_HASHES)
        for name, expected in FILE_HASHES.items():
            assert hashlib.sha256((RUNTIME/name).read_bytes()).hexdigest() == expected
    assert not RUNTIME.exists()
    # Original K3 uses this fixed destination for aggregate retention counts.
    # Delete only its known diagnostic files; raw predictions were in /tmp.
    for pattern in ('retention_guard_*.jsonl', 'detector_coordinates_*.jsonl'):
        for path in OUTPUT.glob(pattern):
            path.unlink()
    report['temporary_files_removed'] = True
    new_files = {p.relative_to(OUTPUT).as_posix() for p in OUTPUT.rglob('*') if p.is_file()} - INITIAL_FILES
    assert new_files == {'submission.csv'}, new_files
    report['only_final_csv_and_report_exported'] = True
    (OUTPUT/'experiment_diagnostics.json').write_text(json.dumps(report, indent=2, allow_nan=False))
    print(json.dumps(dict(complete=True, videos=len(report['test_stems']),
        added_edges=report['transfer']['csv']['added_edges'], seconds=report['elapsed_seconds'])))
except Exception as exc:
    (OUTPUT/'submission.csv').unlink(missing_ok=True)
    (OUTPUT/'failure.json').write_text(json.dumps(dict(complete=False, error_type=type(exc).__name__, error=str(exc))))
    raise
